In [1]:
import sys
sys.path.insert(0, r'C:\projects\hedge_fund')

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from data.store.duckdb_client import DuckDBClient
from data.store.s3_client import S3Client

db = DuckDBClient()
s  = S3Client()

# Load all clean USDINR data (both contracts)
all_files  = s.list_files('processed/1s/USDINR26SEPFUT/')
all_files += s.list_files('processed/1s/USDINR26OCTFUT/')

all_df = []
for f in all_files:
    parts = f.split('/')
    sym   = parts[2]
    date  = parts[3].replace('.parquet','')
    try:
        df = db.read_parquet(f'processed/1s/{sym}/{date}.parquet')
        df['ist_sec'] = (df['ts_sec'] + 19800) % 86400
        df = df[(df['ist_sec'] >= 9*3600) &
                (df['ist_sec'] <= 17*3600)].reset_index(drop=True)
        df['date']   = date
        df['symbol'] = sym
        all_df.append(df)
        print(f'{sym}/{date}: {len(df)} bars')
    except Exception as e:
        print(f'ERROR {sym}/{date}: {e}')

db.close()
df_raw = pd.concat(all_df, ignore_index=True).reset_index(drop=True)
print(f'\nTotal: {len(df_raw):,} bars across {df_raw["date"].nunique()} days')
print(f'Dates: {sorted(df_raw["date"].unique())}')
print(f'Columns: {len(df_raw.columns)}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26SEPFUT/2026-09-22: 12568 bars
USDINR26SEPFUT/2026-09-23: 11031 bars
USDINR26SEPFUT/2026-09-24: 10385 bars
USDINR26SEPFUT/2026-09-25: 15469 bars
USDINR26SEPFUT/2026-09-28: 2921 bars
USDINR26OCTFUT/2026-09-30: 16701 bars
USDINR26OCTFUT/2026-10-01: 16554 bars
USDINR26OCTFUT/2026-10-05: 14944 bars
USDINR26OCTFUT/2026-10-06: 18290 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-07: 19336 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-08: 20968 bars
USDINR26OCTFUT/2026-10-09: 20206 bars
USDINR26OCTFUT/2026-10-12: 18406 bars
USDINR26OCTFUT/2026-10-13: 18435 bars


... (4 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-21: 17745 bars
USDINR26OCTFUT/2026-10-22: 20903 bars


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

USDINR26OCTFUT/2026-10-23: 18438 bars
USDINR26OCTFUT/2026-10-26: 13693 bars
USDINR26OCTFUT/2026-10-27: 9264 bars

Total: 364,987 bars across 23 days
Dates: ['2026-09-22', '2026-09-23', '2026-09-24', '2026-09-25', '2026-09-28', '2026-09-30', '2026-10-01', '2026-10-05', '2026-10-06', '2026-10-07', '2026-10-08', '2026-10-09', '2026-10-12', '2026-10-13', '2026-10-14', '2026-10-15', '2026-10-16', '2026-10-19', '2026-10-21', '2026-10-22', '2026-10-23', '2026-10-26', '2026-10-27']
Columns: 55


In [2]:
df = (df_raw.sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True))
print(df.shape)
print(df["date"].nunique())
print(df.head())

(364987, 55)
23
           symbol      ts_sec     open  high      low  close  volume_delta  \
0  USDINR26OCTFUT  1790739000  96.2725  96.3  96.2725   96.3             1   
1  USDINR26OCTFUT  1790739001  96.3000  96.3  96.3000   96.3             0   
2  USDINR26OCTFUT  1790739002  96.3000  96.3  96.3000   96.3             0   
3  USDINR26OCTFUT  1790739003  96.3000  96.3  96.3000   96.3             0   
4  USDINR26OCTFUT  1790739004  96.3000  96.3  96.3000   96.3             0   

   tick_count       vwap      oi  ...  imbalance_ma_30s  imbalance_ma_60s  \
0           2  96.272647  961002  ...          0.696133          0.696133   
1           1  96.272647  961002  ...          0.671351          0.671351   
2           1  96.272647  961002  ...          0.655217          0.655217   
3           1  96.272647  961002  ...          0.654130          0.654130   
4           1  96.272647  961002  ...          0.653632          0.653632   

   spread_zscore  volume_ratio  price_mom_10s  price

In [3]:
duplicates = df.duplicated(subset=[
        "symbol",
        "date",
        "ts_sec"],
    keep=False)
print("Duplicate symbol-date-second rows:", duplicates.sum())

Duplicate symbol-date-second rows: 0


In [4]:
df["is_observed"] = 1
df["had_update"] = 1

In [5]:
def regularize_day(day):
    day = day.sort_values("ts_sec").copy()
    start = int(day["ts_sec"].min())
    end = int(day["ts_sec"].max())
    full_seconds = pd.DataFrame({
        "ts_sec": np.arange(
            start,
            end + 1)
    })
    symbol = day["symbol"].iloc[0]
    date = day["date"].iloc[0]
    full_seconds["symbol"] = symbol
    full_seconds["date"] = date

    out = full_seconds.merge(day,
            on=[
            "symbol",
            "date",
            "ts_sec"], how="left")

    return out

In [6]:
df_1s = (
    df
    .groupby(
        ["symbol", "date"],
        group_keys=False
    )
    .apply(regularize_day)
    .reset_index(drop=True)
)

In [7]:
df_1s["is_observed"] = (
    df_1s["is_observed"]
    .fillna(0)
    .astype(int)
)

df_1s["had_update"] = (
    df_1s["had_update"]
    .fillna(0)
    .astype(int)
)

In [8]:
state_cols = [
    "oi",

    "bid_p1", "bid_q1",
    "bid_p2", "bid_q2",
    "bid_p3", "bid_q3",
    "bid_p4", "bid_q4",
    "bid_p5", "bid_q5",

    "ask_p1", "ask_q1",
    "ask_p2", "ask_q2",
    "ask_p3", "ask_q3",
    "ask_p4", "ask_q4",
    "ask_p5", "ask_q5",
]

In [9]:
state_cols = [
    c for c in state_cols
    if c in df_1s.columns
]

In [10]:
df_1s[state_cols] = (
    df_1s
    .groupby(
        ["symbol", "date"]
    )[state_cols]
    .ffill()
)

In [11]:
flow_cols = [
    "volume_delta",
    "tick_count"
]

for col in flow_cols:

    if col in df_1s.columns:

        df_1s[col] = (
            df_1s[col]
            .fillna(0)
        )

In [12]:
df_1s["mid"] = (
    df_1s["bid_p1"]
    +
    df_1s["ask_p1"]
) / 2
df_1s["spread"] = (
    df_1s["ask_p1"]
    -
    df_1s["bid_p1"]
)
eps = 1e-12

df_1s["spread_bps_calc"] = (
    1e4
    *
    df_1s["spread"]
    /
    (df_1s["mid"] + eps)
)
df_1s["log_mid"] = np.log(
    df_1s["mid"]
)
df_1s["ret_1s"] = (
    df_1s["log_mid"]
    -
    df_1s
    .groupby(
        ["symbol", "date"]
    )["log_mid"]
    .shift(1)
)
bid_cols = [
    f"bid_q{i}"
    for i in range(1, 6)
]

ask_cols = [
    f"ask_q{i}"
    for i in range(1, 6)
]

df_1s["bid_depth_l5"] = (
    df_1s[bid_cols]
    .sum(axis=1)
)

df_1s["ask_depth_l5"] = (
    df_1s[ask_cols]
    .sum(axis=1)
)

df_1s["total_depth_l5"] = (
    df_1s["bid_depth_l5"]
    +
    df_1s["ask_depth_l5"]
)
session_summary = (
    df_1s
    .groupby(["symbol", "date"])
    .agg(
        first_sec=("ts_sec", "min"),
        last_sec=("ts_sec", "max"),
        rows=("ts_sec", "size"),
        observed_rows=("is_observed", "sum")
    )
)

session_summary["observation_rate"] = (
    session_summary["observed_rows"]
    /
    session_summary["rows"]
)

print(session_summary)

                            first_sec    last_sec   rows  observed_rows  \
symbol         date                                                       
USDINR26OCTFUT 2026-09-30  1790739000  1790767748  28749          16701   
               2026-10-01  1790825780  1790854181  28402          16554   
               2026-10-05  1791171000  1791199766  28767          14944   
               2026-10-06  1791257400  1791286154  28755          18290   
               2026-10-07  1791343800  1791372551  28752          19336   
               2026-10-08  1791430200  1791458948  28749          20968   
               2026-10-09  1791516600  1791545347  28748          20206   
               2026-10-12  1791775800  1791804553  28754          18406   
               2026-10-13  1791862200  1791890992  28793          18435   
               2026-10-14  1791948600  1791977391  28792          17290   
               2026-10-15  1792035000  1792063796  28797          17281   
               2026-10-16

In [13]:
print(df_1s.columns.tolist())

print("\nDataset shape:")
print(df_1s.shape)

print("\nAvailable observation indicators:")
print([
    c for c in df_1s.columns
    if any(
        word in c.lower()
        for word in [
            "observed", "synthetic", "filled",
            "imputed", "original", "update"
        ]
    )
])

['ts_sec', 'symbol', 'date', 'open', 'high', 'low', 'close', 'volume_delta', 'tick_count', 'vwap', 'oi', 'spread_mean', 'spread_max', 'spread_bps', 'imbalance_mean', 'imbalance_std', 'imbalance_last', 'total_bid_qty', 'total_ask_qty', 'weighted_mid', 'price_impact', 'bid_p1', 'bid_q1', 'bid_p2', 'bid_q2', 'bid_p3', 'bid_q3', 'bid_p4', 'bid_q4', 'bid_p5', 'bid_q5', 'ask_p1', 'ask_q1', 'ask_p2', 'ask_q2', 'ask_p3', 'ask_q3', 'ask_p4', 'ask_q4', 'ask_p5', 'ask_q5', 'realized_vol_10s', 'realized_vol_30s', 'realized_vol_60s', 'realized_vol_300s', 'imbalance_ma_10s', 'imbalance_ma_30s', 'imbalance_ma_60s', 'spread_zscore', 'volume_ratio', 'price_mom_10s', 'price_mom_30s', 'price_mom_60s', 'ts_ist', 'ist_sec', 'is_observed', 'had_update', 'mid', 'spread', 'spread_bps_calc', 'log_mid', 'ret_1s', 'bid_depth_l5', 'ask_depth_l5', 'total_depth_l5']

Dataset shape:
(639440, 65)

Available observation indicators:
['is_observed', 'had_update']


In [14]:
import numpy as np
import pandas as pd

print("===== OBSERVATION INDICATORS =====")

print(
    df_1s[["is_observed", "had_update"]]
    .value_counts(dropna=False)
    .to_string()
)

print("\n===== INDICATOR RATES BY DATE =====")

print(
    df_1s.groupby(["symbol", "date"])
    .agg(
        rows=("ts_sec", "size"),
        observed_rate=("is_observed", "mean"),
        update_rate=("had_update", "mean")
    )
    .round(4)
    .to_string()
)

print("\n===== SNAPSHOT CHANGES VS INDICATORS =====")

book_columns = [
    f"{side}_{kind}{i}"
    for side in ["bid", "ask"]
    for i in range(1, 6)
    for kind in ["p", "q"]
]

work = df_1s.sort_values(
    ["symbol", "date", "ts_sec"]
).copy()

# A change in any displayed price or quantity
# compared with the previous row in the session.
previous = work.groupby(
    ["symbol", "date"],
    sort=False
)[book_columns].shift(1)

changed = work[book_columns].ne(previous).any(axis=1)

first_row = work.groupby(
    ["symbol", "date"]
).cumcount().eq(0)

changed = changed & ~first_row

work["_book_changed"] = changed

print(
    pd.crosstab(
        [
            work["is_observed"],
            work["had_update"]
        ],
        work["_book_changed"],
        margins=True
    )
)

print("\n===== COMPLETE =====")

===== OBSERVATION INDICATORS =====
is_observed  had_update
1            1             364987
0            0             274453

===== INDICATOR RATES BY DATE =====
                            rows  observed_rate  update_rate
symbol         date                                         
USDINR26OCTFUT 2026-09-30  28749         0.5809       0.5809
               2026-10-01  28402         0.5828       0.5828
               2026-10-05  28767         0.5195       0.5195
               2026-10-06  28755         0.6361       0.6361
               2026-10-07  28752         0.6725       0.6725
               2026-10-08  28749         0.7293       0.7293
               2026-10-09  28748         0.7029       0.7029
               2026-10-12  28754         0.6401       0.6401
               2026-10-13  28793         0.6403       0.6403
               2026-10-14  28792         0.6005       0.6005
               2026-10-15  28797         0.6001       0.6001
               2026-10-16  28796         0.

In [15]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# EXPERIMENT 34A — EVENT-AWARE BOOK RESILIENCY
#
# Uses only the ORIGINAL research sessions
# through October 19, 2026.
#
# OCT 21-23 are kept untouched.
#
# Raw input: df_1s
#
# Outputs:
#   calendar_exp34a_features
#   calendar_exp34a_daily
#   calendar_exp34a_summary
#
# No feature promotion.
# ============================================================

HORIZONS_34A = [30, 60, 300]
SHOCK_THRESHOLD_34A = 0.20
RECOVERY_CAP_34A = 2.0

KEYS_34A = ["symbol", "date", "ts_sec"]

CANDIDATES_34A = [
    "l5_recovery_asym_34a",
    "l5_bid_recovery_34a",
    "l5_ask_recovery_34a",
    "l1_sameprice_recovery_asym_34a"
]

required = (
    ["symbol", "ts_sec", "is_observed"]
    + [f"bid_p{i}" for i in range(1, 6)]
    + [f"ask_p{i}" for i in range(1, 6)]
    + [f"bid_q{i}" for i in range(1, 6)]
    + [f"ask_q{i}" for i in range(1, 6)]
)

missing = sorted(set(required) - set(df_1s.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")

# ------------------------------------------------------------
# 1. Select original research dates
# ------------------------------------------------------------

x34 = df_1s[required].copy()
x34["symbol"] = x34["symbol"].astype(str)
x34["ts_sec"] = pd.to_numeric(
    x34["ts_sec"], errors="raise"
).astype("int64")

ist = pd.to_datetime(
    x34["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x34["date"] = ist.dt.date

clock = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second - 32400
)

x34 = x34.loc[
    (x34["date"].astype(str) <= "2026-10-19")
    & (x34["date"].astype(str) != "2026-09-28")
    & clock.between(0, 28799)
].copy()

counts = x34.groupby(
    ["symbol", "date"]
)["ts_sec"].transform("nunique")

x34 = x34.loc[
    counts >= 0.97 * 28800
].sort_values(
    KEYS_34A
).reset_index(drop=True)

if x34.duplicated(KEYS_34A).any():
    raise ValueError("Duplicate session timestamps")

for col in required:
    if col not in ["symbol", "ts_sec"]:
        x34[col] = pd.to_numeric(
            x34[col], errors="coerce"
        )

print("===== 34A RESEARCH SAMPLE =====")
print("Rows:", len(x34))
print(
    "Sessions:",
    x34.groupby(["symbol", "date"]).ngroups
)
print(
    "Latest date:",
    x34["date"].max()
)

# ------------------------------------------------------------
# 2. Aggregate displayed depth
# ------------------------------------------------------------

bq = [f"bid_q{i}" for i in range(1, 6)]
aq = [f"ask_q{i}" for i in range(1, 6)]

BQ = x34[bq].to_numpy(dtype=np.float64)
AQ = x34[aq].to_numpy(dtype=np.float64)

bid_depth = np.where(
    np.isfinite(BQ).all(axis=1)
    & (BQ >= 0).all(axis=1),
    BQ.sum(axis=1),
    np.nan
)

ask_depth = np.where(
    np.isfinite(AQ).all(axis=1)
    & (AQ >= 0).all(axis=1),
    AQ.sum(axis=1),
    np.nan
)

bid_l1 = x34["bid_q1"].to_numpy(dtype=float)
ask_l1 = x34["ask_q1"].to_numpy(dtype=float)

bid_p1 = x34["bid_p1"].to_numpy(dtype=float)
ask_p1 = x34["ask_p1"].to_numpy(dtype=float)

mid = (bid_p1 + ask_p1) / 2

mid[
    (~np.isfinite(mid))
    | (bid_p1 <= 0)
    | (ask_p1 < bid_p1)
] = np.nan

observed = x34["is_observed"].eq(1).to_numpy()

# ------------------------------------------------------------
# 3. Exact timestamp lookup utility
#
# offset = -60, -30, +30 etc.
# Session-safe; never crosses dates.
# ------------------------------------------------------------

session_index = pd.MultiIndex.from_frame(
    x34[KEYS_34A]
)

def lookup_34a(array, offset):
    lookup_index = pd.MultiIndex.from_arrays(
        [
            x34["symbol"],
            x34["date"],
            x34["ts_sec"].to_numpy() + offset
        ],
        names=KEYS_34A
    )

    positions = session_index.get_indexer(
        lookup_index
    )

    result = np.full(
        len(x34), np.nan, dtype=np.float64
    )

    valid = positions >= 0

    result[valid] = np.asarray(
        array, dtype=np.float64
    )[positions[valid]]

    return result

# ------------------------------------------------------------
# 4. Recovery calculation
#
# t-60: Before
# t-30: Disturbed
# t:    Current recovery
#
# Disturbance anchor must have a real observation.
# ------------------------------------------------------------

observed_at_shock = (
    lookup_34a(observed.astype(float), -30) == 1
)

def recovery_34a(depth):

    before = lookup_34a(depth, -60)
    shocked = lookup_34a(depth, -30)
    current = np.asarray(depth, dtype=float)

    depletion = before - shocked

    eligible = (
        observed_at_shock
        & np.isfinite(before)
        & np.isfinite(shocked)
        & np.isfinite(current)
        & (before > 0)
        & (shocked >= 0)
        & (current >= 0)
        & (
            depletion
            >= SHOCK_THRESHOLD_34A * before
        )
    )

    recovered = np.maximum(
        current - shocked, 0
    )

    ratio = np.full(len(depth), np.nan)

    ratio[eligible] = np.clip(
        recovered[eligible]
        / depletion[eligible],
        0,
        RECOVERY_CAP_34A
    )

    return ratio, eligible

l5_bid, bid_event = recovery_34a(bid_depth)
l5_ask, ask_event = recovery_34a(ask_depth)

l1_bid, bid_l1_event = recovery_34a(bid_l1)
l1_ask, ask_l1_event = recovery_34a(ask_l1)

# Ensure same L1 best quote at the three sampled times.
bid_same_price = (
    (bid_p1 == lookup_34a(bid_p1, -30))
    & (bid_p1 == lookup_34a(bid_p1, -60))
)

ask_same_price = (
    (ask_p1 == lookup_34a(ask_p1, -30))
    & (ask_p1 == lookup_34a(ask_p1, -60))
)

l1_bid[~bid_same_price] = np.nan
l1_ask[~ask_same_price] = np.nan

# ------------------------------------------------------------
# 5. Features
#
# Recovery asymmetry requires BOTH sides to have
# an eligible disturbance.
#
# This can be sparse. Coverage is diagnostic.
# ------------------------------------------------------------

l5_asym = l5_bid - l5_ask
l1_asym = l1_bid - l1_ask

calendar_exp34a_features = x34[
    KEYS_34A
].copy()

calendar_exp34a_features[
    "l5_recovery_asym_34a"
] = l5_asym.astype(np.float32)

calendar_exp34a_features[
    "l5_bid_recovery_34a"
] = l5_bid.astype(np.float32)

calendar_exp34a_features[
    "l5_ask_recovery_34a"
] = l5_ask.astype(np.float32)

calendar_exp34a_features[
    "l1_sameprice_recovery_asym_34a"
] = l1_asym.astype(np.float32)

# ------------------------------------------------------------
# 6. Exact future-return labels
# ------------------------------------------------------------

for h in HORIZONS_34A:

    future_mid = lookup_34a(mid, h)

    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):
        future_return = (
            1e4 * np.log(future_mid / mid)
        )

    future_return[
        ~np.isfinite(future_return)
    ] = np.nan

    calendar_exp34a_features[
        f"future_return_{h}_34a"
    ] = future_return.astype(np.float32)

# ------------------------------------------------------------
# 7. Daily Spearman IC
# ------------------------------------------------------------

def daily_ic_34a(a, b):

    valid = np.isfinite(a) & np.isfinite(b)

    n = int(valid.sum())

    if n < 100:
        return np.nan, n

    if (
        np.unique(a[valid]).size < 2
        or np.unique(b[valid]).size < 2
    ):
        return np.nan, n

    return (
        pd.Series(a[valid]).corr(
            pd.Series(b[valid]),
            method="spearman"
        ),
        n
    )

records = []

for (symbol, date), day in (
    calendar_exp34a_features.groupby(
        ["symbol", "date"], sort=False
    )
):

    for h in HORIZONS_34A:

        y = day[
            f"future_return_{h}_34a"
        ].to_numpy(dtype=float)

        for feature in CANDIDATES_34A:

            values = day[feature].to_numpy(
                dtype=float
            )

            ic, n = daily_ic_34a(values, y)

            records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "feature": feature,
                "ic": ic,
                "valid_rows": n
            })

calendar_exp34a_daily = pd.DataFrame(records)

calendar_exp34a_summary = (
    calendar_exp34a_daily
    .groupby(["horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 8. Diagnostics
# ------------------------------------------------------------

print("\n===== 34A EVENT COVERAGE =====")

print(
    "Bid L5 shock events:",
    int(bid_event.sum()),
    "coverage %:",
    round(100 * bid_event.mean(), 4)
)

print(
    "Ask L5 shock events:",
    int(ask_event.sum()),
    "coverage %:",
    round(100 * ask_event.mean(), 4)
)

print(
    "Bid L1 shock events:",
    int((bid_l1_event & bid_same_price).sum())
)

print(
    "Ask L1 shock events:",
    int((ask_l1_event & ask_same_price).sum())
)

print("\n===== 34A FEATURE VALIDITY =====")

for col in CANDIDATES_34A:
    s = calendar_exp34a_features[col]

    print(
        col,
        "valid_pct=",
        round(100*s.notna().mean(), 4),
        "std=",
        round(s.std(), 6),
        "median=",
        round(s.median(), 6)
    )

print("\n===== 34A DAILY SIGNED IC =====")

print(
    calendar_exp34a_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== EXPERIMENT 34A COMPLETE =====")
print("Original df_1s unchanged.")
print("October 21-23 excluded.")
print("Core feature count remains 37.")

gc.collect()


===== 34A RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Latest date: 2026-10-19

===== 34A EVENT COVERAGE =====
Bid L5 shock events: 59161 coverage %: 12.8857
Ask L5 shock events: 61739 coverage %: 13.4472
Bid L1 shock events: 10475
Ask L1 shock events: 12663

===== 34A FEATURE VALIDITY =====
l5_recovery_asym_34a valid_pct= 3.1218 std= 0.922057 median= 0.0
l5_bid_recovery_34a valid_pct= 12.8857 std= 0.64723 median= 0.051943
l5_ask_recovery_34a valid_pct= 13.4472 std= 0.651856 median= 0.168605
l1_sameprice_recovery_asym_34a valid_pct= 0.1925 std= 0.981556 median= 0.0

===== 34A DAILY SIGNED IC =====
 horizon                        feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows
      30 l1_sameprice_recovery_asym_34a     0       NaN        NaN              0              0               52.5
      30            l5_ask_recovery_34a    16 -0.088811  -0.091878              0             16             3917.0
      30            l5_bid_recovery_34a    1

1209

In [16]:
import pandas as pd

core_31 = pd.read_parquet("usdinr_31_features.parquet")

In [17]:
print("Shape:", core_31.shape)
print("Columns:", core_31.columns.tolist())
print(core_31[["symbol", "ts_sec"]].head())

Shape: (524325, 33)
Columns: ['symbol', 'ts_sec', 'ofi_l1_sum_60s_signed_log', 'ofi_rate_diff_30_300_sl', 'ofi_activity_60s', 'ofi_activity_300s', 'update_rate_300s', 'price_move_rate_30s', 'impact_lambda_1800s', 'impact_r2_1800s', 'ofi_response_gap_z', 'queue_imbalance_l1', 'depth_imbalance_l5', 'queue_replenishment_pressure_60s', 'queue_imbalance_delta', 'depth_imbalance_delta', 'spread_rank_1800s', 'price_mom_10s_new', 'price_mom_30s_new', 'price_mom_60s_new', 'log_rv_300s_new', 'rv_ratio_60_300_new', 'ret_acf_lag1_300s', 'log_vr_5_300s', 'log_vr_30_300s', 'semivar_imbalance_300s', 'semivar_imbalance_900s', 'sign_persistence_900s', 'jump_ratio_300s', 'range_position_300s', 'spread_depth_stress_900', 'recovery_available_300', 'ofi_disagreement_weighted_60']
           symbol      ts_sec
0  USDINR26OCTFUT  1790739000
1  USDINR26OCTFUT  1790739001
2  USDINR26OCTFUT  1790739002
3  USDINR26OCTFUT  1790739003
4  USDINR26OCTFUT  1790739004


In [18]:
import pandas as pd

raw_keys = df_1s[["symbol", "ts_sec"]].copy()
saved_keys = core_31[["symbol", "ts_sec"]].copy()

print("===== DATASET CHECK =====")
print("Raw rows:", len(raw_keys))
print("Saved feature rows:", len(saved_keys))

print("Raw duplicate keys:", raw_keys.duplicated().sum())
print("Feature duplicate keys:", saved_keys.duplicated().sum())

raw_idx = pd.MultiIndex.from_frame(raw_keys)
saved_idx = pd.MultiIndex.from_frame(saved_keys)

print("Saved rows found in raw:", saved_idx.isin(raw_idx).sum())
print("Raw rows with saved features:", raw_idx.isin(saved_idx).sum())

dates = pd.to_datetime(
    core_31["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

print("\n===== SAVED FEATURE DATE COVERAGE =====")
print(dates.value_counts().sort_index().to_string())

===== DATASET CHECK =====
Raw rows: 639440
Saved feature rows: 524325
Raw duplicate keys: 0
Feature duplicate keys: 0
Saved rows found in raw: 524325
Raw rows with saved features: 524325

===== SAVED FEATURE DATE COVERAGE =====
ts_sec
2026-09-22    27175
2026-09-23    28437
2026-09-24    28607
2026-09-25    28440
2026-09-28     9281
2026-09-30    28749
2026-10-01    28402
2026-10-05    28767
2026-10-06    28755
2026-10-07    28752
2026-10-08    28749
2026-10-09    28748
2026-10-12    28754
2026-10-13    28793
2026-10-14    28792
2026-10-15    28797
2026-10-16    28796
2026-10-19    28784
2026-10-21    28747


In [19]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# REBUILD RESEARCH FEATURE FRAMES
#
# Input:
#   df_1s      : updated 1-second order-book data
#   core_31    : loaded from usdinr_31_features.parquet
#
# Outputs:
#   calendar_exp24a_features
#   calendar_exp28a_features
#   calendar_exp29a_features
#   calendar_exp31a_features
#   usdinr_rebuilt_37_research
#
# IMPORTANT:
# Quote retreat, slope, curvature, and nine controls
# are RECONSTRUCTIONS, not verified original formulas.
#
# Do not interpret results as an exact historical
# 37-feature-baseline replication until parity verified.
# ============================================================

KEYS = ["symbol", "ts_sec"]
RESEARCH_END = pd.Timestamp("2026-10-19").date()
MIN_COVERAGE = 0.97
EPS = 1e-12

CORE31 = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

ADDED6 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

assert len(CORE31) == 31
assert len(ADDED6) == 6
assert len(CONTROLS9) == 9

# ------------------------------------------------------------
# 1. Research-only raw order book
# ------------------------------------------------------------

prices = (
    [f"bid_p{i}" for i in range(1, 6)]
    + [f"ask_p{i}" for i in range(1, 6)]
)

quantities = (
    [f"bid_q{i}" for i in range(1, 6)]
    + [f"ask_q{i}" for i in range(1, 6)]
)

required = (
    KEYS + prices + quantities + ["is_observed"]
)

missing = [
    c for c in required if c not in df_1s.columns
]

if missing:
    raise ValueError(f"Missing raw columns: {missing}")

x = df_1s[required].copy()

x["symbol"] = x["symbol"].astype(str)
x["ts_sec"] = pd.to_numeric(
    x["ts_sec"], errors="raise"
).astype("int64")

ist = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ist.dt.date

x["clock_seconds"] = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

# Explicit chronological research boundary.
x = x.loc[
    (x["date"] <= RESEARCH_END)
    & (x["date"].astype(str) != "2026-09-28")
    & x["clock_seconds"].between(0, 28799)
].copy()

coverage = (
    x.groupby(["symbol", "date"])["ts_sec"]
    .transform("nunique")
)

x = x.loc[
    coverage >= MIN_COVERAGE * 28800
].copy()

x = (
    x.sort_values(
        ["symbol", "date", "ts_sec"]
    )
    .reset_index(drop=True)
)

if x.duplicated(KEYS).any():
    raise ValueError("Duplicate symbol/timestamp keys")

if x.empty:
    raise ValueError("No eligible research sessions")

for col in prices + quantities:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

print("===== RESEARCH SAMPLE =====")
print("Rows:", len(x))
print(
    "Sessions:",
    x.groupby(["symbol", "date"]).ngroups
)
print("Latest date:", x["date"].max())

# ------------------------------------------------------------
# 2. Book matrices
# ------------------------------------------------------------

B = x[
    [f"bid_p{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

A = x[
    [f"ask_p{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

BQ = x[
    [f"bid_q{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

AQ = x[
    [f"ask_q{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

n = len(x)

valid_book = (
    np.isfinite(B).all(axis=1)
    & np.isfinite(A).all(axis=1)
    & np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (B > 0).all(axis=1)
    & (A > 0).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (A[:, 0] >= B[:, 0])
)

mid = (B[:, 0] + A[:, 0]) / 2
mid[~valid_book] = np.nan

bid_total = BQ.sum(axis=1)
ask_total = AQ.sum(axis=1)

bid_total[~valid_book] = np.nan
ask_total[~valid_book] = np.nan

total_depth = bid_total + ask_total

# ------------------------------------------------------------
# 3. Exact timestamp lookup
# ------------------------------------------------------------

idx = pd.MultiIndex.from_frame(
    x[["symbol", "date", "ts_sec"]]
)

symbols = x["symbol"]
dates = x["date"]
timestamps = x["ts_sec"].to_numpy(
    dtype=np.int64
)

def lookup_exact(array, offset):

    lookup_idx = pd.MultiIndex.from_arrays(
        [
            symbols,
            dates,
            timestamps + offset
        ],
        names=["symbol", "date", "ts_sec"]
    )

    positions = idx.get_indexer(lookup_idx)
    found = positions >= 0

    result = np.full(
        n, np.nan, dtype=np.float64
    )

    arr = np.asarray(
        array, dtype=np.float64
    )

    result[found] = arr[positions[found]]

    return result

# ------------------------------------------------------------
# 4. Experiment 24 — Quote retreat reconstruction
#
# Counts observed outward movements of the best
# bid/ask over a trailing 30-snapshot window.
#
# This is NOT VERIFIED as the original 24A formula.
# ------------------------------------------------------------

bid_prev = lookup_exact(B[:, 0], -1)
ask_prev = lookup_exact(A[:, 0], -1)

is_observed = (
    x["is_observed"].eq(1).to_numpy()
)

# Bid retreats when best bid falls.
bid_retreat = (
    is_observed
    & np.isfinite(bid_prev)
    & (B[:, 0] < bid_prev)
).astype(float)

# Ask retreats when best ask rises.
ask_retreat = (
    is_observed
    & np.isfinite(ask_prev)
    & (A[:, 0] > ask_prev)
).astype(float)

scratch = pd.DataFrame({
    "symbol": x["symbol"],
    "date": x["date"],
    "bid_retreat": bid_retreat,
    "ask_retreat": ask_retreat
})

g = scratch.groupby(
    ["symbol", "date"], sort=False
)

bid_retreat_30 = g["bid_retreat"].transform(
    lambda s: s.rolling(30, min_periods=30).sum()
)

ask_retreat_30 = g["ask_retreat"].transform(
    lambda s: s.rolling(30, min_periods=30).sum()
)

# Positive = greater ask retreat than bid retreat.
quote_retreat_30 = (
    ask_retreat_30.to_numpy(dtype=float)
    - bid_retreat_30.to_numpy(dtype=float)
) / (
    ask_retreat_30.to_numpy(dtype=float)
    + bid_retreat_30.to_numpy(dtype=float)
    + 1.0
)

# ------------------------------------------------------------
# 5. Experiment 28 — Book slope and curvature
#
# Reconstructed geometry:
# Fit log(1+quantity) to a centered quadratic
# over book levels 1..5 on each side.
#
# slope = first-order coefficient difference
# curvature = second-order coefficient difference
#
# NOT VERIFIED as original 28A implementation.
# ------------------------------------------------------------

level = np.arange(1, 6, dtype=float)

u = level - level.mean()
v = u**2
v_center = v - v.mean()

# u and centered v are orthogonal for levels 1..5.
u_denom = np.sum(u**2)
v_denom = np.sum(v_center**2)

log_bq = np.log1p(
    np.maximum(BQ, 0)
)

log_aq = np.log1p(
    np.maximum(AQ, 0)
)

bid_slope = (
    log_bq @ u
) / u_denom

ask_slope = (
    log_aq @ u
) / u_denom

bid_curvature = (
    log_bq @ v_center
) / v_denom

ask_curvature = (
    log_aq @ v_center
) / v_denom

slope_asym = bid_slope - ask_slope

curvature_asym = (
    bid_curvature - ask_curvature
)

slope_asym[~valid_book] = np.nan
curvature_asym[~valid_book] = np.nan

# ------------------------------------------------------------
# 6. Experiment 29 — Weighted gaps and fragility
#
# Reconstructed from established mathematical
# definitions of Exp29:
#
# Adjacent gaps in bps:
#   bid_i - bid_(i+1)
#   ask_(i+1) - ask_i
#
# w_i = exp(-0.7*i), normalized over i=0..3
# ------------------------------------------------------------

bid_gaps = (
    1e4 * (B[:, :-1] - B[:, 1:])
    / mid[:, None]
)

ask_gaps = (
    1e4 * (A[:, 1:] - A[:, :-1])
    / mid[:, None]
)

weights = np.exp(
    -0.7 * np.arange(4)
)

weights = weights / weights.sum()

weighted_gap_asym = (
    (bid_gaps - ask_gaps) @ weights
)

bid_fragility = (
    bid_gaps
    / np.sqrt(1.0 + BQ[:, :-1])
)

ask_fragility = (
    ask_gaps
    / np.sqrt(1.0 + AQ[:, :-1])
)

fragility_asym = (
    (bid_fragility - ask_fragility)
    @ weights
)

weighted_gap_asym[~valid_book] = np.nan
fragility_asym[~valid_book] = np.nan

# ------------------------------------------------------------
# 7. Experiment 31 — Relative migration
#
# Depth-weighted center of mass of distances
# from each side's best quote, in bps.
#
# Uses exact 60-second historical lookback.
# ------------------------------------------------------------

bid_distance = (
    1e4 * (
        B[:, [0]] - B
    ) / mid[:, None]
)

ask_distance = (
    1e4 * (
        A - A[:, [0]]
    ) / mid[:, None]
)

with np.errstate(
    divide="ignore", invalid="ignore"
):

    mu_bid = (
        (bid_distance * BQ).sum(axis=1)
        / bid_total
    )

    mu_ask = (
        (ask_distance * AQ).sum(axis=1)
        / ask_total
    )

mu_bid[~valid_book] = np.nan
mu_ask[~valid_book] = np.nan

mu_bid_lag = lookup_exact(mu_bid, -60)
mu_ask_lag = lookup_exact(mu_ask, -60)

delta_bid = mu_bid - mu_bid_lag
delta_ask = mu_ask - mu_ask_lag

relative_migration = (
    delta_bid - delta_ask
) / (
    np.abs(delta_bid)
    + np.abs(delta_ask)
    + EPS
)

# ------------------------------------------------------------
# 8. Experiment 28 — Nine controls
#
# Explicit reconstruction, not verified parity.
# ------------------------------------------------------------

with np.errstate(
    divide="ignore", invalid="ignore"
):

    queue_imbalance = (
        BQ[:, 0] - AQ[:, 0]
    ) / (
        BQ[:, 0] + AQ[:, 0] + EPS
    )

    depth_imbalance = (
        bid_total - ask_total
    ) / (
        total_depth + EPS
    )

    log_total_depth = np.log1p(
        total_depth
    )

    spread_bps = (
        1e4 * (A[:, 0] - B[:, 0]) / mid
    )

    bid_l1_concentration = (
        BQ[:, 0] / (bid_total + EPS)
    )

    ask_l1_concentration = (
        AQ[:, 0] / (ask_total + EPS)
    )

    past_mid30 = lookup_exact(mid, -30)
    past_mid60 = lookup_exact(mid, -60)

    momentum_30 = (
        1e4 * np.log(mid / past_mid30)
    )

    momentum_60 = (
        1e4 * np.log(mid / past_mid60)
    )

controls = {
    "queue_imbalance_28a": queue_imbalance,
    "depth_imbalance_28a": depth_imbalance,
    "log_total_depth_28a": log_total_depth,
    "spread_bps_28a": spread_bps,
    "bid_l1_concentration_28a": bid_l1_concentration,
    "ask_l1_concentration_28a": ask_l1_concentration,
    "momentum_30_28a": momentum_30,
    "momentum_60_28a": momentum_60,
    "clock_seconds": x["clock_seconds"].to_numpy()
}

for name in controls:

    values = np.asarray(
        controls[name], dtype=np.float64
    )

    if name != "clock_seconds":
        values[~valid_book] = np.nan

    controls[name] = values

# ------------------------------------------------------------
# 9. Construct original frame names
# ------------------------------------------------------------

keys_frame = x[KEYS].copy()

def make_feature_frame(feature_dict):

    out = keys_frame.copy()

    for name, values in feature_dict.items():

        arr = np.asarray(
            values, dtype=np.float64
        ).copy()

        arr[~np.isfinite(arr)] = np.nan

        out[name] = arr.astype(np.float32)

    return out

calendar_exp24a_features = make_feature_frame({
    "quote_retreat_imbalance_30_24a":
        quote_retreat_30
})

calendar_exp28a_features = make_feature_frame({
    "slope_asymmetry_28a": slope_asym,
    "curvature_asymmetry_28a": curvature_asym,
    **controls
})

calendar_exp29a_features = make_feature_frame({
    "weighted_gap_asymmetry_29a":
        weighted_gap_asym,
    "fragility_asymmetry_29a":
        fragility_asym
})

calendar_exp31a_features = make_feature_frame({
    "relative_migration_60_31a":
        relative_migration
})

# ------------------------------------------------------------
# 10. Align original 31 features
#
# Exact key matching, no assumptions about row order.
# ------------------------------------------------------------

core_index = pd.MultiIndex.from_arrays(
    [
        core_31["symbol"].astype(str),
        pd.to_numeric(
            core_31["ts_sec"],
            errors="raise"
        ).astype("int64")
    ],
    names=KEYS
)

if core_index.has_duplicates:
    raise ValueError(
        "Duplicate symbol/timestamp keys in core_31"
    )

target_index = pd.MultiIndex.from_frame(
    keys_frame
)

core_positions = core_index.get_indexer(
    target_index
)

matched = core_positions >= 0

print("\n===== CORE ALIGNMENT =====")

print(
    "Core key match:",
    f"{100 * matched.mean():.4f}%"
)

if not matched.all():
    raise ValueError(
        "Some research rows are absent from core_31"
    )

combined_arrays = {}

for col in CORE31:

    if col not in core_31.columns:
        raise ValueError(f"Core column missing: {col}")

    source_values = pd.to_numeric(
        core_31[col],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    combined_arrays[col] = source_values[
        core_positions
    ]

for frame, cols in [
    (calendar_exp24a_features, ADDED6[:1]),
    (calendar_exp28a_features, ADDED6[1:3]),
    (calendar_exp29a_features, ADDED6[3:5]),
    (calendar_exp31a_features, ADDED6[5:])
]:

    for col in cols:
        combined_arrays[col] = (
            frame[col].to_numpy(dtype=np.float32)
        )

for col in CONTROLS9:
    combined_arrays[col] = (
        calendar_exp28a_features[col]
        .to_numpy(dtype=np.float32)
    )

# Build consolidated array in one operation.
all_names = CORE31 + ADDED6 + CONTROLS9

matrix = np.column_stack([
    combined_arrays[c]
    for c in all_names
]).astype(np.float32)

matrix[~np.isfinite(matrix)] = np.nan

usdinr_rebuilt_37_research = pd.concat(
    [
        keys_frame,
        pd.DataFrame(matrix, columns=all_names)
    ],
    axis=1
)

# ------------------------------------------------------------
# 11. Validation
# ------------------------------------------------------------

assert len(all_names) == 46
assert len(set(all_names)) == 46

assert (
    usdinr_rebuilt_37_research
    .columns.is_unique
)

assert (
    ~usdinr_rebuilt_37_research[KEYS]
    .duplicated()
).all()

validity = (
    usdinr_rebuilt_37_research[all_names]
    .notna()
    .mean()
    .mul(100)
)

print("\n===== REBUILT FEATURE COVERAGE =====")

print(
    validity.loc[
        ADDED6 + CONTROLS9
    ].round(4).to_string()
)

print("\n===== FULL BASELINE =====")

complete_rows = (
    np.isfinite(matrix).all(axis=1)
)

print("Total rows:", len(matrix))
print("Engineered features:", 37)
print("Controls:", 9)
print("Total model inputs:", 46)

print(
    "Fully complete rows:",
    int(complete_rows.sum())
)

print(
    "Fully complete coverage:",
    f"{100 * complete_rows.mean():.4f}%"
)

# ------------------------------------------------------------
# 12. Save reusable feature files
# ------------------------------------------------------------

calendar_exp24a_features.to_parquet(
    "calendar_exp24a_rebuilt.parquet",
    index=False
)

calendar_exp28a_features.to_parquet(
    "calendar_exp28a_rebuilt.parquet",
    index=False
)

calendar_exp29a_features.to_parquet(
    "calendar_exp29a_rebuilt.parquet",
    index=False
)

calendar_exp31a_features.to_parquet(
    "calendar_exp31a_rebuilt.parquet",
    index=False
)

# Explicit 'rebuilt' label to avoid confusing this
# artifact with the verified historical 37-feature set.
usdinr_rebuilt_37_research.to_parquet(
    "usdinr_37_features_rebuilt_research.parquet",
    index=False
)

print("\n===== SAVED =====")

print(
    "calendar_exp24a_rebuilt.parquet"
)
print(
    "calendar_exp28a_rebuilt.parquet"
)
print(
    "calendar_exp29a_rebuilt.parquet"
)
print(
    "calendar_exp31a_rebuilt.parquet"
)
print(
    "usdinr_37_features_rebuilt_research.parquet"
)

print("\nHoldout dates excluded: Oct 21-23")
print("Core registry remains 37 provisional features.")
print("Reconstruction parity not yet verified.")

del (
    B, A, BQ, AQ,
    matrix, combined_arrays, scratch
)

gc.collect()


===== RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Latest date: 2026-10-19

===== CORE ALIGNMENT =====
Core key match: 100.0000%

===== REBUILT FEATURE COVERAGE =====
quote_retreat_imbalance_30_24a     99.8989
slope_asymmetry_28a               100.0000
curvature_asymmetry_28a           100.0000
weighted_gap_asymmetry_29a        100.0000
fragility_asymmetry_29a           100.0000
relative_migration_60_31a          99.7909
queue_imbalance_28a               100.0000
depth_imbalance_28a               100.0000
log_total_depth_28a               100.0000
spread_bps_28a                    100.0000
bid_l1_concentration_28a          100.0000
ask_l1_concentration_28a          100.0000
momentum_30_28a                    99.8955
momentum_60_28a                    99.7909
clock_seconds                     100.0000

===== FULL BASELINE =====
Total rows: 459122
Engineered features: 37
Controls: 9
Total model inputs: 46
Fully complete rows: 402979
Fully complete coverage: 87.7717%

===== SAVED =====

27

In [20]:
required = [
    "core_31",
    "calendar_exp24a_features",
    "calendar_exp28a_features",
    "calendar_exp29a_features",
    "calendar_exp31a_features",
    "calendar_exp34a_features"
]

for name in required:
    print(
        name,
        "READY" if name in globals() else "MISSING"
    )

core_31 READY
calendar_exp24a_features READY
calendar_exp28a_features READY
calendar_exp29a_features READY
calendar_exp31a_features READY
calendar_exp34a_features READY


In [21]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# REBUILD RESEARCH FEATURE FRAMES
#
# Input:
#   df_1s      : updated 1-second order-book data
#   core_31    : loaded from usdinr_31_features.parquet
#
# Outputs:
#   calendar_exp24a_features
#   calendar_exp28a_features
#   calendar_exp29a_features
#   calendar_exp31a_features
#   usdinr_rebuilt_37_research
#
# IMPORTANT:
# Quote retreat, slope, curvature, and nine controls
# are RECONSTRUCTIONS, not verified original formulas.
#
# Do not interpret results as an exact historical
# 37-feature-baseline replication until parity verified.
# ============================================================

KEYS = ["symbol", "ts_sec"]
RESEARCH_END = pd.Timestamp("2026-10-19").date()
MIN_COVERAGE = 0.97
EPS = 1e-12

CORE31 = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

ADDED6 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

assert len(CORE31) == 31
assert len(ADDED6) == 6
assert len(CONTROLS9) == 9

# ------------------------------------------------------------
# 1. Research-only raw order book
# ------------------------------------------------------------

prices = (
    [f"bid_p{i}" for i in range(1, 6)]
    + [f"ask_p{i}" for i in range(1, 6)]
)

quantities = (
    [f"bid_q{i}" for i in range(1, 6)]
    + [f"ask_q{i}" for i in range(1, 6)]
)

required = (
    KEYS + prices + quantities + ["is_observed"]
)

missing = [
    c for c in required if c not in df_1s.columns
]

if missing:
    raise ValueError(f"Missing raw columns: {missing}")

x = df_1s[required].copy()

x["symbol"] = x["symbol"].astype(str)
x["ts_sec"] = pd.to_numeric(
    x["ts_sec"], errors="raise"
).astype("int64")

ist = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ist.dt.date

x["clock_seconds"] = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

# Explicit chronological research boundary.
x = x.loc[
    (x["date"] <= RESEARCH_END)
    & (x["date"].astype(str) != "2026-09-28")
    & x["clock_seconds"].between(0, 28799)
].copy()

coverage = (
    x.groupby(["symbol", "date"])["ts_sec"]
    .transform("nunique")
)

x = x.loc[
    coverage >= MIN_COVERAGE * 28800
].copy()

x = (
    x.sort_values(
        ["symbol", "date", "ts_sec"]
    )
    .reset_index(drop=True)
)

if x.duplicated(KEYS).any():
    raise ValueError("Duplicate symbol/timestamp keys")

if x.empty:
    raise ValueError("No eligible research sessions")

for col in prices + quantities:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

print("===== RESEARCH SAMPLE =====")
print("Rows:", len(x))
print(
    "Sessions:",
    x.groupby(["symbol", "date"]).ngroups
)
print("Latest date:", x["date"].max())

# ------------------------------------------------------------
# 2. Book matrices
# ------------------------------------------------------------

B = x[
    [f"bid_p{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

A = x[
    [f"ask_p{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

BQ = x[
    [f"bid_q{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

AQ = x[
    [f"ask_q{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

n = len(x)

valid_book = (
    np.isfinite(B).all(axis=1)
    & np.isfinite(A).all(axis=1)
    & np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (B > 0).all(axis=1)
    & (A > 0).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (A[:, 0] >= B[:, 0])
)

mid = (B[:, 0] + A[:, 0]) / 2
mid[~valid_book] = np.nan

bid_total = BQ.sum(axis=1)
ask_total = AQ.sum(axis=1)

bid_total[~valid_book] = np.nan
ask_total[~valid_book] = np.nan

total_depth = bid_total + ask_total

# ------------------------------------------------------------
# 3. Exact timestamp lookup
# ------------------------------------------------------------

idx = pd.MultiIndex.from_frame(
    x[["symbol", "date", "ts_sec"]]
)

symbols = x["symbol"]
dates = x["date"]
timestamps = x["ts_sec"].to_numpy(
    dtype=np.int64
)

def lookup_exact(array, offset):

    lookup_idx = pd.MultiIndex.from_arrays(
        [
            symbols,
            dates,
            timestamps + offset
        ],
        names=["symbol", "date", "ts_sec"]
    )

    positions = idx.get_indexer(lookup_idx)
    found = positions >= 0

    result = np.full(
        n, np.nan, dtype=np.float64
    )

    arr = np.asarray(
        array, dtype=np.float64
    )

    result[found] = arr[positions[found]]

    return result

# ------------------------------------------------------------
# 4. Experiment 24 — Quote retreat reconstruction
#
# Counts observed outward movements of the best
# bid/ask over a trailing 30-snapshot window.
#
# This is NOT VERIFIED as the original 24A formula.
# ------------------------------------------------------------

bid_prev = lookup_exact(B[:, 0], -1)
ask_prev = lookup_exact(A[:, 0], -1)

is_observed = (
    x["is_observed"].eq(1).to_numpy()
)

# Bid retreats when best bid falls.
bid_retreat = (
    is_observed
    & np.isfinite(bid_prev)
    & (B[:, 0] < bid_prev)
).astype(float)

# Ask retreats when best ask rises.
ask_retreat = (
    is_observed
    & np.isfinite(ask_prev)
    & (A[:, 0] > ask_prev)
).astype(float)

scratch = pd.DataFrame({
    "symbol": x["symbol"],
    "date": x["date"],
    "bid_retreat": bid_retreat,
    "ask_retreat": ask_retreat
})

g = scratch.groupby(
    ["symbol", "date"], sort=False
)

bid_retreat_30 = g["bid_retreat"].transform(
    lambda s: s.rolling(30, min_periods=30).sum()
)

ask_retreat_30 = g["ask_retreat"].transform(
    lambda s: s.rolling(30, min_periods=30).sum()
)

# Positive = greater ask retreat than bid retreat.
quote_retreat_30 = (
    ask_retreat_30.to_numpy(dtype=float)
    - bid_retreat_30.to_numpy(dtype=float)
) / (
    ask_retreat_30.to_numpy(dtype=float)
    + bid_retreat_30.to_numpy(dtype=float)
    + 1.0
)

# ------------------------------------------------------------
# 5. Experiment 28 — Book slope and curvature
#
# Reconstructed geometry:
# Fit log(1+quantity) to a centered quadratic
# over book levels 1..5 on each side.
#
# slope = first-order coefficient difference
# curvature = second-order coefficient difference
#
# NOT VERIFIED as original 28A implementation.
# ------------------------------------------------------------

level = np.arange(1, 6, dtype=float)

u = level - level.mean()
v = u**2
v_center = v - v.mean()

# u and centered v are orthogonal for levels 1..5.
u_denom = np.sum(u**2)
v_denom = np.sum(v_center**2)

log_bq = np.log1p(
    np.maximum(BQ, 0)
)

log_aq = np.log1p(
    np.maximum(AQ, 0)
)

bid_slope = (
    log_bq @ u
) / u_denom

ask_slope = (
    log_aq @ u
) / u_denom

bid_curvature = (
    log_bq @ v_center
) / v_denom

ask_curvature = (
    log_aq @ v_center
) / v_denom

slope_asym = bid_slope - ask_slope

curvature_asym = (
    bid_curvature - ask_curvature
)

slope_asym[~valid_book] = np.nan
curvature_asym[~valid_book] = np.nan

# ------------------------------------------------------------
# 6. Experiment 29 — Weighted gaps and fragility
#
# Reconstructed from established mathematical
# definitions of Exp29:
#
# Adjacent gaps in bps:
#   bid_i - bid_(i+1)
#   ask_(i+1) - ask_i
#
# w_i = exp(-0.7*i), normalized over i=0..3
# ------------------------------------------------------------

bid_gaps = (
    1e4 * (B[:, :-1] - B[:, 1:])
    / mid[:, None]
)

ask_gaps = (
    1e4 * (A[:, 1:] - A[:, :-1])
    / mid[:, None]
)

weights = np.exp(
    -0.7 * np.arange(4)
)

weights = weights / weights.sum()

weighted_gap_asym = (
    (bid_gaps - ask_gaps) @ weights
)

bid_fragility = (
    bid_gaps
    / np.sqrt(1.0 + BQ[:, :-1])
)

ask_fragility = (
    ask_gaps
    / np.sqrt(1.0 + AQ[:, :-1])
)

fragility_asym = (
    (bid_fragility - ask_fragility)
    @ weights
)

weighted_gap_asym[~valid_book] = np.nan
fragility_asym[~valid_book] = np.nan

# ------------------------------------------------------------
# 7. Experiment 31 — Relative migration
#
# Depth-weighted center of mass of distances
# from each side's best quote, in bps.
#
# Uses exact 60-second historical lookback.
# ------------------------------------------------------------

bid_distance = (
    1e4 * (
        B[:, [0]] - B
    ) / mid[:, None]
)

ask_distance = (
    1e4 * (
        A - A[:, [0]]
    ) / mid[:, None]
)

with np.errstate(
    divide="ignore", invalid="ignore"
):

    mu_bid = (
        (bid_distance * BQ).sum(axis=1)
        / bid_total
    )

    mu_ask = (
        (ask_distance * AQ).sum(axis=1)
        / ask_total
    )

mu_bid[~valid_book] = np.nan
mu_ask[~valid_book] = np.nan

mu_bid_lag = lookup_exact(mu_bid, -60)
mu_ask_lag = lookup_exact(mu_ask, -60)

delta_bid = mu_bid - mu_bid_lag
delta_ask = mu_ask - mu_ask_lag

relative_migration = (
    delta_bid - delta_ask
) / (
    np.abs(delta_bid)
    + np.abs(delta_ask)
    + EPS
)

# ------------------------------------------------------------
# 8. Experiment 28 — Nine controls
#
# Explicit reconstruction, not verified parity.
# ------------------------------------------------------------

with np.errstate(
    divide="ignore", invalid="ignore"
):

    queue_imbalance = (
        BQ[:, 0] - AQ[:, 0]
    ) / (
        BQ[:, 0] + AQ[:, 0] + EPS
    )

    depth_imbalance = (
        bid_total - ask_total
    ) / (
        total_depth + EPS
    )

    log_total_depth = np.log1p(
        total_depth
    )

    spread_bps = (
        1e4 * (A[:, 0] - B[:, 0]) / mid
    )

    bid_l1_concentration = (
        BQ[:, 0] / (bid_total + EPS)
    )

    ask_l1_concentration = (
        AQ[:, 0] / (ask_total + EPS)
    )

    past_mid30 = lookup_exact(mid, -30)
    past_mid60 = lookup_exact(mid, -60)

    momentum_30 = (
        1e4 * np.log(mid / past_mid30)
    )

    momentum_60 = (
        1e4 * np.log(mid / past_mid60)
    )

controls = {
    "queue_imbalance_28a": queue_imbalance,
    "depth_imbalance_28a": depth_imbalance,
    "log_total_depth_28a": log_total_depth,
    "spread_bps_28a": spread_bps,
    "bid_l1_concentration_28a": bid_l1_concentration,
    "ask_l1_concentration_28a": ask_l1_concentration,
    "momentum_30_28a": momentum_30,
    "momentum_60_28a": momentum_60,
    "clock_seconds": x["clock_seconds"].to_numpy()
}

for name in controls:

    values = np.asarray(
        controls[name], dtype=np.float64
    )

    if name != "clock_seconds":
        values[~valid_book] = np.nan

    controls[name] = values

# ------------------------------------------------------------
# 9. Construct original frame names
# ------------------------------------------------------------

keys_frame = x[KEYS].copy()

def make_feature_frame(feature_dict):

    out = keys_frame.copy()

    for name, values in feature_dict.items():

        arr = np.asarray(
            values, dtype=np.float64
        ).copy()

        arr[~np.isfinite(arr)] = np.nan

        out[name] = arr.astype(np.float32)

    return out

calendar_exp24a_features = make_feature_frame({
    "quote_retreat_imbalance_30_24a":
        quote_retreat_30
})

calendar_exp28a_features = make_feature_frame({
    "slope_asymmetry_28a": slope_asym,
    "curvature_asymmetry_28a": curvature_asym,
    **controls
})

calendar_exp29a_features = make_feature_frame({
    "weighted_gap_asymmetry_29a":
        weighted_gap_asym,
    "fragility_asymmetry_29a":
        fragility_asym
})

calendar_exp31a_features = make_feature_frame({
    "relative_migration_60_31a":
        relative_migration
})

# ------------------------------------------------------------
# 10. Align original 31 features
#
# Exact key matching, no assumptions about row order.
# ------------------------------------------------------------

core_index = pd.MultiIndex.from_arrays(
    [
        core_31["symbol"].astype(str),
        pd.to_numeric(
            core_31["ts_sec"],
            errors="raise"
        ).astype("int64")
    ],
    names=KEYS
)

if core_index.has_duplicates:
    raise ValueError(
        "Duplicate symbol/timestamp keys in core_31"
    )

target_index = pd.MultiIndex.from_frame(
    keys_frame
)

core_positions = core_index.get_indexer(
    target_index
)

matched = core_positions >= 0

print("\n===== CORE ALIGNMENT =====")

print(
    "Core key match:",
    f"{100 * matched.mean():.4f}%"
)

if not matched.all():
    raise ValueError(
        "Some research rows are absent from core_31"
    )

combined_arrays = {}

for col in CORE31:

    if col not in core_31.columns:
        raise ValueError(f"Core column missing: {col}")

    source_values = pd.to_numeric(
        core_31[col],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    combined_arrays[col] = source_values[
        core_positions
    ]

for frame, cols in [
    (calendar_exp24a_features, ADDED6[:1]),
    (calendar_exp28a_features, ADDED6[1:3]),
    (calendar_exp29a_features, ADDED6[3:5]),
    (calendar_exp31a_features, ADDED6[5:])
]:

    for col in cols:
        combined_arrays[col] = (
            frame[col].to_numpy(dtype=np.float32)
        )

for col in CONTROLS9:
    combined_arrays[col] = (
        calendar_exp28a_features[col]
        .to_numpy(dtype=np.float32)
    )

# Build consolidated array in one operation.
all_names = CORE31 + ADDED6 + CONTROLS9

matrix = np.column_stack([
    combined_arrays[c]
    for c in all_names
]).astype(np.float32)

matrix[~np.isfinite(matrix)] = np.nan

usdinr_rebuilt_37_research = pd.concat(
    [
        keys_frame,
        pd.DataFrame(matrix, columns=all_names)
    ],
    axis=1
)

# ------------------------------------------------------------
# 11. Validation
# ------------------------------------------------------------

assert len(all_names) == 46
assert len(set(all_names)) == 46

assert (
    usdinr_rebuilt_37_research
    .columns.is_unique
)

assert (
    ~usdinr_rebuilt_37_research[KEYS]
    .duplicated()
).all()

validity = (
    usdinr_rebuilt_37_research[all_names]
    .notna()
    .mean()
    .mul(100)
)

print("\n===== REBUILT FEATURE COVERAGE =====")

print(
    validity.loc[
        ADDED6 + CONTROLS9
    ].round(4).to_string()
)

print("\n===== FULL BASELINE =====")

complete_rows = (
    np.isfinite(matrix).all(axis=1)
)

print("Total rows:", len(matrix))
print("Engineered features:", 37)
print("Controls:", 9)
print("Total model inputs:", 46)

print(
    "Fully complete rows:",
    int(complete_rows.sum())
)

print(
    "Fully complete coverage:",
    f"{100 * complete_rows.mean():.4f}%"
)

# ------------------------------------------------------------
# 12. Save reusable feature files
# ------------------------------------------------------------

calendar_exp24a_features.to_parquet(
    "calendar_exp24a_rebuilt.parquet",
    index=False
)

calendar_exp28a_features.to_parquet(
    "calendar_exp28a_rebuilt.parquet",
    index=False
)

calendar_exp29a_features.to_parquet(
    "calendar_exp29a_rebuilt.parquet",
    index=False
)

calendar_exp31a_features.to_parquet(
    "calendar_exp31a_rebuilt.parquet",
    index=False
)

# Explicit 'rebuilt' label to avoid confusing this
# artifact with the verified historical 37-feature set.
usdinr_rebuilt_37_research.to_parquet(
    "usdinr_37_features_rebuilt_research.parquet",
    index=False
)

print("\n===== SAVED =====")

print(
    "calendar_exp24a_rebuilt.parquet"
)
print(
    "calendar_exp28a_rebuilt.parquet"
)
print(
    "calendar_exp29a_rebuilt.parquet"
)
print(
    "calendar_exp31a_rebuilt.parquet"
)
print(
    "usdinr_37_features_rebuilt_research.parquet"
)

print("\nHoldout dates excluded: Oct 21-23")
print("Core registry remains 37 provisional features.")
print("Reconstruction parity not yet verified.")

del (
    B, A, BQ, AQ,
    matrix, combined_arrays, scratch
)

gc.collect()


===== RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Latest date: 2026-10-19

===== CORE ALIGNMENT =====
Core key match: 100.0000%

===== REBUILT FEATURE COVERAGE =====
quote_retreat_imbalance_30_24a     99.8989
slope_asymmetry_28a               100.0000
curvature_asymmetry_28a           100.0000
weighted_gap_asymmetry_29a        100.0000
fragility_asymmetry_29a           100.0000
relative_migration_60_31a          99.7909
queue_imbalance_28a               100.0000
depth_imbalance_28a               100.0000
log_total_depth_28a               100.0000
spread_bps_28a                    100.0000
bid_l1_concentration_28a          100.0000
ask_l1_concentration_28a          100.0000
momentum_30_28a                    99.8955
momentum_60_28a                    99.7909
clock_seconds                     100.0000

===== FULL BASELINE =====
Total rows: 459122
Engineered features: 37
Controls: 9
Total model inputs: 46
Fully complete rows: 402979
Fully complete coverage: 87.7717%

===== SAVED =====

29

In [22]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import gc

# ============================================================
# EXPERIMENT 34A-2
#
# FULL 37-FEATURE ABLATION:
#  A: baseline
#  B: baseline + bid recovery + bid event presence
#  C: baseline + ask recovery + ask event presence
#  D: baseline + both recoveries + both presence flags
#
# TARGETS:
#  Future signed midprice log returns:
#  30s, 60s, 300s
#
# Research sessions only, through 2026-10-19.
# Never uses October 21-23 holdout sessions.
#
# Outputs:
#  exp34a2_summary
#  exp34a2_daily
#  exp34a2_daily_gains
#  exp34a2_regime_summary
#  exp34a2_importance
#
# No features automatically promoted.
# ============================================================

# ------------------------------------------------------------
# 0. Explicit existing 31-feature registry
# ------------------------------------------------------------

CORE31_34A2 = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

ADDED6_34A2 = {
    "quote_retreat_imbalance_30_24a":
        "calendar_exp24a_features",

    "slope_asymmetry_28a":
        "calendar_exp28a_features",

    "curvature_asymmetry_28a":
        "calendar_exp28a_features",

    "weighted_gap_asymmetry_29a":
        "calendar_exp29a_features",

    "fragility_asymmetry_29a":
        "calendar_exp29a_features",

    "relative_migration_60_31a":
        "calendar_exp31a_features"
}

CONTROL9_34A2 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

RECOVERY_34A2 = [
    "l5_bid_recovery_34a",
    "l5_ask_recovery_34a"
]

BASE_FEATURES_34A2 = (
    CORE31_34A2
    + list(ADDED6_34A2)
    + CONTROL9_34A2
)

assert len(CORE31_34A2) == 31
assert len(BASE_FEATURES_34A2) == 46
assert len(set(BASE_FEATURES_34A2)) == 46

HORIZONS_34A2 = [30, 60, 300]

MIN_TRAIN_DAYS_34A2 = 5
TRAIN_STRIDE_34A2 = 10

KEYS_34A2 = ["symbol", "ts_sec"]

PARAMS_34A2 = dict(
    objective="regression",
    n_estimators=180,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=5,
    min_child_samples=200,
    reg_lambda=2,
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

# ------------------------------------------------------------
# 1. Validate required research objects
# ------------------------------------------------------------

required_objects = [
    "core_31",
    "calendar_exp24a_features",
    "calendar_exp28a_features",
    "calendar_exp29a_features",
    "calendar_exp31a_features",
    "calendar_exp34a_features"
]

for name in required_objects:
    if name not in globals():
        raise ValueError(
            f"Missing {name}. "
            "Run or load its research feature frame first."
        )

SOURCE_MAP_34A2 = {
    "core_31": CORE31_34A2,

    "calendar_exp24a_features": [
        "quote_retreat_imbalance_30_24a"
    ],

    "calendar_exp28a_features": [
        "slope_asymmetry_28a",
        "curvature_asymmetry_28a"
    ] + CONTROL9_34A2,

    "calendar_exp29a_features": [
        "weighted_gap_asymmetry_29a",
        "fragility_asymmetry_29a"
    ],

    "calendar_exp31a_features": [
        "relative_migration_60_31a"
    ]
}

for source_name, columns in SOURCE_MAP_34A2.items():

    source = globals()[source_name]

    missing = [
        c for c in KEYS_34A2 + columns
        if c not in source.columns
    ]

    if missing:
        raise ValueError(
            f"{source_name} missing: {missing}"
        )

# ------------------------------------------------------------
# 2. Use Experiment 34A sample and targets
#
# Already restricted to original research sessions.
# ------------------------------------------------------------

target_columns = [
    f"future_return_{h}_34a"
    for h in HORIZONS_34A2
]

needed_34a = (
    ["symbol", "date", "ts_sec"]
    + RECOVERY_34A2
    + target_columns
)

missing = [
    c for c in needed_34a
    if c not in calendar_exp34a_features.columns
]

if missing:
    raise ValueError(
        f"Experiment 34A missing: {missing}"
    )

base = calendar_exp34a_features[
    needed_34a
].copy()

base["symbol"] = base["symbol"].astype(str)

base["ts_sec"] = pd.to_numeric(
    base["ts_sec"], errors="raise"
).astype("int64")

base["date"] = pd.to_datetime(
    base["date"]
).dt.date

base = (
    base.loc[
        base["date"].astype(str) <= "2026-10-19"
    ]
    .sort_values(["date", "symbol", "ts_sec"])
    .reset_index(drop=True)
)

if base.duplicated(KEYS_34A2).any():
    raise ValueError("Duplicate target keys")

N = len(base)

if N == 0:
    raise ValueError("Empty research sample")

print("===== 34A-2 SAMPLE =====")
print("Rows:", N)
print(
    "Trading sessions:",
    base.groupby(["symbol", "date"]).ngroups
)
print("First date:", base["date"].min())
print("Last date:", base["date"].max())

# ------------------------------------------------------------
# 3. Memory-efficient feature alignment
#
# Reuse positional alignment per source frame.
# No repeated DataFrame merges.
# ------------------------------------------------------------

target_index = pd.MultiIndex.from_frame(
    base[KEYS_34A2]
)

aligned = {}

for source_name, columns in SOURCE_MAP_34A2.items():

    source = globals()[source_name]

    source_index = pd.MultiIndex.from_arrays(
        [
            source["symbol"].astype(str),
            pd.to_numeric(
                source["ts_sec"], errors="raise"
            ).astype("int64")
        ],
        names=KEYS_34A2
    )

    if source_index.has_duplicates:
        raise ValueError(
            f"Duplicate keys in {source_name}"
        )

    positions = source_index.get_indexer(
        target_index
    )

    found = positions >= 0

    match_rate = float(found.mean())

    print(
        source_name,
        "key match:",
        f"{100 * match_rate:.4f}%"
    )

    if match_rate < 0.99:
        raise ValueError(
            f"Poor alignment with {source_name}"
        )

    for col in columns:

        values = pd.to_numeric(
            source[col],
            errors="coerce"
        ).to_numpy(dtype=np.float32)

        output = np.full(
            N, np.nan, dtype=np.float32
        )

        output[found] = values[
            positions[found]
        ]

        aligned[col] = output

    del source_index, positions, found
    gc.collect()

# ------------------------------------------------------------
# 4. Candidate recovery variables and presence flags
#
# Key principle:
#   NaN recovery = no qualifying event
#   0 recovery   = event occurred but no net recovery
#
# Add flags rather than replacing all NaNs with zero.
# ------------------------------------------------------------

bid_recovery = pd.to_numeric(
    base["l5_bid_recovery_34a"],
    errors="coerce"
).to_numpy(dtype=np.float32)

ask_recovery = pd.to_numeric(
    base["l5_ask_recovery_34a"],
    errors="coerce"
).to_numpy(dtype=np.float32)

bid_recovery[
    ~np.isfinite(bid_recovery)
] = np.nan

ask_recovery[
    ~np.isfinite(ask_recovery)
] = np.nan

bid_presence = np.isfinite(
    bid_recovery
).astype(np.float32)

ask_presence = np.isfinite(
    ask_recovery
).astype(np.float32)

print("\n===== 34A-2 RECOVERY COVERAGE =====")

print(
    "Bid recovery:",
    f"{100 * bid_presence.mean():.4f}%"
)

print(
    "Ask recovery:",
    f"{100 * ask_presence.mean():.4f}%"
)

print(
    "Either recovery:",
    f"{100 * np.mean((bid_presence + ask_presence) > 0):.4f}%"
)

print(
    "Both recoveries:",
    f"{100 * np.mean((bid_presence + ask_presence) == 2):.4f}%"
)

# ------------------------------------------------------------
# 5. Assemble one compact feature matrix
# ------------------------------------------------------------

extra_features = [
    "l5_bid_recovery_34a",
    "l5_bid_event_present_34a2",
    "l5_ask_recovery_34a",
    "l5_ask_event_present_34a2"
]

for name, values in [
    ("l5_bid_recovery_34a", bid_recovery),
    ("l5_bid_event_present_34a2", bid_presence),
    ("l5_ask_recovery_34a", ask_recovery),
    ("l5_ask_event_present_34a2", ask_presence)
]:
    aligned[name] = values

ALL_COLS = BASE_FEATURES_34A2 + extra_features

X = np.column_stack([
    aligned[c] for c in ALL_COLS
]).astype(np.float32, copy=False)

X[~np.isfinite(X)] = np.nan

column_positions = {
    col: i
    for i, col in enumerate(ALL_COLS)
}

base_idx = list(range(
    len(BASE_FEATURES_34A2)
))

bid_idx = [
    column_positions[
        "l5_bid_recovery_34a"
    ],
    column_positions[
        "l5_bid_event_present_34a2"
    ]
]

ask_idx = [
    column_positions[
        "l5_ask_recovery_34a"
    ],
    column_positions[
        "l5_ask_event_present_34a2"
    ]
]

MODELS_34A2 = {
    "A_full37": base_idx,
    "B_bid_recovery": base_idx + bid_idx,
    "C_ask_recovery": base_idx + ask_idx,
    "D_both_recoveries": (
        base_idx + bid_idx + ask_idx
    )
}

# Baseline-complete sample:
# Do not require finite recovery features.
# LightGBM natively handles NaN.
#
# This ensures every model uses the same full-day rows.
base_complete = np.isfinite(
    X[:, base_idx]
).all(axis=1)

print("\n===== 34A-2 MATCHED SAMPLE =====")

print(
    "Baseline complete:",
    int(base_complete.sum())
)

print(
    "Coverage:",
    f"{100 * base_complete.mean():.4f}%"
)

# ------------------------------------------------------------
# 6. Prepare chronological dates and targets
# ------------------------------------------------------------

dates = base["date"].to_numpy()

timestamps = base["ts_sec"].to_numpy(
    dtype=np.int64
)

unique_dates = sorted(
    pd.unique(dates)
)

if len(unique_dates) <= MIN_TRAIN_DAYS_34A2:
    raise ValueError("Insufficient trading dates")

Y = {}

for h in HORIZONS_34A2:

    col = f"future_return_{h}_34a"

    Y[h] = pd.to_numeric(
        base[col], errors="coerce"
    ).to_numpy(dtype=np.float32)

# Shock-state codes:
# 0 = neither side
# 1 = bid only
# 2 = ask only
# 3 = both
shock_state = (
    bid_presence.astype(np.int8)
    + 2 * ask_presence.astype(np.int8)
)

del aligned
gc.collect()

# ------------------------------------------------------------
# 7. Expanding chronological LightGBM ablation
#
# First five dates used to initialize training.
# Remaining eleven dates evaluated.
#
# Train rows sampled every ten seconds.
# Test rows use all eligible 1-second observations.
# ------------------------------------------------------------

daily_records = []
regime_records = []
importance_records = []

for day_idx in range(
    MIN_TRAIN_DAYS_34A2,
    len(unique_dates)
):

    test_date = unique_dates[day_idx]

    train_days = unique_dates[:day_idx]

    historical_mask = np.isin(
        dates, train_days
    )

    train_base_mask = (
        historical_mask
        & base_complete
        & ((timestamps % TRAIN_STRIDE_34A2) == 0)
    )

    test_base_mask = (
        (dates == test_date)
        & base_complete
    )

    for h in HORIZONS_34A2:

        y = Y[h]

        train_rows = np.flatnonzero(
            train_base_mask & np.isfinite(y)
        )

        test_rows = np.flatnonzero(
            test_base_mask & np.isfinite(y)
        )

        if (
            len(train_rows) < 1000
            or len(test_rows) < 200
        ):
            continue

        y_train = y[train_rows]

        y_test = y[
            test_rows
        ].astype(np.float64)

        test_states = shock_state[test_rows]

        for model_name, selected_idx in (
            MODELS_34A2.items()
        ):

            model = lgb.LGBMRegressor(
                **PARAMS_34A2
            )

            X_train = X[np.ix_(
                train_rows, selected_idx
            )]

            X_test = X[np.ix_(
                test_rows, selected_idx
            )]

            model.fit(X_train, y_train)

            pred = model.predict(
                X_test
            )

            err = y_test - pred

            abs_err = np.abs(err)
            sq_err = err**2

            daily_records.append({
                "date": test_date,
                "horizon": h,
                "model": model_name,
                "rows": len(test_rows),
                "sse": float(sq_err.sum()),
                "sae": float(abs_err.sum()),
                "mse": float(sq_err.mean()),
                "mae": float(abs_err.mean())
            })

            # Compare prediction quality separately
            # in different shock states.
            for state_id, state_name in [
                (0, "neither_shock"),
                (1, "bid_only"),
                (2, "ask_only"),
                (3, "both_shocks"),
                (4, "any_shock")
            ]:

                if state_id == 4:
                    mask = test_states != 0
                else:
                    mask = test_states == state_id

                n_regime = int(mask.sum())

                if n_regime == 0:
                    continue

                regime_records.append({
                    "date": test_date,
                    "horizon": h,
                    "model": model_name,
                    "regime": state_name,
                    "rows": n_regime,
                    "sse": float(
                        sq_err[mask].sum()
                    ),
                    "sae": float(
                        abs_err[mask].sum()
                    )
                })

            # Feature gain importance:
            # use the actual selected column labels.
            if model_name != "A_full37":

                gains = model.booster_.feature_importance(
                    importance_type="gain"
                )

                total_gain = float(gains.sum())

                for local_idx, full_idx in enumerate(
                    selected_idx
                ):

                    feature_name = ALL_COLS[full_idx]

                    if feature_name not in extra_features:
                        continue

                    importance_records.append({
                        "date": test_date,
                        "horizon": h,
                        "model": model_name,
                        "feature": feature_name,
                        "gain_share_pct": (
                            100 * float(
                                gains[local_idx]
                            ) / total_gain
                            if total_gain > 0
                            else 0.0
                        )
                    })

            del (
                model, X_train, X_test,
                pred, err, abs_err, sq_err
            )

    print("Completed:", test_date)

# ------------------------------------------------------------
# 8. Organize daily results
# ------------------------------------------------------------

exp34a2_daily = pd.DataFrame(daily_records)
exp34a2_regime_daily = pd.DataFrame(regime_records)
exp34a2_importance = pd.DataFrame(
    importance_records
)

if exp34a2_daily.empty:
    raise ValueError("No model evaluations completed")

comparisons = {
    "bid_vs_full37": "B_bid_recovery",
    "ask_vs_full37": "C_ask_recovery",
    "both_vs_full37": "D_both_recoveries",
    "both_vs_bid": "D_both_recoveries",
    "both_vs_ask": "D_both_recoveries"
}

reference_models = {
    "bid_vs_full37": "A_full37",
    "ask_vs_full37": "A_full37",
    "both_vs_full37": "A_full37",
    "both_vs_bid": "B_bid_recovery",
    "both_vs_ask": "C_ask_recovery"
}

summary_records = []
daily_gain_records = []

for h, part in exp34a2_daily.groupby("horizon"):

    pooled = part.groupby("model")[
        ["sse", "sae"]
    ].sum()

    mse_wide = part.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    mae_wide = part.pivot(
        index="date",
        columns="model",
        values="mae"
    )

    for comparison, candidate in (
        comparisons.items()
    ):

        reference = reference_models[comparison]

        mse_daily = 100 * (
            1 - mse_wide[candidate]
            / mse_wide[reference]
        )

        mae_daily = 100 * (
            1 - mae_wide[candidate]
            / mae_wide[reference]
        )

        pooled_mse_gain = 100 * (
            1 - pooled.loc[candidate, "sse"]
            / pooled.loc[reference, "sse"]
        )

        pooled_mae_gain = 100 * (
            1 - pooled.loc[candidate, "sae"]
            / pooled.loc[reference, "sae"]
        )

        summary_records.append({
            "horizon": h,
            "comparison": comparison,
            "days": len(mse_daily),
            "pooled_mse_gain_pct": pooled_mse_gain,
            "pooled_mae_gain_pct": pooled_mae_gain,
            "mse_win_days": int(
                (mse_daily > 0).sum()
            ),
            "mae_win_days": int(
                (mae_daily > 0).sum()
            ),
            "median_daily_mse_gain_pct":
                float(mse_daily.median()),
            "worst_daily_mse_gain_pct":
                float(mse_daily.min())
        })

        for date in mse_daily.index:

            daily_gain_records.append({
                "date": date,
                "horizon": h,
                "comparison": comparison,
                "mse_gain_pct": float(
                    mse_daily.loc[date]
                ),
                "mae_gain_pct": float(
                    mae_daily.loc[date]
                )
            })

exp34a2_summary = pd.DataFrame(
    summary_records
)

exp34a2_daily_gains = pd.DataFrame(
    daily_gain_records
)

# ------------------------------------------------------------
# 9. Regime-specific pooled comparisons
#
# Purely diagnostic:
# regime classification is based on the
# historical information available at time t.
#
# Does not alter training or sample selection.
# ------------------------------------------------------------

regime_summary_records = []

regime_pooled = (
    exp34a2_regime_daily
    .groupby(
        ["horizon", "regime", "model"]
    )[["rows", "sse", "sae"]]
    .sum()
)

for h in HORIZONS_34A2:

    for regime in [
        "neither_shock",
        "bid_only",
        "ask_only",
        "both_shocks",
        "any_shock"
    ]:

        for comparison, candidate in (
            comparisons.items()
        ):

            reference = reference_models[
                comparison
            ]

            candidate_key = (
                h, regime, candidate
            )

            reference_key = (
                h, regime, reference
            )

            if (
                candidate_key not in regime_pooled.index
                or reference_key
                    not in regime_pooled.index
            ):
                continue

            a = regime_pooled.loc[
                candidate_key
            ]

            b = regime_pooled.loc[
                reference_key
            ]

            if a["rows"] != b["rows"]:
                raise ValueError(
                    "Regime sample mismatch"
                )

            regime_summary_records.append({
                "horizon": h,
                "regime": regime,
                "comparison": comparison,
                "rows": int(a["rows"]),
                "mse_gain_pct": 100 * (
                    1 - a["sse"] / b["sse"]
                ),
                "mae_gain_pct": 100 * (
                    1 - a["sae"] / b["sae"]
                )
            })

exp34a2_regime_summary = pd.DataFrame(
    regime_summary_records
)

# ------------------------------------------------------------
# 10. Print outputs
# ------------------------------------------------------------

print("\n===== 34A-2 FULL37 ABLATION =====")

print(
    exp34a2_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 34A-2 DAILY MSE GAINS =====")

for comparison in comparisons:

    part = exp34a2_daily_gains.loc[
        exp34a2_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print("\n", comparison)

    print(
        part.pivot(
            index="date",
            columns="horizon",
            values="mse_gain_pct"
        )
        .round(6)
        .to_string()
    )

print("\n===== 34A-2 DAILY MAE GAINS =====")

for comparison in [
    "bid_vs_full37",
    "ask_vs_full37",
    "both_vs_full37"
]:

    part = exp34a2_daily_gains.loc[
        exp34a2_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print("\n", comparison)

    print(
        part.pivot(
            index="date",
            columns="horizon",
            values="mae_gain_pct"
        )
        .round(6)
        .to_string()
    )

print("\n===== 34A-2 REGIME PERFORMANCE =====")

print(
    exp34a2_regime_summary.loc[
        exp34a2_regime_summary[
            "comparison"
        ].isin([
            "bid_vs_full37",
            "ask_vs_full37",
            "both_vs_full37"
        ])
    ]
    .round(6)
    .to_string(index=False)
)

print("\n===== 34A-2 FEATURE IMPORTANCE =====")

print(
    exp34a2_importance
    .groupby([
        "horizon", "model", "feature"
    ])["gain_share_pct"]
    .agg(["mean", "median"])
    .round(6)
    .to_string()
)

print("\n===== EXPERIMENT 34A-2 COMPLETE =====")
print("Baseline engineered features: 37")
print("Baseline total model inputs: 46")
print("New features promoted: 0")
print("October 21-23 untouched.")

gc.collect()


===== 34A-2 SAMPLE =====
Rows: 459122
Trading sessions: 16
First date: 2026-09-23
Last date: 2026-10-19
core_31 key match: 100.0000%
calendar_exp24a_features key match: 100.0000%
calendar_exp28a_features key match: 100.0000%
calendar_exp29a_features key match: 100.0000%
calendar_exp31a_features key match: 100.0000%

===== 34A-2 RECOVERY COVERAGE =====
Bid recovery: 12.8857%
Ask recovery: 13.4472%
Either recovery: 23.2110%
Both recoveries: 3.1218%

===== 34A-2 MATCHED SAMPLE =====
Baseline complete: 402979
Coverage: 87.7717%
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 34A-2 FULL37 ABLATION =====
 horizon     comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  mse_win_days  mae_win_days  median_daily_mse_gain_pct  worst_daily_mse_gain_pct
      30  bid_vs_full37    11           

28

In [23]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# EXPERIMENT 34B — EVENT-ANCHORED L5 RECOVERY
#
# Required:
#   df_1s
#
# Optional redundancy sources:
#   core_31
#   calendar_exp34a_features
#
# Outputs:
#   calendar_exp34b_features
#   calendar_exp34b_summary
#   calendar_exp34b_daily
#   calendar_exp34b_redundancy
#   calendar_exp34b_event_summary
#
# Research only: original 16 sessions
# No October 21-23 holdout access
# No feature promotion
# ============================================================

LOOKBACK_34B = 60
SHOCK_THRESHOLD_34B = 0.20
RECOVERY_CAP_34B = 2.0
HORIZONS_34B = [30, 60, 300]

CANDIDATES_34B = [
    "bid_recovery_34b",
    "ask_recovery_34b",
    "bid_shock_age_34b",
    "ask_shock_age_34b",
    "recovery_pressure_34b",
    "shock_presence_asym_34b"
]

# ------------------------------------------------------------
# 1. Strict research sample
# ------------------------------------------------------------

required = (
    ["symbol", "ts_sec", "is_observed"]
    + [f"bid_p{i}" for i in range(1, 6)]
    + [f"ask_p{i}" for i in range(1, 6)]
    + [f"bid_q{i}" for i in range(1, 6)]
    + [f"ask_q{i}" for i in range(1, 6)]
)

missing = [c for c in required if c not in df_1s.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

x = df_1s[required].copy()

x["symbol"] = x["symbol"].astype(str)
x["ts_sec"] = pd.to_numeric(
    x["ts_sec"], errors="raise"
).astype("int64")

ist = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ist.dt.date

clock = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

x = x.loc[
    (x["date"].astype(str) <= "2026-10-19")
    & (x["date"].astype(str) != "2026-09-28")
    & clock.between(0, 28799)
].copy()

coverage = x.groupby(
    ["symbol", "date"]
)["ts_sec"].transform("nunique")

x = (
    x.loc[coverage >= 0.97 * 28800]
    .sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True)
)

if x.empty:
    raise ValueError("No eligible research rows")

if x.duplicated(["symbol", "date", "ts_sec"]).any():
    raise ValueError("Duplicate research timestamps")

book_cols = (
    [f"bid_p{i}" for i in range(1, 6)]
    + [f"ask_p{i}" for i in range(1, 6)]
    + [f"bid_q{i}" for i in range(1, 6)]
    + [f"ask_q{i}" for i in range(1, 6)]
)

for c in book_cols:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    )

print("===== 34B RESEARCH SAMPLE =====")
print("Rows:", len(x))
print("Sessions:", x.groupby(
    ["symbol", "date"]
).ngroups)
print("Last date:", x["date"].max())

# ------------------------------------------------------------
# 2. Build genuine book-change indicators
# ------------------------------------------------------------

groups = x.groupby(
    ["symbol", "date"],
    sort=False
)

prev_book = groups[book_cols].shift(1)

book_changed = x[book_cols].ne(
    prev_book
).any(axis=1).to_numpy()

previous_ts = groups[
    "ts_sec"
].shift(1).to_numpy(dtype=float)

timestamps = x["ts_sec"].to_numpy(
    dtype=np.int64
)

consecutive = (
    np.isfinite(previous_ts)
    & ((timestamps - previous_ts) == 1)
)

observed = (
    x["is_observed"]
    .fillna(0)
    .astype(bool)
    .to_numpy()
)

genuine_book_change = (
    observed
    & consecutive
    & book_changed
)

# ------------------------------------------------------------
# 3. Displayed L5 depth
# ------------------------------------------------------------

bq = x[
    [f"bid_q{i}" for i in range(1, 6)]
].to_numpy(dtype=float)

aq = x[
    [f"ask_q{i}" for i in range(1, 6)]
].to_numpy(dtype=float)

valid_bid = (
    np.isfinite(bq).all(axis=1)
    & (bq >= 0).all(axis=1)
)

valid_ask = (
    np.isfinite(aq).all(axis=1)
    & (aq >= 0).all(axis=1)
)

bid_depth = np.where(
    valid_bid,
    bq.sum(axis=1),
    np.nan
)

ask_depth = np.where(
    valid_ask,
    aq.sum(axis=1),
    np.nan
)

bid_p1 = x["bid_p1"].to_numpy(dtype=float)
ask_p1 = x["ask_p1"].to_numpy(dtype=float)

valid_quote = (
    np.isfinite(bid_p1)
    & np.isfinite(ask_p1)
    & (bid_p1 > 0)
    & (ask_p1 >= bid_p1)
)

mid = np.where(
    valid_quote,
    (bid_p1 + ask_p1) / 2,
    np.nan
)

# ------------------------------------------------------------
# 4. Detect exact observed shocks
#
# We require:
#   - Actual displayed book change
#   - Genuine observation at the current second
#   - Previous second exists in the same session
#   - 20% or greater aggregate L5 depth drop
#
# A price-level change can still produce a shock.
# This measures displayed liquidity, not
# necessarily execution/cancellation activity.
# ------------------------------------------------------------

def detect_shocks(depth):

    depth_series = pd.Series(depth)

    frame = pd.DataFrame({
        "symbol": x["symbol"],
        "date": x["date"],
        "depth": depth_series
    })

    prior = frame.groupby(
        ["symbol", "date"],
        sort=False
    )["depth"].shift(1).to_numpy(dtype=float)

    loss = prior - depth

    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):
        loss_fraction = loss / prior

    shock = (
        genuine_book_change
        & np.isfinite(prior)
        & np.isfinite(depth)
        & (prior > 0)
        & (depth >= 0)
        & (loss_fraction >= SHOCK_THRESHOLD_34B)
    )

    return shock, loss

bid_shock, bid_loss = detect_shocks(
    bid_depth
)

ask_shock, ask_loss = detect_shocks(
    ask_depth
)

print("\n===== 34B DISTINCT SHOCK EVENTS =====")
print("Bid shocks:", int(bid_shock.sum()))
print("Ask shocks:", int(ask_shock.sum()))

# ------------------------------------------------------------
# 5. Propagate most recent event within each session
#
# The last shock timestamp, post-shock depth,
# and shock size are carried forward together.
#
# Crucial:
#   Never carry an event across session boundaries.
#
# Event lifetime is 60 calendar seconds.
# ------------------------------------------------------------

def propagate_events(
    shock_mask,
    loss,
    current_depth
):

    temporary = pd.DataFrame({
        "symbol": x["symbol"],
        "date": x["date"],
        "event_ts": np.where(
            shock_mask, timestamps, np.nan
        ),
        "shock_depth": np.where(
            shock_mask, current_depth, np.nan
        ),
        "shock_loss": np.where(
            shock_mask, loss, np.nan
        )
    })

    g = temporary.groupby(
        ["symbol", "date"],
        sort=False
    )

    last_ts = g["event_ts"].ffill().to_numpy(
        dtype=float
    )

    post_depth = g["shock_depth"].ffill().to_numpy(
        dtype=float
    )

    last_loss = g["shock_loss"].ffill().to_numpy(
        dtype=float
    )

    age = timestamps - last_ts

    active = (
        np.isfinite(last_ts)
        & np.isfinite(last_loss)
        & np.isfinite(post_depth)
        & np.isfinite(current_depth)
        & (last_loss > 0)
        & (age >= 0)
        & (age < LOOKBACK_34B)
    )

    recovery = np.full(
        len(x), np.nan, dtype=float
    )

    recovery[active] = np.clip(
        (
            current_depth[active]
            - post_depth[active]
        ) / last_loss[active],
        0,
        RECOVERY_CAP_34B
    )

    shock_age = np.full(
        len(x), np.nan, dtype=float
    )

    shock_age[active] = age[active]

    return recovery, shock_age, active

bid_recovery, bid_age, bid_active = (
    propagate_events(
        bid_shock, bid_loss, bid_depth
    )
)

ask_recovery, ask_age, ask_active = (
    propagate_events(
        ask_shock, ask_loss, ask_depth
    )
)

# ------------------------------------------------------------
# 6. Candidate features
# ------------------------------------------------------------

# Missing event contributions become zero only for
# this aggregate pressure feature.
# Individual recoveries retain NaN.

recovery_pressure = (
    np.where(
        bid_active, bid_recovery, 0.0
    )
    - np.where(
        ask_active, ask_recovery, 0.0
    )
)

shock_presence_asym = (
    bid_active.astype(float)
    - ask_active.astype(float)
)

calendar_exp34b_features = x[
    ["symbol", "ts_sec", "date"]
].copy()

feature_values = {
    "bid_recovery_34b": bid_recovery,
    "ask_recovery_34b": ask_recovery,
    "bid_shock_age_34b": bid_age,
    "ask_shock_age_34b": ask_age,
    "recovery_pressure_34b": recovery_pressure,
    "shock_presence_asym_34b": shock_presence_asym
}

# Additional diagnostic indicators for future ablation.
feature_values["bid_shock_present_34b"] = (
    bid_active.astype(float)
)
feature_values["ask_shock_present_34b"] = (
    ask_active.astype(float)
)

for col, arr in feature_values.items():

    arr = np.asarray(arr, dtype=float).copy()
    arr[~np.isfinite(arr)] = np.nan

    calendar_exp34b_features[col] = (
        arr.astype(np.float32)
    )

# ------------------------------------------------------------
# 7. Exact future returns
# ------------------------------------------------------------

session_index = pd.MultiIndex.from_frame(
    x[["symbol", "date", "ts_sec"]]
)

for h in HORIZONS_34B:

    future_index = pd.MultiIndex.from_arrays(
        [
            x["symbol"],
            x["date"],
            timestamps + h
        ],
        names=["symbol", "date", "ts_sec"]
    )

    positions = session_index.get_indexer(
        future_index
    )

    future_mid = np.full(
        len(x), np.nan
    )

    found = positions >= 0

    future_mid[found] = mid[
        positions[found]
    ]

    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):
        ret = 1e4 * np.log(
            future_mid / mid
        )

    ret[~np.isfinite(ret)] = np.nan

    calendar_exp34b_features[
        f"future_return_{h}_34b"
    ] = ret.astype(np.float32)

# ------------------------------------------------------------
# 8. Event diagnostics
# ------------------------------------------------------------

calendar_exp34b_event_summary = pd.DataFrame({
    "side": ["bid", "ask"],
    "distinct_shocks": [
        int(bid_shock.sum()),
        int(ask_shock.sum())
    ],
    "active_rows": [
        int(bid_active.sum()),
        int(ask_active.sum())
    ],
    "active_coverage_pct": [
        100 * bid_active.mean(),
        100 * ask_active.mean()
    ],
    "median_age_seconds": [
        float(np.nanmedian(bid_age))
            if bid_active.any() else np.nan,
        float(np.nanmedian(ask_age))
            if ask_active.any() else np.nan
    ],
    "median_recovery": [
        float(np.nanmedian(bid_recovery))
            if bid_active.any() else np.nan,
        float(np.nanmedian(ask_recovery))
            if ask_active.any() else np.nan
    ]
})

# ------------------------------------------------------------
# 9. Daily Spearman IC
#
# Conditional IC for sparse features.
# Full-population IC for dense features.
#
# Require at least 100 valid samples/day.
# ------------------------------------------------------------

def spearman_34b(a, b):

    valid = np.isfinite(a) & np.isfinite(b)

    count = int(valid.sum())

    if count < 100:
        return np.nan, count

    aa = pd.Series(a[valid])
    bb = pd.Series(b[valid])

    if (
        aa.nunique() < 2
        or bb.nunique() < 2
    ):
        return np.nan, count

    return float(
        aa.corr(bb, method="spearman")
    ), count

daily_records = []

for (symbol, date), day in (
    calendar_exp34b_features.groupby(
        ["symbol", "date"],
        sort=False
    )
):

    for h in HORIZONS_34B:

        y = day[
            f"future_return_{h}_34b"
        ].to_numpy(dtype=float)

        for col in CANDIDATES_34B:

            a = day[col].to_numpy(
                dtype=float
            )

            ic, count = spearman_34b(a, y)

            daily_records.append({
                "symbol": symbol,
                "date": date,
                "horizon": h,
                "feature": col,
                "ic": ic,
                "valid_rows": count
            })

calendar_exp34b_daily = pd.DataFrame(
    daily_records
)

calendar_exp34b_summary = (
    calendar_exp34b_daily
    .groupby(["horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 10. Optional redundancy checks
#
# One source at a time, exact key alignment.
# No large merged DataFrames.
# ------------------------------------------------------------

redundancy_records = []

def add_redundancy_34b(source, source_cols):

    target_keys = pd.MultiIndex.from_frame(
        calendar_exp34b_features[
            ["symbol", "ts_sec"]
        ]
    )

    source_keys = pd.MultiIndex.from_arrays(
        [
            source["symbol"].astype(str),
            pd.to_numeric(
                source["ts_sec"]
            ).astype("int64")
        ],
        names=["symbol", "ts_sec"]
    )

    if source_keys.has_duplicates:
        raise ValueError(
            "Duplicate keys in redundancy source"
        )

    positions = source_keys.get_indexer(
        target_keys
    )

    found = positions >= 0

    for ref_col in source_cols:

        if ref_col not in source.columns:
            continue

        values = pd.to_numeric(
            source[ref_col],
            errors="coerce"
        ).to_numpy(dtype=float)

        aligned = np.full(
            len(calendar_exp34b_features),
            np.nan
        )

        aligned[found] = values[
            positions[found]
        ]

        for candidate in CANDIDATES_34B:

            a = calendar_exp34b_features[
                candidate
            ].to_numpy(dtype=float)

            valid = (
                np.isfinite(a)
                & np.isfinite(aligned)
            )

            corr = (
                pd.Series(a[valid]).corr(
                    pd.Series(aligned[valid]),
                    method="spearman"
                )
                if valid.sum() >= 200
                else np.nan
            )

            redundancy_records.append({
                "candidate": candidate,
                "reference": ref_col,
                "spearman": corr
            })

if "core_31" in globals():

    add_redundancy_34b(
        core_31,
        [
            "queue_replenishment_pressure_60s",
            "recovery_available_300",
            "queue_imbalance_l1",
            "depth_imbalance_l5",
            "ofi_l1_sum_60s_signed_log"
        ]
    )

if "calendar_exp34a_features" in globals():

    add_redundancy_34b(
        calendar_exp34a_features,
        [
            "l5_bid_recovery_34a",
            "l5_ask_recovery_34a"
        ]
    )

calendar_exp34b_redundancy = pd.DataFrame(
    redundancy_records
)

# ------------------------------------------------------------
# 11. Display final results
# ------------------------------------------------------------

print("\n===== 34B EVENT SUMMARY =====")

print(
    calendar_exp34b_event_summary
    .round(5)
    .to_string(index=False)
)

print("\n===== 34B FEATURE COVERAGE =====")

for col in CANDIDATES_34B:

    s = calendar_exp34b_features[col]

    print(
        col,
        "valid_pct=",
        round(100 * s.notna().mean(), 4),
        "std=",
        round(s.std(), 6)
    )

print("\n===== 34B DAILY SIGNED IC =====")

print(
    calendar_exp34b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 34B REDUNDANCY =====")

if not calendar_exp34b_redundancy.empty:

    print(
        calendar_exp34b_redundancy
        .pivot(
            index="candidate",
            columns="reference",
            values="spearman"
        )
        .round(5)
        .to_string()
    )

else:
    print("No optional reference frames available")

print("\n===== EXPERIMENT 34B COMPLETE =====")
print("Original raw data unchanged.")
print("October 21-23 excluded.")
print("Core feature count remains 37.")

gc.collect()


===== 34B RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Last date: 2026-10-19

===== 34B DISTINCT SHOCK EVENTS =====
Bid shocks: 11364
Ask shocks: 26552

===== 34B EVENT SUMMARY =====
side  distinct_shocks  active_rows  active_coverage_pct  median_age_seconds  median_recovery
 bid            11364       262293             57.12926                17.0          0.12667
 ask            26552       304438             66.30874                10.0          0.26796

===== 34B FEATURE COVERAGE =====
bid_recovery_34b valid_pct= 57.1293 std= 0.693722
ask_recovery_34b valid_pct= 66.3087 std= 0.674849
bid_shock_age_34b valid_pct= 57.1293 std= 16.91501
ask_shock_age_34b valid_pct= 66.3087 std= 16.308256
recovery_pressure_34b valid_pct= 100.0 std= 0.828076
shock_presence_asym_34b valid_pct= 100.0 std= 0.62561

===== 34B DAILY SIGNED IC =====
 horizon                 feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows
      30        ask_recovery_34b    16 -0.052643 

29

In [24]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import gc

# ============================================================
# EXPERIMENT 34C — FULL 37-FEATURE ABLATION
#
# Compares:
# A — Full reconstructed baseline
# B — Old ask recovery (34A)
# C — Event-anchored ask recovery (34B)
# D — Event-anchored bid + ask recovery
# E — Recovery pressure
# F — Ask shock age
# G — Recovery pressure + ask shock age
#
# Required:
#   usdinr_rebuilt_37_research
#   calendar_exp34a_features
#   calendar_exp34b_features
#
# No holdout access.
# No feature promotion.
# ============================================================

HORIZONS = [30, 60, 300]
KEYS = ["symbol", "ts_sec"]

CORE31 = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

ADDED6 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

BASE_COLS = CORE31 + ADDED6 + CONTROLS9
assert len(BASE_COLS) == 46

EXTRA_COLS = [
    "l5_ask_recovery_34a",
    "old_ask_present_34c",
    "ask_recovery_34b",
    "ask_shock_present_34b",
    "bid_recovery_34b",
    "bid_shock_present_34b",
    "recovery_pressure_34b",
    "ask_shock_age_34b"
]

PARAMS = dict(
    objective="regression",
    n_estimators=180,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=5,
    min_child_samples=200,
    reg_lambda=2,
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

# ------------------------------------------------------------
# 1. Validate input frames
# ------------------------------------------------------------

needed_frames = [
    "usdinr_rebuilt_37_research",
    "calendar_exp34a_features",
    "calendar_exp34b_features"
]

for name in needed_frames:
    if name not in globals():
        raise ValueError(f"Missing DataFrame: {name}")

baseline = usdinr_rebuilt_37_research
old = calendar_exp34a_features
new = calendar_exp34b_features

for name, frame, cols in [
    ("baseline", baseline, BASE_COLS),
    ("34A", old, ["l5_ask_recovery_34a"]),
    ("34B", new, [
        "ask_recovery_34b",
        "ask_shock_present_34b",
        "bid_recovery_34b",
        "bid_shock_present_34b",
        "recovery_pressure_34b",
        "ask_shock_age_34b"
    ] + [
        f"future_return_{h}_34b"
        for h in HORIZONS
    ])
]:
    missing = [
        c for c in KEYS + cols
        if c not in frame.columns
    ]

    if missing:
        raise ValueError(
            f"{name} missing columns: {missing}"
        )

# ------------------------------------------------------------
# 2. Research sample from Experiment 34B
# ------------------------------------------------------------

date_series = pd.to_datetime(
    new["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

research_mask = (
    date_series.astype(str) <= "2026-10-19"
)

target_cols = [
    f"future_return_{h}_34b"
    for h in HORIZONS
]

research = new.loc[
    research_mask,
    KEYS + target_cols
].copy()

research["symbol"] = research["symbol"].astype(str)
research["ts_sec"] = pd.to_numeric(
    research["ts_sec"], errors="raise"
).astype("int64")

research["date"] = pd.to_datetime(
    research["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

research = (
    research.sort_values(
        ["date", "symbol", "ts_sec"]
    )
    .reset_index(drop=True)
)

if research.duplicated(KEYS).any():
    raise ValueError("Duplicate research keys")

N = len(research)

print("===== 34C SAMPLE =====")
print("Rows:", N)
print("Days:", research["date"].nunique())
print("First:", research["date"].min())
print("Last:", research["date"].max())

# ------------------------------------------------------------
# 3. Exact-key alignment without DataFrame merges
# ------------------------------------------------------------

target_index = pd.MultiIndex.from_frame(
    research[KEYS]
)

def source_positions(source, label):

    idx = pd.MultiIndex.from_arrays(
        [
            source["symbol"].astype(str),
            pd.to_numeric(
                source["ts_sec"], errors="raise"
            ).astype("int64")
        ],
        names=KEYS
    )

    if idx.has_duplicates:
        raise ValueError(
            f"Duplicate timestamp keys: {label}"
        )

    pos = idx.get_indexer(target_index)

    match = pos >= 0

    print(
        label,
        "key match:",
        f"{100 * match.mean():.4f}%"
    )

    if not match.all():
        raise ValueError(
            f"Missing research keys in {label}"
        )

    return pos

base_pos = source_positions(
    baseline, "Rebuilt baseline"
)

old_pos = source_positions(
    old, "Experiment 34A"
)

new_pos = source_positions(
    new, "Experiment 34B"
)

def aligned_numeric(source, col, pos):
    return pd.to_numeric(
        source[col], errors="coerce"
    ).to_numpy(dtype=np.float32)[pos]

# ------------------------------------------------------------
# 4. Build baseline and candidate arrays
# ------------------------------------------------------------

X_base = np.column_stack([
    aligned_numeric(baseline, c, base_pos)
    for c in BASE_COLS
]).astype(np.float32)

X_base[~np.isfinite(X_base)] = np.nan

old_ask = aligned_numeric(
    old, "l5_ask_recovery_34a", old_pos
)

new_ask = aligned_numeric(
    new, "ask_recovery_34b", new_pos
)

new_bid = aligned_numeric(
    new, "bid_recovery_34b", new_pos
)

new_pressure = aligned_numeric(
    new, "recovery_pressure_34b", new_pos
)

new_ask_age = aligned_numeric(
    new, "ask_shock_age_34b", new_pos
)

# Preserve missing event-specific values.
# Presence columns distinguish missing from zero.

old_present = np.isfinite(
    old_ask
).astype(np.float32)

new_ask_present = aligned_numeric(
    new, "ask_shock_present_34b", new_pos
)

new_bid_present = aligned_numeric(
    new, "bid_shock_present_34b", new_pos
)

extra_arrays = {
    "l5_ask_recovery_34a": old_ask,
    "old_ask_present_34c": old_present,
    "ask_recovery_34b": new_ask,
    "ask_shock_present_34b": new_ask_present,
    "bid_recovery_34b": new_bid,
    "bid_shock_present_34b": new_bid_present,
    "recovery_pressure_34b": new_pressure,
    "ask_shock_age_34b": new_ask_age
}

X_extra = np.column_stack([
    extra_arrays[c]
    for c in EXTRA_COLS
]).astype(np.float32)

X_extra[~np.isfinite(X_extra)] = np.nan

X = np.column_stack([
    X_base, X_extra
]).astype(np.float32)

ALL_COLS = BASE_COLS + EXTRA_COLS

col_idx = {
    name: i for i, name in enumerate(ALL_COLS)
}

base_indices = list(range(len(BASE_COLS)))

def select(*names):
    return (
        base_indices
        + [col_idx[name] for name in names]
    )

MODELS = {
    "A_full37": base_indices,

    "B_old_ask": select(
        "l5_ask_recovery_34a",
        "old_ask_present_34c"
    ),

    "C_new_ask": select(
        "ask_recovery_34b",
        "ask_shock_present_34b"
    ),

    "D_new_bid_ask": select(
        "bid_recovery_34b",
        "bid_shock_present_34b",
        "ask_recovery_34b",
        "ask_shock_present_34b"
    ),

    "E_recovery_pressure": select(
        "recovery_pressure_34b"
    ),

    "F_ask_age": select(
        "ask_shock_age_34b",
        "ask_shock_present_34b"
    ),

    "G_pressure_ask_age": select(
        "recovery_pressure_34b",
        "ask_shock_age_34b",
        "ask_shock_present_34b"
    )
}

# The baseline-complete sample is common to all models.
# Do NOT require candidate features to be nonmissing.

complete = np.isfinite(X_base).all(axis=1)

print("\n===== 34C MATCHED SAMPLE =====")

print("Baseline complete rows:", int(complete.sum()))
print(
    "Baseline coverage:",
    f"{100 * complete.mean():.4f}%"
)

print(
    "Old ask valid:",
    f"{100 * np.isfinite(old_ask).mean():.4f}%"
)

print(
    "New ask valid:",
    f"{100 * np.isfinite(new_ask).mean():.4f}%"
)

print(
    "New bid valid:",
    f"{100 * np.isfinite(new_bid).mean():.4f}%"
)

# ------------------------------------------------------------
# 5. Chronological walk-forward
# ------------------------------------------------------------

dates = research["date"].to_numpy()
ts = research["ts_sec"].to_numpy(dtype="int64")

unique_dates = sorted(pd.unique(dates))

MIN_TRAIN_DAYS = 5
TRAIN_STRIDE = 10

daily_records = []
importance_records = []

for day_i in range(
    MIN_TRAIN_DAYS,
    len(unique_dates)
):

    test_date = unique_dates[day_i]

    train_mask = (
        np.isin(dates, unique_dates[:day_i])
        & complete
        & (ts % TRAIN_STRIDE == 0)
    )

    test_mask = (
        (dates == test_date)
        & complete
    )

    for h in HORIZONS:

        y = pd.to_numeric(
            research[f"future_return_{h}_34b"],
            errors="coerce"
        ).to_numpy(dtype=np.float32)

        train_rows = np.flatnonzero(
            train_mask & np.isfinite(y)
        )

        test_rows = np.flatnonzero(
            test_mask & np.isfinite(y)
        )

        if (
            len(train_rows) < 1000
            or len(test_rows) < 200
        ):
            continue

        y_train = y[train_rows]
        y_test = y[test_rows].astype(float)

        for model_name, selected in MODELS.items():

            model = lgb.LGBMRegressor(**PARAMS)

            X_train = X[np.ix_(
                train_rows, selected
            )]

            X_test = X[np.ix_(
                test_rows, selected
            )]

            model.fit(X_train, y_train)

            pred = model.predict(X_test)
            err = y_test - pred

            daily_records.append({
                "date": test_date,
                "horizon": h,
                "model": model_name,
                "rows": len(test_rows),
                "sse": float(np.sum(err**2)),
                "sae": float(np.sum(np.abs(err))),
                "mse": float(np.mean(err**2)),
                "mae": float(np.mean(np.abs(err)))
            })

            if model_name != "A_full37":

                gains = (
                    model.booster_
                    .feature_importance(
                        importance_type="gain"
                    )
                )

                total = float(gains.sum())

                for j, full_i in enumerate(selected):

                    feature = ALL_COLS[full_i]

                    if feature not in EXTRA_COLS:
                        continue

                    importance_records.append({
                        "date": test_date,
                        "horizon": h,
                        "model": model_name,
                        "feature": feature,
                        "gain_share_pct": (
                            100 * float(gains[j]) / total
                            if total > 0 else 0.0
                        )
                    })

            del (
                model, X_train,
                X_test, pred, err
            )

    print("Completed:", test_date)

calendar_exp34c_daily = pd.DataFrame(
    daily_records
)

calendar_exp34c_importance = pd.DataFrame(
    importance_records
)

if calendar_exp34c_daily.empty:
    raise ValueError("No walk-forward results")

# ------------------------------------------------------------
# 6. Pooled and paired model comparisons
# ------------------------------------------------------------

COMPARE = {
    "old_ask_vs_base": (
        "B_old_ask", "A_full37"
    ),
    "new_ask_vs_base": (
        "C_new_ask", "A_full37"
    ),
    "new_bidask_vs_base": (
        "D_new_bid_ask", "A_full37"
    ),
    "pressure_vs_base": (
        "E_recovery_pressure", "A_full37"
    ),
    "ask_age_vs_base": (
        "F_ask_age", "A_full37"
    ),
    "pressure_age_vs_base": (
        "G_pressure_ask_age", "A_full37"
    ),
    "new_ask_vs_old_ask": (
        "C_new_ask", "B_old_ask"
    ),
    "pressure_vs_old_ask": (
        "E_recovery_pressure", "B_old_ask"
    ),
    "pressure_age_vs_old_ask": (
        "G_pressure_ask_age", "B_old_ask"
    )
}

summary_records = []
daily_gain_records = []

for h, part in calendar_exp34c_daily.groupby(
    "horizon"
):

    pooled = part.groupby("model")[
        ["sse", "sae"]
    ].sum()

    mse = part.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    mae = part.pivot(
        index="date",
        columns="model",
        values="mae"
    )

    for name, (candidate, reference) in (
        COMPARE.items()
    ):

        daily_mse_gain = (
            100 * (1 - mse[candidate] / mse[reference])
        )

        daily_mae_gain = (
            100 * (1 - mae[candidate] / mae[reference])
        )

        summary_records.append({
            "horizon": h,
            "comparison": name,
            "days": len(daily_mse_gain),
            "pooled_mse_gain_pct": 100 * (
                1
                - pooled.loc[candidate, "sse"]
                / pooled.loc[reference, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1
                - pooled.loc[candidate, "sae"]
                / pooled.loc[reference, "sae"]
            ),
            "mse_win_days": int(
                (daily_mse_gain > 0).sum()
            ),
            "mae_win_days": int(
                (daily_mae_gain > 0).sum()
            ),
            "median_daily_mse_gain_pct": float(
                daily_mse_gain.median()
            ),
            "worst_daily_mse_gain_pct": float(
                daily_mse_gain.min()
            )
        })

        for date in daily_mse_gain.index:

            daily_gain_records.append({
                "date": date,
                "horizon": h,
                "comparison": name,
                "mse_gain_pct": float(
                    daily_mse_gain.loc[date]
                ),
                "mae_gain_pct": float(
                    daily_mae_gain.loc[date]
                )
            })

calendar_exp34c_summary = pd.DataFrame(
    summary_records
)

calendar_exp34c_daily_gains = pd.DataFrame(
    daily_gain_records
)

# ------------------------------------------------------------
# 7. Print result tables
# ------------------------------------------------------------

print("\n===== 34C FULL37 ABLATION =====")

print(
    calendar_exp34c_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 34C DAILY MSE GAINS =====")

for comparison in COMPARE:

    subset = calendar_exp34c_daily_gains.loc[
        calendar_exp34c_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print("\n", comparison)

    print(
        subset.pivot(
            index="date",
            columns="horizon",
            values="mse_gain_pct"
        )
        .round(6)
        .to_string()
    )

print("\n===== 34C FEATURE IMPORTANCE =====")

print(
    calendar_exp34c_importance
    .groupby([
        "horizon", "model", "feature"
    ])["gain_share_pct"]
    .agg(["mean", "median"])
    .round(6)
    .to_string()
)

print("\n===== EXPERIMENT 34C COMPLETE =====")
print("Baseline engineered features: 37")
print("Research holdout October 21-23 untouched")
print("New features promoted: 0")

gc.collect()


===== 34C SAMPLE =====
Rows: 459122
Days: 16
First: 2026-09-23
Last: 2026-10-19
Rebuilt baseline key match: 100.0000%
Experiment 34A key match: 100.0000%
Experiment 34B key match: 100.0000%

===== 34C MATCHED SAMPLE =====
Baseline complete rows: 402979
Baseline coverage: 87.7717%
Old ask valid: 13.4472%
New ask valid: 66.3087%
New bid valid: 57.1293%
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 34C FULL37 ABLATION =====
 horizon              comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  mse_win_days  mae_win_days  median_daily_mse_gain_pct  worst_daily_mse_gain_pct
      30         old_ask_vs_base    11            -0.031095             0.015397             5             6                   0.000000                 -0.190240
      30         new_ask_vs_base    11          

8

In [25]:

import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# EXPERIMENT 35A — RECONSTRUCTION AUDIT
#
# Required:
#   core_31
#   usdinr_rebuilt_37_research
#   df_1s
#
# Optional ORIGINAL experiment Parquets:
#   Set paths below only if original files exist.
#
# No holdout inspection.
# No model training.
# ============================================================

ORIGINAL_FILES = {
    "24A": None,
    "28A": None,
    "29A": None,
    "31A": None,
}

ADDED6 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a",
]

CONTROLS9 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds",
]

KEYS = ["symbol", "ts_sec"]

rebuilt = usdinr_rebuilt_37_research

research_dates = (
    pd.to_datetime(
        rebuilt["ts_sec"], unit="s", utc=True
    )
    .dt.tz_convert("Asia/Kolkata")
    .dt.date
)

assert max(research_dates).isoformat() <= "2026-10-19"
assert len(rebuilt) == 459122
assert not rebuilt.duplicated(KEYS).any()

core_features = [
    c for c in core_31.columns
    if c not in KEYS
]

assert len(core_features) == 31

# ------------------------------------------------------------
# 1. Verify original 31 feature values were preserved
# ------------------------------------------------------------

source_idx = pd.MultiIndex.from_frame(
    core_31[KEYS]
)

target_idx = pd.MultiIndex.from_frame(
    rebuilt[KEYS]
)

positions = source_idx.get_indexer(target_idx)

assert (positions >= 0).all()

core_mismatches = []

for col in core_features:

    a = pd.to_numeric(
        core_31[col], errors="coerce"
    ).to_numpy(dtype=np.float64)[positions]

    b = pd.to_numeric(
        rebuilt[col], errors="coerce"
    ).to_numpy(dtype=np.float64)

    equal = np.isclose(
        a, b,
        rtol=1e-6,
        atol=1e-6,
        equal_nan=True
    )

    if not equal.all():
        core_mismatches.append(
            (col, int((~equal).sum()))
        )

print("===== ORIGINAL CORE PRESERVATION =====")

print("Core features:", len(core_features))
print("Mismatching columns:", core_mismatches)
print(
    "Status:",
    "PASS" if not core_mismatches else "FAIL"
)

# ------------------------------------------------------------
# 2. Reconstructed feature diagnostics
# ------------------------------------------------------------

audit_rows = []

for col in ADDED6 + CONTROLS9:

    s = pd.to_numeric(
        rebuilt[col], errors="coerce"
    )

    finite = s[np.isfinite(s)]

    audit_rows.append({
        "feature": col,
        "coverage_pct": 100 * np.isfinite(s).mean(),
        "zero_pct": 100 * (s == 0).mean(),
        "nunique": s.nunique(dropna=True),
        "std": s.std(),
        "min": s.min(),
        "median": s.median(),
        "max": s.max()
    })

exp35a_diagnostics = pd.DataFrame(audit_rows)

print("\n===== RECONSTRUCTED FEATURE DIAGNOSTICS =====")

print(
    exp35a_diagnostics
    .round(6)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 3. Mathematical invariant checks
# ------------------------------------------------------------

checks = []

def add_check(name, passed, detail=""):
    checks.append({
        "check": name,
        "passed": bool(passed),
        "detail": str(detail)
    })

for col in [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "relative_migration_60_31a"
]:
    values = pd.to_numeric(
        rebuilt[col], errors="coerce"
    ).to_numpy(dtype=float)

    valid = values[np.isfinite(values)]

    add_check(
        f"{col}: bounded [-1,1]",
        bool(
            (valid >= -1.000001).all()
            and (valid <= 1.000001).all()
        ),
        f"valid_rows={len(valid)}"
    )

for col in [
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a"
]:
    values = pd.to_numeric(
        rebuilt[col], errors="coerce"
    ).to_numpy(dtype=float)

    valid = values[np.isfinite(values)]

    add_check(
        f"{col}: bounded [0,1]",
        bool(
            (valid >= -1e-8).all()
            and (valid <= 1.000001).all()
        )
    )

spread = pd.to_numeric(
    rebuilt["spread_bps_28a"],
    errors="coerce"
).to_numpy(dtype=float)

add_check(
    "Spread nonnegative",
    bool(
        (spread[np.isfinite(spread)] >= 0).all()
    )
)

clock = pd.to_numeric(
    rebuilt["clock_seconds"],
    errors="coerce"
).to_numpy(dtype=float)

add_check(
    "Clock in session range",
    bool(
        np.isfinite(clock).all()
        and (clock >= 0).all()
        and (clock < 28800).all()
    )
)

extra_matrix = rebuilt[
    ADDED6 + CONTROLS9
].to_numpy(dtype=float)

add_check(
    "No infinite reconstructed values",
    not np.isinf(extra_matrix).any()
)

exp35a_checks = pd.DataFrame(checks)

print("\n===== MATHEMATICAL CHECKS =====")

print(exp35a_checks.to_string(index=False))

# ------------------------------------------------------------
# 4. Compare ORIGINAL reference frames if supplied
#
# Do not confuse *_rebuilt.parquet with originals.
# The parity check is skipped if an original file
# was not supplied.
# ------------------------------------------------------------

expected_columns = {
    "24A": ADDED6[0:1],
    "28A": ADDED6[1:3] + CONTROLS9,
    "29A": ADDED6[3:5],
    "31A": ADDED6[5:6],
}

parity_records = []

for experiment, path in ORIGINAL_FILES.items():

    if path is None:
        print(
            f"\n{experiment}: ORIGINAL FILE NOT AVAILABLE"
        )
        continue

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(path)

    original = pd.read_parquet(path)

    original_idx = pd.MultiIndex.from_frame(
        original[KEYS]
    )

    if original_idx.has_duplicates:
        raise ValueError(
            f"Duplicate original keys: {experiment}"
        )

    pos = original_idx.get_indexer(target_idx)
    matched = pos >= 0

    if matched.sum() == 0:
        raise ValueError(
            f"No overlapping keys: {experiment}"
        )

    for col in expected_columns[experiment]:

        if col not in original.columns:
            raise ValueError(
                f"Original {experiment} missing {col}"
            )

        a = pd.to_numeric(
            original[col], errors="coerce"
        ).to_numpy(dtype=float)[pos[matched]]

        b = pd.to_numeric(
            rebuilt[col], errors="coerce"
        ).to_numpy(dtype=float)[matched]

        finite_both = (
            np.isfinite(a) & np.isfinite(b)
        )

        both_nan = np.isnan(a) & np.isnan(b)

        equal = np.isclose(
            a, b,
            rtol=1e-6,
            atol=1e-6,
            equal_nan=True
        )

        differences = (
            np.abs(a[finite_both] - b[finite_both])
        )

        parity_records.append({
            "experiment": experiment,
            "feature": col,
            "matched_rows": int(matched.sum()),
            "equal_pct": 100 * equal.mean(),
            "missing_pattern_match_pct":
                100 * (
                    finite_both | both_nan
                ).mean(),
            "max_abs_diff": (
                float(differences.max())
                if len(differences) else np.nan
            ),
            "parity_pass": bool(equal.all())
        })

exp35a_parity = pd.DataFrame(parity_records)

print("\n===== ORIGINAL FORMULA PARITY =====")

if exp35a_parity.empty:
    print(
        "UNVERIFIED: No original experiment "
        "reference files were supplied."
    )
else:
    print(
        exp35a_parity
        .round(8)
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 5. Overall summary
# ------------------------------------------------------------

full_cols = core_features + ADDED6 + CONTROLS9

complete = np.isfinite(
    rebuilt[full_cols].to_numpy(dtype=float)
).all(axis=1)

print("\n===== EXPERIMENT 35A SUMMARY =====")

print("Research rows:", len(rebuilt))
print("Baseline inputs:", len(full_cols))
print("Complete rows:", int(complete.sum()))
print(
    "Complete coverage:",
    f"{100 * complete.mean():.4f}%"
)

print(
    "Core 31 preserved:",
    not bool(core_mismatches)
)

print(
    "Mathematical checks passed:",
    bool(exp35a_checks["passed"].all())
)

print(
    "Original reconstructed-feature parity:",
    "UNVERIFIED"
    if exp35a_parity.empty
    else (
        "PASS"
        if (
            len(exp35a_parity) == 15
            and exp35a_parity["parity_pass"].all()
        )
        else "INCOMPLETE OR FAILED"
    )
)

print("October 21-23 holdout: NOT ACCESSED")
print("Feature count: 37 + 2 challengers")


===== ORIGINAL CORE PRESERVATION =====
Core features: 31
Mismatching columns: []
Status: PASS

===== RECONSTRUCTED FEATURE DIAGNOSTICS =====
                       feature  coverage_pct  zero_pct  nunique         std        min       median          max
quote_retreat_imbalance_30_24a     99.898938 56.027592      146    0.344194  -0.923077     0.000000     0.947368
           slope_asymmetry_28a    100.000000  0.000000   123315    0.755552  -2.917996     0.064550     3.025912
       curvature_asymmetry_28a    100.000000  0.000000   154986    0.610247  -2.336486     0.031190     2.755216
    weighted_gap_asymmetry_29a    100.000000  0.340650    49544    0.408416 -23.251713     0.014943    19.165913
       fragility_asymmetry_29a    100.000000  0.000000   154193    0.094136  -3.050687    -0.001983    13.094525
     relative_migration_60_31a     99.790905  3.346169   118023    0.834048  -1.000000     0.000000     1.000000
           queue_imbalance_28a    100.000000  0.909344    30185    0

In [26]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# EXPERIMENT 36A — BOOK UPDATE CLUSTERING
#
# Required:
#   df_1s
#
# Optional:
#   core_31
#
# Outputs:
#   calendar_exp36a_features
#   calendar_exp36a_daily
#   calendar_exp36a_summary
#   calendar_exp36a_redundancy
#
# Research only: original 16 eligible sessions.
# October 21-23 excluded.
# ============================================================

HORIZONS_36A = [30, 60, 300]

CANDIDATES_36A = [
    "update_acceleration_36a",
    "event_burstiness_60_36a",
    "interarrival_cv_60_36a",
    "side_update_imbalance_60_36a"
]

REFS_36A = [
    "update_rate_300s",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "price_move_rate_30s",
    "ofi_l1_sum_60s_signed_log",
    "queue_imbalance_l1",
    "depth_imbalance_l5"
]

KEYS_36A = ["symbol", "ts_sec"]
SESSION_KEYS_36A = ["symbol", "date", "ts_sec"]

BOOK_36A = [
    f"{side}_{field}{level}"
    for side in ["bid", "ask"]
    for field in ["p", "q"]
    for level in range(1, 6)
]

required = KEYS_36A + ["is_observed"] + BOOK_36A

missing = sorted(set(required) - set(df_1s.columns))
if missing:
    raise ValueError(f"Missing columns: {missing}")

# ------------------------------------------------------------
# 1. Research sample
# ------------------------------------------------------------

x = df_1s[required].copy()

x["symbol"] = x["symbol"].astype(str)
x["ts_sec"] = pd.to_numeric(
    x["ts_sec"], errors="raise"
).astype(np.int64)

ist = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ist.dt.date

clock = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 9 * 3600
)

x = x.loc[
    (x["date"].astype(str) <= "2026-10-19")
    & (x["date"].astype(str) != "2026-09-28")
    & clock.between(0, 28799)
].copy()

coverage = x.groupby(
    ["symbol", "date"]
)["ts_sec"].transform("nunique")

x = (
    x.loc[coverage >= 0.97 * 28800]
    .sort_values(SESSION_KEYS_36A)
    .reset_index(drop=True)
)

if x.empty:
    raise ValueError("No eligible research sessions")

if x.duplicated(SESSION_KEYS_36A).any():
    raise ValueError("Duplicate timestamps")

for col in BOOK_36A:
    x[col] = pd.to_numeric(
        x[col], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

N = len(x)

print("===== 36A RESEARCH SAMPLE =====")
print("Rows:", N)
print(
    "Sessions:",
    x.groupby(["symbol", "date"]).ngroups
)
print("Last date:", x["date"].max())

# ------------------------------------------------------------
# 2. Genuine observed book changes
#
# Compare against previous available snapshot in
# same session; require observed current snapshot.
#
# Missing book values do not qualify as valid events.
# ------------------------------------------------------------

g = x.groupby(
    ["symbol", "date"], sort=False
)

prev = g[BOOK_36A].shift(1)

bid_cols = [
    c for c in BOOK_36A
    if c.startswith("bid_")
]

ask_cols = [
    c for c in BOOK_36A
    if c.startswith("ask_")
]

valid_current = x[BOOK_36A].notna().all(axis=1)
valid_prev = prev[BOOK_36A].notna().all(axis=1)

valid_pair = (
    valid_current & valid_prev
).to_numpy()

observed = (
    x["is_observed"].eq(1).to_numpy()
)

bid_changed = (
    x[bid_cols].ne(prev[bid_cols])
    .any(axis=1)
    .to_numpy()
)

ask_changed = (
    x[ask_cols].ne(prev[ask_cols])
    .any(axis=1)
    .to_numpy()
)

bid_event = (
    observed & valid_pair & bid_changed
).astype(np.int8)

ask_event = (
    observed & valid_pair & ask_changed
).astype(np.int8)

any_event = (
    (bid_event == 1) | (ask_event == 1)
).astype(np.int8)

print("\n===== 36A EVENT DIAGNOSTICS =====")
print("Observed seconds:", int(observed.sum()))
print("Book-change seconds:", int(any_event.sum()))
print("Bid-change seconds:", int(bid_event.sum()))
print("Ask-change seconds:", int(ask_event.sum()))
print(
    "Both-sides change seconds:",
    int(((bid_event == 1) & (ask_event == 1)).sum())
)

del prev, g
gc.collect()

# ------------------------------------------------------------
# 3. Exact clock-window counting
#
# For a session's timestamps t, use a cumulative
# sum with searchsorted to count events within:
#
#   [t - window + 1, t]
#
# Missing seconds contribute zero events.
#
# This does NOT use row-count rolling windows.
# ------------------------------------------------------------

def trailing_counts_36a(ts, event, window):
    """
    ts: sorted int64 timestamps for one session
    event: 0/1 array, same length
    """
    prefix = np.empty(
        len(ts) + 1, dtype=np.int64
    )
    prefix[0] = 0
    np.cumsum(event, out=prefix[1:])

    left = np.searchsorted(
        ts, ts - window + 1, side="left"
    )

    return (
        prefix[np.arange(len(ts)) + 1]
        - prefix[left]
    )

# Build results by contiguous session slices.
# All sessions are sorted by symbol/date/time.

session_groups = x.groupby(
    ["symbol", "date"],
    sort=False
).indices

ts_all = x["ts_sec"].to_numpy(
    dtype=np.int64
)

count10 = np.full(N, np.nan)
count60 = np.full(N, np.nan)
bid_count60 = np.full(N, np.nan)
ask_count60 = np.full(N, np.nan)

burstiness = np.full(N, np.nan)
interarrival_cv = np.full(N, np.nan)

# Historical-window completeness:
# Strictly require full 60 seconds of market history
# after the session's first available timestamp.
# Internal unrepresented seconds are treated as
# zero observed updates, not fabricated book events.

for (symbol, day), idx in session_groups.items():

    idx = np.sort(idx)

    ts = ts_all[idx]

    ev = any_event[idx]
    bev = bid_event[idx]
    aev = ask_event[idx]

    c10 = trailing_counts_36a(
        ts, ev, 10
    )

    c60 = trailing_counts_36a(
        ts, ev, 60
    )

    b60 = trailing_counts_36a(
        ts, bev, 60
    )

    a60 = trailing_counts_36a(
        ts, aev, 60
    )

    count10[idx] = c10
    count60[idx] = c60
    bid_count60[idx] = b60
    ask_count60[idx] = a60

    # --------------------------------------------------------
    # Six non-overlapping 10-second blocks:
    #
    # [t-59,t-50], [t-49,t-40], ... [t-9,t]
    #
    # Calculate interval counts using cumulative sums.
    # --------------------------------------------------------

    prefix = np.concatenate([
        [0], np.cumsum(ev, dtype=np.int64)
    ])

    blocks = np.empty(
        (len(ts), 6), dtype=np.float64
    )

    for j in range(6):

        # Oldest block first, newest last.
        left_seconds = 59 - 10*j
        right_seconds = 50 - 10*j

        left_index = np.searchsorted(
            ts, ts - left_seconds,
            side="left"
        )

        right_index = np.searchsorted(
            ts, ts - right_seconds,
            side="right"
        )

        blocks[:, j] = (
            prefix[right_index]
            - prefix[left_index]
        )

    mean_blocks = blocks.mean(axis=1)
    variance_blocks = blocks.var(axis=1)

    fano = (
        variance_blocks
        / (mean_blocks + 1e-8)
    )

    # --------------------------------------------------------
    # Interarrival CV:
    #
    # First form intervals between genuine events.
    # A gap is assigned to its later event timestamp.
    #
    # For each current time, compute CV over gaps
    # whose later event occurred in the last 60s.
    #
    # One interval may start before the lookback
    # window. This is a historical, causal boundary
    # convention, not future leakage.
    # --------------------------------------------------------

    event_timestamps = ts[ev == 1]

    cv = np.full(
        len(ts), np.nan, dtype=np.float64
    )

    if len(event_timestamps) >= 4:

        later_event_ts = event_timestamps[1:]

        gaps = np.diff(
            event_timestamps
        ).astype(np.float64)

        gap_prefix = np.concatenate([
            [0.0],
            np.cumsum(gaps)
        ])

        gap_sq_prefix = np.concatenate([
            [0.0],
            np.cumsum(gaps**2)
        ])

        left = np.searchsorted(
            later_event_ts,
            ts - 59,
            side="left"
        )

        right = np.searchsorted(
            later_event_ts,
            ts,
            side="right"
        )

        number_gaps = right - left

        sufficient = number_gaps >= 3

        if sufficient.any():

            ng = number_gaps[
                sufficient
            ].astype(float)

            sum_gap = (
                gap_prefix[right[sufficient]]
                - gap_prefix[left[sufficient]]
            )

            sum_sq = (
                gap_sq_prefix[right[sufficient]]
                - gap_sq_prefix[left[sufficient]]
            )

            mean_gap = sum_gap / ng

            variance_gap = np.maximum(
                sum_sq / ng - mean_gap**2,
                0
            )

            cv[sufficient] = (
                np.sqrt(variance_gap)
                / (mean_gap + 1e-8)
            )

    # Remove early session values with insufficient
    # elapsed calendar history.
    warm = ts >= ts[0] + 59

    fano[~warm] = np.nan
    cv[~warm] = np.nan

    burstiness[idx] = fano
    interarrival_cv[idx] = cv

# ------------------------------------------------------------
# 4. Final candidate construction
# ------------------------------------------------------------

# Our definition compares the 10-second count
# against 1/6 of the 60-second count.

update_acceleration = np.log(
    (1.0 + count10)
    / (1.0 + count60 / 6.0)
)

side_imbalance = (
    (bid_count60 - ask_count60)
    / (bid_count60 + ask_count60 + 1e-8)
)

# Apply consistent 60-second warm-up by session.
for idx in session_groups.values():
    idx = np.sort(idx)
    warm = ts_all[idx] >= ts_all[idx[0]] + 59

    update_acceleration[idx[~warm]] = np.nan
    side_imbalance[idx[~warm]] = np.nan

calendar_exp36a_features = x[
    ["symbol", "ts_sec", "date"]
].copy()

arrays = {
    "update_acceleration_36a":
        update_acceleration,
    "event_burstiness_60_36a":
        burstiness,
    "interarrival_cv_60_36a":
        interarrival_cv,
    "side_update_imbalance_60_36a":
        side_imbalance
}

for name, values in arrays.items():
    arr = np.asarray(
        values, dtype=np.float64
    ).copy()
    arr[~np.isfinite(arr)] = np.nan
    calendar_exp36a_features[name] = (
        arr.astype(np.float32)
    )

# ------------------------------------------------------------
# 5. Exact future midprice-return targets
# ------------------------------------------------------------

bid1 = x["bid_p1"].to_numpy(dtype=float)
ask1 = x["ask_p1"].to_numpy(dtype=float)

valid_quote = (
    np.isfinite(bid1)
    & np.isfinite(ask1)
    & (bid1 > 0)
    & (ask1 >= bid1)
)

mid = np.where(
    valid_quote,
    (bid1 + ask1) / 2.0,
    np.nan
)

session_index = pd.MultiIndex.from_frame(
    x[SESSION_KEYS_36A]
)

for h in HORIZONS_36A:

    future_idx = pd.MultiIndex.from_arrays(
        [
            x["symbol"],
            x["date"],
            ts_all + h
        ],
        names=SESSION_KEYS_36A
    )

    positions = session_index.get_indexer(
        future_idx
    )

    found = positions >= 0
    future_mid = np.full(N, np.nan)

    future_mid[found] = mid[
        positions[found]
    ]

    with np.errstate(
        divide="ignore",
        invalid="ignore"
    ):
        y = 1e4 * np.log(
            future_mid / mid
        )

    y[~np.isfinite(y)] = np.nan

    calendar_exp36a_features[
        f"future_return_{h}_36a"
    ] = y.astype(np.float32)

    calendar_exp36a_features[
        f"future_absret_{h}_36a"
    ] = np.abs(y).astype(np.float32)

# ------------------------------------------------------------
# 6. Daily Spearman IC
# ------------------------------------------------------------

def daily_ic_36a(a, b):

    valid = (
        np.isfinite(a) & np.isfinite(b)
    )

    n = int(valid.sum())

    if n < 200:
        return np.nan, n

    sa = pd.Series(a[valid])
    sb = pd.Series(b[valid])

    if (
        sa.nunique() < 2
        or sb.nunique() < 2
    ):
        return np.nan, n

    return float(
        sa.corr(sb, method="spearman")
    ), n

records = []

for (symbol, date), day in (
    calendar_exp36a_features.groupby(
        ["symbol", "date"],
        sort=False
    )
):

    for h in HORIZONS_36A:

        for task in ["signed", "absret"]:

            target_name = (
                f"future_return_{h}_36a"
                if task == "signed"
                else f"future_absret_{h}_36a"
            )

            y = day[
                target_name
            ].to_numpy(dtype=float)

            for feature in CANDIDATES_36A:

                a = day[
                    feature
                ].to_numpy(dtype=float)

                ic, n = daily_ic_36a(a, y)

                records.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": n
                })

calendar_exp36a_daily = pd.DataFrame(records)

calendar_exp36a_summary = (
    calendar_exp36a_daily
    .groupby(
        ["task", "horizon", "feature"]
    )
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic",
            lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic",
            lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. Redundancy against the original 31
# ------------------------------------------------------------

redundancy_records = []

if "core_31" in globals():

    available_refs = [
        c for c in REFS_36A
        if c in core_31.columns
    ]

    source_idx = pd.MultiIndex.from_arrays(
        [
            core_31["symbol"].astype(str),
            pd.to_numeric(
                core_31["ts_sec"],
                errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS_36A
    )

    target_idx = pd.MultiIndex.from_frame(
        calendar_exp36a_features[KEYS_36A]
    )

    if source_idx.has_duplicates:
        raise ValueError(
            "Duplicate keys in core_31"
        )

    positions = source_idx.get_indexer(
        target_idx
    )

    found = positions >= 0

    for ref_col in available_refs:

        source_values = pd.to_numeric(
            core_31[ref_col],
            errors="coerce"
        ).to_numpy(dtype=float)

        reference = np.full(
            N, np.nan
        )

        reference[found] = source_values[
            positions[found]
        ]

        for candidate in CANDIDATES_36A:

            a = calendar_exp36a_features[
                candidate
            ].to_numpy(dtype=float)

            valid = (
                np.isfinite(a)
                & np.isfinite(reference)
            )

            corr = (
                pd.Series(a[valid]).corr(
                    pd.Series(reference[valid]),
                    method="spearman"
                )
                if valid.sum() >= 200
                else np.nan
            )

            redundancy_records.append({
                "candidate": candidate,
                "reference": ref_col,
                "spearman": corr
            })

calendar_exp36a_redundancy = (
    pd.DataFrame(redundancy_records)
)

# ------------------------------------------------------------
# 8. Diagnostics and printed results
# ------------------------------------------------------------

print("\n===== 36A FEATURE COVERAGE =====")

for feature in CANDIDATES_36A:

    s = calendar_exp36a_features[feature]

    print(
        feature,
        "valid_pct=",
        round(100 * s.notna().mean(), 4),
        "zero_pct=",
        round(100 * s.eq(0).mean(), 4),
        "std=",
        round(s.std(), 6),
        "median=",
        round(s.median(), 6)
    )

print("\n===== 36A SIGNED RETURN IC =====")

print(
    calendar_exp36a_summary.loc[
        calendar_exp36a_summary.task.eq(
            "signed"
        )
    ].round(6).to_string(index=False)
)

print("\n===== 36A ABSOLUTE RETURN IC =====")

print(
    calendar_exp36a_summary.loc[
        calendar_exp36a_summary.task.eq(
            "absret"
        )
    ].round(6).to_string(index=False)
)

print("\n===== 36A REDUNDANCY =====")

if not calendar_exp36a_redundancy.empty:

    print(
        calendar_exp36a_redundancy.pivot(
            index="candidate",
            columns="reference",
            values="spearman"
        ).round(5).to_string()
    )

else:
    print("core_31 unavailable")

print("\n===== 36A CANDIDATE CORRELATIONS =====")

print(
    calendar_exp36a_features[
        CANDIDATES_36A
    ].corr(method="spearman")
    .round(5)
    .to_string()
)

print("\n===== EXPERIMENT 36A COMPLETE =====")
print("Research rows:", N)
print("Original df_1s unchanged.")
print("Oct 21-23 untouched.")
print("Core remains 37 provisional features.")

gc.collect()


===== 36A RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Last date: 2026-10-19

===== 36A EVENT DIAGNOSTICS =====
Observed seconds: 269455
Book-change seconds: 194598
Bid-change seconds: 79628
Ask-change seconds: 147020
Both-sides change seconds: 32050

===== 36A FEATURE COVERAGE =====
update_acceleration_36a valid_pct= 99.7944 zero_pct= 8.1397 std= 0.501884 median= 0.0
event_burstiness_60_36a valid_pct= 99.7944 zero_pct= 5.2123 std= 0.923134 median= 0.7
interarrival_cv_60_36a valid_pct= 93.1206 zero_pct= 2.5754 std= 0.556781 median= 0.802315
side_update_imbalance_60_36a valid_pct= 99.7944 zero_pct= 7.2517 std= 0.579938 median= -0.2

===== 36A SIGNED RETURN IC =====
  task  horizon                      feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows
signed       30      event_burstiness_60_36a    16  0.009706   0.006995             10              6            28664.0
signed       30       interarrival_cv_60_36a    16  0.010321   0.010253           

0

In [27]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import gc

# ============================================================
# EXPERIMENT 36B — FULL BASELINE ABLATION
#
# Signed return:
#   A_full37
#   B_side_update_imbalance
#
# Absolute return:
#   A_full37
#   C_update_acceleration
#   D_event_burstiness
#   E_acceleration_burstiness
#
# Horizons: 30, 60, 300 seconds
#
# Required DataFrames:
#   usdinr_rebuilt_37_research
#   calendar_exp36a_features
#
# Outputs:
#   calendar_exp36b_summary
#   calendar_exp36b_daily
#   calendar_exp36b_daily_gains
#   calendar_exp36b_importance
#   calendar_exp36b_regime_summary
#
# Research only: through 2026-10-19.
# October 21-23 holdout excluded.
# No automatic feature promotion.
# ============================================================

HORIZONS_36B = [30, 60, 300]
MIN_TRAIN_DAYS_36B = 5
TRAIN_STRIDE_36B = 10

KEYS_36B = ["symbol", "ts_sec"]

CORE31_36B = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

ADDED6_36B = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9_36B = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

BASE_36B = (
    CORE31_36B
    + ADDED6_36B
    + CONTROLS9_36B
)

SIDE_36B = "side_update_imbalance_60_36a"
ACCEL_36B = "update_acceleration_36a"
BURST_36B = "event_burstiness_60_36a"

CANDIDATES_36B = [
    SIDE_36B,
    ACCEL_36B,
    BURST_36B
]

assert len(CORE31_36B) == 31
assert len(BASE_36B) == 46
assert len(set(BASE_36B)) == 46

PARAMS_36B = dict(
    objective="regression",
    n_estimators=180,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=5,
    min_child_samples=200,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

# ------------------------------------------------------------
# 1. Validate source DataFrames
# ------------------------------------------------------------

for name in [
    "usdinr_rebuilt_37_research",
    "calendar_exp36a_features"
]:
    if name not in globals():
        raise ValueError(
            f"{name} is missing. Load or generate "
            "this DataFrame before running 36B."
        )

baseline_source = usdinr_rebuilt_37_research
candidate_source = calendar_exp36a_features

TARGET_COLS_36B = [
    f"future_{kind}_{h}_36a"
    for kind in ["return", "absret"]
    for h in HORIZONS_36B
]

for label, source, required_cols in [
    ("baseline", baseline_source, BASE_36B),
    (
        "36A",
        candidate_source,
        CANDIDATES_36B + TARGET_COLS_36B
    )
]:
    missing = [
        c for c in KEYS_36B + required_cols
        if c not in source.columns
    ]

    if missing:
        raise ValueError(
            f"{label} is missing columns: {missing}"
        )

# ------------------------------------------------------------
# 2. Restrict research sample
# ------------------------------------------------------------

research = candidate_source[
    KEYS_36B + CANDIDATES_36B + TARGET_COLS_36B
].copy()

research["symbol"] = research["symbol"].astype(str)
research["ts_sec"] = pd.to_numeric(
    research["ts_sec"], errors="raise"
).astype(np.int64)

ist_dates = pd.to_datetime(
    research["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

research["date"] = ist_dates

research = research.loc[
    (research["date"].astype(str) <= "2026-10-19")
    & (research["date"].astype(str) != "2026-09-28")
].copy()

research = (
    research.sort_values(
        ["date", "symbol", "ts_sec"]
    )
    .reset_index(drop=True)
)

if research.empty:
    raise ValueError("Research sample is empty")

if research.duplicated(KEYS_36B).any():
    raise ValueError("Duplicate candidate timestamps")

N = len(research)

print("===== 36B RESEARCH SAMPLE =====")
print("Rows:", N)
print(
    "Sessions:",
    research.groupby(
        ["symbol", "date"]
    ).ngroups
)
print("First date:", research["date"].min())
print("Last date:", research["date"].max())

# ------------------------------------------------------------
# 3. Align reconstructed baseline by exact keys
#
# Avoid DataFrame merges and repeated copying.
# ------------------------------------------------------------

research_index = pd.MultiIndex.from_frame(
    research[KEYS_36B]
)

baseline_index = pd.MultiIndex.from_arrays(
    [
        baseline_source["symbol"].astype(str),
        pd.to_numeric(
            baseline_source["ts_sec"],
            errors="raise"
        ).astype(np.int64)
    ],
    names=KEYS_36B
)

if baseline_index.has_duplicates:
    raise ValueError(
        "Duplicate baseline timestamp keys"
    )

positions = baseline_index.get_indexer(
    research_index
)

found = positions >= 0

print(
    "Baseline key match:",
    f"{100 *found.mean():.4f}%"
)

if not found.all():
    raise ValueError(
        "Some research rows do not match the baseline"
    )

X_base = np.column_stack([
    pd.to_numeric(
        baseline_source[col],
        errors="coerce"
    ).to_numpy(dtype=np.float32)[positions]
    for col in BASE_36B
]).astype(np.float32)

X_base[~np.isfinite(X_base)] = np.nan

X_candidate = np.column_stack([
    pd.to_numeric(
        research[col],
        errors="coerce"
    ).to_numpy(dtype=np.float32)
    for col in CANDIDATES_36B
]).astype(np.float32)

X_candidate[~np.isfinite(X_candidate)] = np.nan

X = np.column_stack([
    X_base,
    X_candidate
]).astype(np.float32)

ALL_COLS_36B = BASE_36B + CANDIDATES_36B

column_index = {
    c: i for i, c in enumerate(ALL_COLS_36B)
}

base_idx = list(range(46))

MODELS_36B = {
    "signed": {
        "A_full37": base_idx,
        "B_side_imbalance": (
            base_idx
            + [column_index[SIDE_36B]]
        )
    },
    "absret": {
        "A_full37": base_idx,
        "C_acceleration": (
            base_idx
            + [column_index[ACCEL_36B]]
        ),
        "D_burstiness": (
            base_idx
            + [column_index[BURST_36B]]
        ),
        "E_accel_burst": (
            base_idx
            + [
                column_index[ACCEL_36B],
                column_index[BURST_36B]
            ]
        )
    }
}

# Restrict to rows with complete baseline inputs.
# For fairness, require all three candidate values
# to be valid as well. All models receive exactly
# the same train/test rows for their task/horizon.
#
# If candidate coverage is low, this will be
# reported and should be investigated.
complete_base = np.isfinite(
    X_base
).all(axis=1)

complete_all = (
    complete_base
    & np.isfinite(X_candidate).all(axis=1)
)

print("\n===== 36B MATCHED FEATURE SAMPLE =====")
print("Baseline complete:", int(complete_base.sum()))
print(
    "Baseline coverage (%):",
    round(100 * complete_base.mean(), 4)
)
print("All candidates complete:", int(complete_all.sum()))
print(
    "All-candidate coverage (%):",
    round(100 * complete_all.mean(), 4)
)
print(
    "Rows excluded by candidate completeness:",
    int(complete_base.sum() - complete_all.sum())
)

# ------------------------------------------------------------
# 4. Prepare targets and chronological dates
# ------------------------------------------------------------

Y = {}

for task in ["signed", "absret"]:

    prefix = (
        "future_return"
        if task == "signed"
        else "future_absret"
    )

    for h in HORIZONS_36B:

        Y[(task, h)] = pd.to_numeric(
            research[f"{prefix}_{h}_36a"],
            errors="coerce"
        ).to_numpy(dtype=np.float32)

dates = research["date"].to_numpy()
timestamps = research["ts_sec"].to_numpy(
    dtype=np.int64
)

unique_dates = sorted(pd.unique(dates))

if len(unique_dates) != 16:
    print(
        "WARNING: Expected 16 research dates, got",
        len(unique_dates)
    )

if len(unique_dates) <= MIN_TRAIN_DAYS_36B:
    raise ValueError("Not enough research dates")

# ------------------------------------------------------------
# 5. Walk-forward training
#
# Train on all earlier dates, sample each 10 seconds.
# Evaluate each subsequent date at full 1s density.
#
# Fixed LightGBM parameters, no horizon tuning.
# ------------------------------------------------------------

daily_records = []
importance_records = []

for day_i in range(
    MIN_TRAIN_DAYS_36B,
    len(unique_dates)
):

    test_date = unique_dates[day_i]

    train_base_mask = (
        np.isin(
            dates,
            unique_dates[:day_i]
        )
        & complete_all
        & (timestamps % TRAIN_STRIDE_36B == 0)
    )

    test_base_mask = (
        (dates == test_date)
        & complete_all
    )

    for task, model_specs in MODELS_36B.items():

        for h in HORIZONS_36B:

            y = Y[(task, h)]

            train_rows = np.flatnonzero(
                train_base_mask & np.isfinite(y)
            )

            test_rows = np.flatnonzero(
                test_base_mask & np.isfinite(y)
            )

            if (
                len(train_rows) < 1000
                or len(test_rows) < 200
            ):
                continue

            y_train = y[train_rows]
            y_test = y[test_rows].astype(np.float64)

            for model_name, selected in (
                model_specs.items()
            ):

                model = lgb.LGBMRegressor(
                    **PARAMS_36B
                )

                X_train = X[np.ix_(
                    train_rows, selected
                )]

                X_test = X[np.ix_(
                    test_rows, selected
                )]

                model.fit(X_train, y_train)

                pred = model.predict(X_test)

                err = y_test - pred

                daily_records.append({
                    "date": test_date,
                    "task": task,
                    "horizon": h,
                    "model": model_name,
                    "rows": len(test_rows),
                    "sse": float(np.sum(err**2)),
                    "sae": float(np.sum(np.abs(err))),
                    "mse": float(np.mean(err**2)),
                    "mae": float(np.mean(np.abs(err)))
                })

                if model_name != "A_full37":

                    gains = (
                        model.booster_
                        .feature_importance(
                            importance_type="gain"
                        )
                    )

                    total_gain = float(gains.sum())

                    for local_i, full_i in enumerate(
                        selected
                    ):

                        feature = ALL_COLS_36B[full_i]

                        if feature not in CANDIDATES_36B:
                            continue

                        importance_records.append({
                            "date": test_date,
                            "task": task,
                            "horizon": h,
                            "model": model_name,
                            "feature": feature,
                            "gain_share_pct": (
                                100 * float(gains[local_i])
                                / total_gain
                                if total_gain > 0
                                else 0.0
                            )
                        })

                del (
                    model, X_train, X_test,
                    pred, err
                )

    print("Completed:", test_date)

calendar_exp36b_daily = pd.DataFrame(
    daily_records
)

calendar_exp36b_importance = pd.DataFrame(
    importance_records
)

if calendar_exp36b_daily.empty:
    raise ValueError(
        "No walk-forward evaluations were produced"
    )

# ------------------------------------------------------------
# 6. Calculate matched model comparisons
# ------------------------------------------------------------

COMPARISONS_36B = {
    "signed": {
        "side_vs_full37": (
            "B_side_imbalance", "A_full37"
        )
    },
    "absret": {
        "accel_vs_full37": (
            "C_acceleration", "A_full37"
        ),
        "burst_vs_full37": (
            "D_burstiness", "A_full37"
        ),
        "both_vs_full37": (
            "E_accel_burst", "A_full37"
        ),
        "both_vs_accel": (
            "E_accel_burst", "C_acceleration"
        ),
        "both_vs_burst": (
            "E_accel_burst", "D_burstiness"
        )
    }
}

summary_records = []
daily_gain_records = []

for (task, h), part in (
    calendar_exp36b_daily.groupby(
        ["task", "horizon"]
    )
):

    pooled = part.groupby("model")[
        ["sse", "sae"]
    ].sum()

    mse_wide = part.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    mae_wide = part.pivot(
        index="date",
        columns="model",
        values="mae"
    )

    for comparison, (candidate, reference) in (
        COMPARISONS_36B[task].items()
    ):

        daily_mse = (
            100 * (
                1 - mse_wide[candidate]
                / mse_wide[reference]
            )
        )

        daily_mae = (
            100 * (
                1 - mae_wide[candidate]
                / mae_wide[reference]
            )
        )

        summary_records.append({
            "task": task,
            "horizon": h,
            "comparison": comparison,
            "days": len(daily_mse),
            "pooled_mse_gain_pct": 100 * (
                1
                - pooled.loc[candidate, "sse"]
                / pooled.loc[reference, "sse"]
            ),
            "pooled_mae_gain_pct": 100 * (
                1
                - pooled.loc[candidate, "sae"]
                / pooled.loc[reference, "sae"]
            ),
            "mse_win_days": int(
                (daily_mse > 0).sum()
            ),
            "mae_win_days": int(
                (daily_mae > 0).sum()
            ),
            "median_daily_mse_gain_pct":
                float(daily_mse.median()),
            "worst_daily_mse_gain_pct":
                float(daily_mse.min())
        })

        for date in daily_mse.index:

            daily_gain_records.append({
                "date": date,
                "task": task,
                "horizon": h,
                "comparison": comparison,
                "mse_gain_pct": float(
                    daily_mse.loc[date]
                ),
                "mae_gain_pct": float(
                    daily_mae.loc[date]
                )
            })

calendar_exp36b_summary = pd.DataFrame(
    summary_records
)

calendar_exp36b_daily_gains = pd.DataFrame(
    daily_gain_records
)

# ------------------------------------------------------------
# 7. Activity-regime diagnostic
#
# This is a diagnostic only, using information
# available at prediction time.
#
# Compare performance on low/high baseline
# update-rate states.
#
# NOTE:
# Regime thresholds need to be calculated on
# training days only for a deployable gate.
# We DO NOT perform a new regime optimization here.
# ------------------------------------------------------------

# This simple diagnostic tests candidate distribution
# and potential redundancy with update activity.
# It does not use future targets.

activity = X[:, column_index["update_rate_300s"]]

calendar_exp36b_regime_summary = pd.DataFrame({
    "metric": [
        "median_update_rate",
        "median_acceleration",
        "median_burstiness",
        "side_imbalance_std",
        "acceleration_activity_spearman",
        "burstiness_activity_spearman"
    ],
    "value": [
        float(np.nanmedian(activity)),
        float(np.nanmedian(
            X[:, column_index[ACCEL_36B]]
        )),
        float(np.nanmedian(
            X[:, column_index[BURST_36B]]
        )),
        float(np.nanstd(
            X[:, column_index[SIDE_36B]]
        )),
        pd.Series(
            X[:, column_index[ACCEL_36B]]
        ).corr(
            pd.Series(activity),
            method="spearman"
        ),
        pd.Series(
            X[:, column_index[BURST_36B]]
        ).corr(
            pd.Series(activity),
            method="spearman"
        )
    ]
})

# ------------------------------------------------------------
# 8. Print results
# ------------------------------------------------------------

print("\n===== 36B FULL37 ABLATION =====")

print(
    calendar_exp36b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 36B DAILY MSE GAINS =====")

for comparison in [
    "side_vs_full37",
    "accel_vs_full37",
    "burst_vs_full37",
    "both_vs_full37"
]:

    part = calendar_exp36b_daily_gains.loc[
        calendar_exp36b_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print("\n", comparison)

    print(
        part.pivot(
            index="date",
            columns="horizon",
            values="mse_gain_pct"
        )
        .round(6)
        .to_string()
    )

print("\n===== 36B DAILY MAE GAINS =====")

for comparison in [
    "side_vs_full37",
    "accel_vs_full37",
    "burst_vs_full37",
    "both_vs_full37"
]:

    part = calendar_exp36b_daily_gains.loc[
        calendar_exp36b_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print("\n", comparison)

    print(
        part.pivot(
            index="date",
            columns="horizon",
            values="mae_gain_pct"
        )
        .round(6)
        .to_string()
    )

print("\n===== 36B FEATURE IMPORTANCE =====")

print(
    calendar_exp36b_importance
    .groupby([
        "task", "horizon", "model", "feature"
    ])["gain_share_pct"]
    .agg(["mean", "median"])
    .round(6)
    .to_string()
)

print("\n===== 36B ACTIVITY DIAGNOSTICS =====")

print(
    calendar_exp36b_regime_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== EXPERIMENT 36B COMPLETE =====")
print("Baseline engineered features: 37")
print("Baseline controls: 9")
print("Candidates tested:", CANDIDATES_36B)
print("New features promoted: 0")
print("October 21-23 untouched.")

gc.collect()


===== 36B RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
First date: 2026-09-23
Last date: 2026-10-19
Baseline key match: 100.0000%

===== 36B MATCHED FEATURE SAMPLE =====
Baseline complete: 402979
Baseline coverage (%): 87.7717
All candidates complete: 402979
All-candidate coverage (%): 87.7717
Rows excluded by candidate completeness: 0
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 36B FULL37 ABLATION =====
  task  horizon      comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  mse_win_days  mae_win_days  median_daily_mse_gain_pct  worst_daily_mse_gain_pct
absret       30 accel_vs_full37    11            -0.175973            -0.337153             3             3                  -0.032045                 -1.287818
absret       30 burst_vs_full37    11             0.030927    

14

In [28]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# EXPERIMENT 37A — LIQUIDITY ENTROPY AND CONCENTRATION
#
# Required:
#   df_1s
#
# Optional:
#   usdinr_rebuilt_37_research
#   core_31
#
# Outputs:
#   calendar_exp37a_features
#   calendar_exp37a_summary
#   calendar_exp37a_daily
#   calendar_exp37a_redundancy
#
# Research: original 16 eligible sessions ONLY.
# October 21-23 are not accessed.
# No new core feature promotion.
# ============================================================

HORIZONS_37A = (30, 60, 300)
EPS_37A = 1e-12

FEATURES_37A = [
    "entropy_asymmetry_37a",
    "concentration_asymmetry_37a",
    "entropy_change_asym_60_37a",
    "entropy_instability_60_37a",
    "entropy_weighted_pressure_37a",
    "mean_book_entropy_37a"
]

REFERENCE_FEATURES_37A = [
    "depth_imbalance_l5",
    "queue_imbalance_l1",
    "depth_imbalance_delta",
    "queue_replenishment_pressure_60s",
    "log_rv_300s_new",
    "spread_rank_1800s",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "fragility_asymmetry_29a",
    "weighted_gap_asymmetry_29a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "log_total_depth_28a",
    "spread_bps_28a"
]

KEYS_37A = ["symbol", "ts_sec"]

QCOLS_37A = (
    [f"bid_q{i}" for i in range(1, 6)]
    + [f"ask_q{i}" for i in range(1, 6)]
)

RAW_COLS_37A = (
    KEYS_37A
    + ["bid_p1", "ask_p1"]
    + QCOLS_37A
)

missing = [
    c for c in RAW_COLS_37A
    if c not in df_1s.columns
]

if missing:
    raise ValueError(
        f"Missing raw columns: {missing}"
    )

# ------------------------------------------------------------
# 1. Restrict to historical development sessions
# ------------------------------------------------------------

x37 = df_1s[RAW_COLS_37A].copy()

x37["symbol"] = x37["symbol"].astype(str)
x37["ts_sec"] = pd.to_numeric(
    x37["ts_sec"], errors="raise"
).astype("int64")

ist = pd.to_datetime(
    x37["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x37["date"] = ist.dt.date

clock = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 32400
)

x37 = x37.loc[
    (x37["date"].astype(str) <= "2026-10-19")
    & (x37["date"].astype(str) != "2026-09-28")
    & clock.between(0, 28799)
].copy()

coverage = (
    x37.groupby(["symbol", "date"])["ts_sec"]
    .transform("nunique")
)

x37 = (
    x37.loc[
        coverage >= 0.97 * 28800
    ]
    .sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True)
)

if x37.empty:
    raise ValueError("No eligible research data")

if x37.duplicated(KEYS_37A).any():
    raise ValueError("Duplicate symbol/timestamp keys")

for c in RAW_COLS_37A[2:]:
    x37[c] = pd.to_numeric(
        x37[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

N = len(x37)

print("===== 37A RESEARCH SAMPLE =====")
print("Rows:", N)
print("Sessions:", x37.groupby(
    ["symbol", "date"]
).ngroups)
print("Last date:", x37["date"].max())

# ------------------------------------------------------------
# 2. Five-level liquidity probability distributions
# ------------------------------------------------------------

BQ = x37[
    [f"bid_q{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

AQ = x37[
    [f"ask_q{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

valid_bid = (
    np.isfinite(BQ).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (BQ.sum(axis=1) > 0)
)

valid_ask = (
    np.isfinite(AQ).all(axis=1)
    & (AQ >= 0).all(axis=1)
    & (AQ.sum(axis=1) > 0)
)

valid_book = valid_bid & valid_ask

bid_depth = BQ.sum(axis=1)
ask_depth = AQ.sum(axis=1)

with np.errstate(
    divide="ignore", invalid="ignore"
):
    PB = BQ / bid_depth[:, None]
    PA = AQ / ask_depth[:, None]

PB[~valid_bid] = np.nan
PA[~valid_ask] = np.nan

# 0*log(0) is defined as 0 for entropy.
# Use np.where with safe clipped logarithm.

def entropy_37a(P):
    with np.errstate(
        divide="ignore", invalid="ignore"
    ):
        terms = np.where(
            P > 0,
            P * np.log(np.clip(P, EPS_37A, 1.0)),
            0.0
        )
        result = -terms.sum(axis=1) / np.log(5.0)

    result[~np.isfinite(P).all(axis=1)] = np.nan

    return result

Hb = entropy_37a(PB)
Ha = entropy_37a(PA)

Cb = np.sum(PB**2, axis=1)
Ca = np.sum(PA**2, axis=1)

entropy_asym = Hb - Ha
concentration_asym = Cb - Ca

mean_entropy = (Hb + Ha) / 2.0

with np.errstate(
    divide="ignore", invalid="ignore"
):
    depth_imbalance = (
        (bid_depth - ask_depth)
        / (bid_depth + ask_depth + EPS_37A)
    )

entropy_pressure = (
    depth_imbalance
    * (1.0 - mean_entropy)
)

for arr in [
    entropy_asym,
    concentration_asym,
    mean_entropy,
    entropy_pressure
]:
    arr[~valid_book] = np.nan

print("\n===== 37A STATIC ENTROPY DIAGNOSTICS =====")

print(
    "Valid book (%):",
    round(100 * valid_book.mean(), 5)
)

print(
    "Median bid entropy:",
    round(float(np.nanmedian(Hb)), 6)
)

print(
    "Median ask entropy:",
    round(float(np.nanmedian(Ha)), 6)
)

print(
    "Median concentration (bid/ask):",
    round(float(np.nanmedian(Cb)), 6),
    round(float(np.nanmedian(Ca)), 6)
)

# ------------------------------------------------------------
# 3. Exact historical timestamp matching
# ------------------------------------------------------------

session_index = pd.MultiIndex.from_frame(
    x37[["symbol", "date", "ts_sec"]]
)

timestamps = x37["ts_sec"].to_numpy(
    dtype=np.int64
)

def exact_lookup_37a(values, offset):
    lookup_index = pd.MultiIndex.from_arrays(
        [
            x37["symbol"],
            x37["date"],
            timestamps + offset
        ],
        names=["symbol", "date", "ts_sec"]
    )

    positions = session_index.get_indexer(
        lookup_index
    )

    result = np.full(N, np.nan, dtype=float)

    valid = positions >= 0
    result[valid] = values[
        positions[valid]
    ]

    return result

Hb_60 = exact_lookup_37a(Hb, -60)
Ha_60 = exact_lookup_37a(Ha, -60)

entropy_change_asym = (
    (Hb - Hb_60) - (Ha - Ha_60)
)

entropy_change_asym[
    ~valid_book
] = np.nan

# ------------------------------------------------------------
# 4. Trailing 60-clock-second entropy instability
#
# For each session calculate the rolling population
# std from cumulative sums, with exact time bounds.
#
# At least 45 valid snapshots in the 60-second
# calendar window, and at least 59 elapsed seconds.
#
# All calculations use observations <= t.
# ------------------------------------------------------------

entropy_instability = np.full(
    N, np.nan, dtype=float
)

session_groups = x37.groupby(
    ["symbol", "date"], sort=False
).indices

for _, positions in session_groups.items():

    pos = np.sort(positions)
    ts = timestamps[pos]
    z = mean_entropy[pos]

    finite = np.isfinite(z)
    safe = np.where(finite, z, 0.0)

    cumulative_n = np.concatenate(
        [[0], np.cumsum(finite.astype(np.int64))]
    )

    cumulative_sum = np.concatenate(
        [[0.0], np.cumsum(safe)]
    )

    cumulative_sq = np.concatenate(
        [[0.0], np.cumsum(safe**2)]
    )

    left = np.searchsorted(
        ts, ts - 59, side="left"
    )

    right = np.arange(len(ts)) + 1

    count = cumulative_n[right] - cumulative_n[left]
    total = cumulative_sum[right] - cumulative_sum[left]
    total_sq = cumulative_sq[right] - cumulative_sq[left]

    eligible = (
        (count >= 45)
        & (ts >= ts[0] + 59)
    )

    result = np.full(
        len(ts), np.nan, dtype=float
    )

    count_safe = np.maximum(count, 1)

    mu = total / count_safe

    variance = np.maximum(
        total_sq / count_safe - mu**2,
        0.0
    )

    result[eligible] = np.sqrt(
        variance[eligible]
    )

    entropy_instability[pos] = result

# ------------------------------------------------------------
# 5. Construct compact candidate feature frame
# ------------------------------------------------------------

calendar_exp37a_features = x37[
    ["symbol", "ts_sec", "date"]
].copy()

candidate_arrays = {
    "entropy_asymmetry_37a":
        entropy_asym,

    "concentration_asymmetry_37a":
        concentration_asym,

    "entropy_change_asym_60_37a":
        entropy_change_asym,

    "entropy_instability_60_37a":
        entropy_instability,

    "entropy_weighted_pressure_37a":
        entropy_pressure,

    "mean_book_entropy_37a":
        mean_entropy
}

for col, values in candidate_arrays.items():

    a = np.asarray(values, dtype=float).copy()
    a[~np.isfinite(a)] = np.nan

    calendar_exp37a_features[col] = (
        a.astype(np.float32)
    )

# ------------------------------------------------------------
# 6. Exact future signed/absolute returns
# ------------------------------------------------------------

bid1 = x37["bid_p1"].to_numpy(dtype=float)
ask1 = x37["ask_p1"].to_numpy(dtype=float)

valid_quotes = (
    np.isfinite(bid1)
    & np.isfinite(ask1)
    & (bid1 > 0)
    & (ask1 >= bid1)
)

mid = np.where(
    valid_quotes,
    (bid1 + ask1) / 2.0,
    np.nan
)

for horizon in HORIZONS_37A:

    future_mid = exact_lookup_37a(
        mid, horizon
    )

    with np.errstate(
        divide="ignore", invalid="ignore"
    ):
        future_return = (
            1e4 * np.log(future_mid / mid)
        )

    future_return[
        ~np.isfinite(future_return)
    ] = np.nan

    calendar_exp37a_features[
        f"future_return_{horizon}_37a"
    ] = future_return.astype(np.float32)

    calendar_exp37a_features[
        f"future_absret_{horizon}_37a"
    ] = np.abs(
        future_return
    ).astype(np.float32)

# ------------------------------------------------------------
# 7. Daily Spearman IC
# ------------------------------------------------------------

def spearman_37a(a, b):

    mask = (
        np.isfinite(a)
        & np.isfinite(b)
    )

    count = int(mask.sum())

    if count < 200:
        return np.nan, count

    sa = pd.Series(a[mask])
    sb = pd.Series(b[mask])

    if sa.nunique() < 2 or sb.nunique() < 2:
        return np.nan, count

    return (
        float(sa.corr(sb, method="spearman")),
        count
    )

daily_results = []

for (symbol, date), day in (
    calendar_exp37a_features.groupby(
        ["symbol", "date"],
        sort=False
    )
):

    for horizon in HORIZONS_37A:

        for task in ["signed", "absret"]:

            y_name = (
                f"future_return_{horizon}_37a"
                if task == "signed"
                else f"future_absret_{horizon}_37a"
            )

            y = day[y_name].to_numpy(dtype=float)

            for name in FEATURES_37A:

                feature = day[name].to_numpy(
                    dtype=float
                )

                ic, count = spearman_37a(
                    feature, y
                )

                daily_results.append({
                    "symbol": symbol,
                    "date": date,
                    "task": task,
                    "horizon": horizon,
                    "feature": name,
                    "ic": ic,
                    "valid_rows": count
                })

calendar_exp37a_daily = pd.DataFrame(
    daily_results
)

calendar_exp37a_summary = (
    calendar_exp37a_daily
    .groupby(["task", "horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 8. Redundancy against existing engineered features
# ------------------------------------------------------------

redundancy_records = []

if "usdinr_rebuilt_37_research" in globals():
    reference_source = usdinr_rebuilt_37_research
elif "core_31" in globals():
    reference_source = core_31
else:
    reference_source = None

if reference_source is not None:

    reference_idx = pd.MultiIndex.from_arrays(
        [
            reference_source["symbol"].astype(str),
            pd.to_numeric(
                reference_source["ts_sec"],
                errors="raise"
            ).astype(np.int64)
        ],
        names=["symbol", "ts_sec"]
    )

    target_idx = pd.MultiIndex.from_frame(
        calendar_exp37a_features[
            ["symbol", "ts_sec"]
        ]
    )

    if reference_idx.has_duplicates:
        raise ValueError("Duplicate reference keys")

    indices = reference_idx.get_indexer(
        target_idx
    )

    found = indices >= 0

    print("\nReference match:",
          f"{100*found.mean():.4f}%")

    for ref_col in REFERENCE_FEATURES_37A:

        if ref_col not in reference_source.columns:
            continue

        values = pd.to_numeric(
            reference_source[ref_col],
            errors="coerce"
        ).to_numpy(dtype=float)

        aligned = np.full(N, np.nan)

        aligned[found] = values[
            indices[found]
        ]

        for feature_name in FEATURES_37A:

            a = calendar_exp37a_features[
                feature_name
            ].to_numpy(dtype=float)

            mask = (
                np.isfinite(a)
                & np.isfinite(aligned)
            )

            corr = (
                pd.Series(a[mask]).corr(
                    pd.Series(aligned[mask]),
                    method="spearman"
                )
                if mask.sum() >= 200
                else np.nan
            )

            redundancy_records.append({
                "candidate": feature_name,
                "reference": ref_col,
                "spearman": corr
            })

calendar_exp37a_redundancy = (
    pd.DataFrame(redundancy_records)
)

# ------------------------------------------------------------
# 9. Final diagnostic outputs
# ------------------------------------------------------------

print("\n===== 37A FEATURE COVERAGE =====")

for col in FEATURES_37A:

    s = calendar_exp37a_features[col]

    print(
        col,
        "valid_pct=",
        round(100*s.notna().mean(), 4),
        "std=",
        round(s.std(), 6),
        "median=",
        round(s.median(), 6)
    )

print("\n===== 37A SIGNED RETURN IC =====")

print(
    calendar_exp37a_summary.loc[
        calendar_exp37a_summary.task.eq("signed")
    ].round(6).to_string(index=False)
)

print("\n===== 37A ABSOLUTE RETURN IC =====")

print(
    calendar_exp37a_summary.loc[
        calendar_exp37a_summary.task.eq("absret")
    ].round(6).to_string(index=False)
)

print("\n===== 37A REDUNDANCY =====")

if not calendar_exp37a_redundancy.empty:

    print(
        calendar_exp37a_redundancy.pivot(
            index="candidate",
            columns="reference",
            values="spearman"
        )
        .round(5)
        .to_string()
    )
else:
    print("No reference features available")

print("\n===== 37A CANDIDATE CORRELATIONS =====")

print(
    calendar_exp37a_features[FEATURES_37A]
    .corr(method="spearman")
    .round(5)
    .to_string()
)

print("\n===== EXPERIMENT 37A COMPLETE =====")
print("Research sessions:", len(session_groups))
print("Research rows:", N)
print("Holdout October 21-23 excluded.")
print("Core remains 37 provisional features.")

del BQ, AQ, PB, PA
gc.collect()


===== 37A RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Last date: 2026-10-19

===== 37A STATIC ENTROPY DIAGNOSTICS =====
Valid book (%): 100.0
Median bid entropy: 0.717157
Median ask entropy: 0.73128
Median concentration (bid/ask): 0.371524 0.365237

Reference match: 100.0000%

===== 37A FEATURE COVERAGE =====
entropy_asymmetry_37a valid_pct= 100.0 std= 0.231386 median= -0.012046
concentration_asymmetry_37a valid_pct= 100.0 std= 0.19584 median= 0.004695
entropy_change_asym_60_37a valid_pct= 99.7909 std= 0.219975 median= 0.0
entropy_instability_60_37a valid_pct= 99.7944 std= 0.029425 median= 0.045091
entropy_weighted_pressure_37a valid_pct= 100.0 std= 0.134193 median= 0.034824
mean_book_entropy_37a valid_pct= 100.0 std= 0.113598 median= 0.71033

===== 37A SIGNED RETURN IC =====
  task  horizon                       feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows
signed       30   concentration_asymmetry_37a    16 -0.050390  -0.046490              0

29

In [29]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import gc

# ============================================================
# EXPERIMENT 37B — LIQUIDITY ENTROPY FULL MODEL ABLATION
#
# Baseline: 37 engineered features + 9 controls
#          = 46 LightGBM inputs
#
# Signed:
#   A_full37
#   B_entropy_asym
#   C_entropy_change
#   D_both_entropy
#
# Absolute:
#   A_full37
#   E_entropy_instability
#
# 16 research days: first 5 train, next 11 evaluate.
# No Oct 21-23 holdout access.
# ============================================================

HORIZONS = [30, 60, 300]
KEYS = ["symbol", "ts_sec"]
FIRST_EVAL_DAY = 5

ENT_ASYM = "entropy_asymmetry_37a"
ENT_CHANGE = "entropy_change_asym_60_37a"
ENT_INST = "entropy_instability_60_37a"

CORE31 = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

ADDED6 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

BASE_FEATURES = CORE31 + ADDED6 + CONTROLS9

assert len(BASE_FEATURES) == 46
assert len(set(BASE_FEATURES)) == 46

PARAMS = dict(
    objective="regression",
    n_estimators=180,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=5,
    min_child_samples=200,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

# ------------------------------------------------------------
# 1. Align research frames
# ------------------------------------------------------------

base = usdinr_rebuilt_37_research
new = calendar_exp37a_features

target_cols = [
    f"future_{kind}_{h}_37a"
    for kind in ["return", "absret"]
    for h in HORIZONS
]

needed_new = [
    ENT_ASYM, ENT_CHANGE, ENT_INST
] + target_cols

for label, frame, cols in [
    ("baseline", base, BASE_FEATURES),
    ("entropy", new, needed_new)
]:
    missing = [
        c for c in KEYS + cols
        if c not in frame.columns
    ]
    if missing:
        raise ValueError(
            f"{label} missing: {missing}"
        )

research = new[
    KEYS + needed_new
].copy()

research["symbol"] = (
    research["symbol"].astype(str)
)

research["ts_sec"] = pd.to_numeric(
    research["ts_sec"], errors="raise"
).astype(np.int64)

research["date"] = pd.to_datetime(
    research["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

research = research.loc[
    (research["date"].astype(str) <= "2026-10-19")
    & (research["date"].astype(str) != "2026-09-28")
].sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

assert not research.duplicated(KEYS).any()
assert len(research) == 459122

base_idx = pd.MultiIndex.from_arrays(
    [
        base["symbol"].astype(str),
        pd.to_numeric(
            base["ts_sec"], errors="raise"
        ).astype(np.int64)
    ],
    names=KEYS
)

research_idx = pd.MultiIndex.from_frame(
    research[KEYS]
)

assert not base_idx.has_duplicates

positions = base_idx.get_indexer(
    research_idx
)

assert (positions >= 0).all(), (
    "Research keys are missing in baseline"
)

# ------------------------------------------------------------
# 2. Build memory-efficient matrices
# ------------------------------------------------------------

X_base = np.column_stack([
    pd.to_numeric(
        base[c], errors="coerce"
    ).to_numpy(dtype=np.float32)[positions]
    for c in BASE_FEATURES
])

X_new = research[
    [ENT_ASYM, ENT_CHANGE, ENT_INST]
].to_numpy(dtype=np.float32)

X_base[~np.isfinite(X_base)] = np.nan
X_new[~np.isfinite(X_new)] = np.nan

# One concatenated matrix and named indices.
X = np.column_stack(
    [X_base, X_new]
).astype(np.float32)

ALL_FEATURES = BASE_FEATURES + [
    ENT_ASYM, ENT_CHANGE, ENT_INST
]

ix = {
    c: i for i, c in enumerate(ALL_FEATURES)
}

A = list(range(46))

MODELS = {
    "signed": {
        "A_full37": A,
        "B_entropy_asym": A + [ix[ENT_ASYM]],
        "C_entropy_change": A + [ix[ENT_CHANGE]],
        "D_both_entropy": A + [
            ix[ENT_ASYM], ix[ENT_CHANGE]
        ]
    },
    "absret": {
        "A_full37": A,
        "E_entropy_instability": A + [
            ix[ENT_INST]
        ]
    }
}

# Matched completeness WITHIN each task only.
# Signed models use signed-feature common rows.
# Absolute models use instability common rows.
# An unrelated candidate cannot exclude rows.

complete_baseline = np.isfinite(
    X_base
).all(axis=1)

complete_signed = (
    complete_baseline
    & np.isfinite(X_new[:, 0])
    & np.isfinite(X_new[:, 1])
)

complete_absret = (
    complete_baseline
    & np.isfinite(X_new[:, 2])
)

MASKS = {
    "signed": complete_signed,
    "absret": complete_absret
}

print("===== 37B MATCHED FEATURE SAMPLE =====")
print("Research rows:", len(research))
print(
    "Baseline complete:",
    int(complete_baseline.sum())
)
print(
    "Signed matched:",
    int(complete_signed.sum())
)
print(
    "Absolute matched:",
    int(complete_absret.sum())
)
print(
    "Signed lost vs baseline:",
    int(
        complete_baseline.sum()
        - complete_signed.sum()
    )
)
print(
    "Absolute lost vs baseline:",
    int(
        complete_baseline.sum()
        - complete_absret.sum()
    )
)

# ------------------------------------------------------------
# 3. Exact targets and evaluation dates
# ------------------------------------------------------------

Y = {}

for task in ["signed", "absret"]:
    kind = (
        "return" if task == "signed"
        else "absret"
    )

    for h in HORIZONS:
        col = f"future_{kind}_{h}_37a"
        Y[(task, h)] = pd.to_numeric(
            research[col], errors="coerce"
        ).to_numpy(dtype=np.float32)

dates = research["date"].to_numpy()
ts = research["ts_sec"].to_numpy(
    dtype=np.int64
)

unique_dates = sorted(pd.unique(dates))

assert len(unique_dates) == 16, (
    f"Expected 16 dates, got {len(unique_dates)}"
)

# ------------------------------------------------------------
# 4. Walk-forward model evaluation
# ------------------------------------------------------------

daily_results = []
importance_results = []

for d in range(
    FIRST_EVAL_DAY,
    len(unique_dates)
):
    test_day = unique_dates[d]

    historical = np.isin(
        dates, unique_dates[:d]
    )

    current = dates == test_day

    for task, model_specs in MODELS.items():

        complete = MASKS[task]

        train_mask = (
            historical
            & complete
            & (ts % 10 == 0)
        )

        test_mask = current & complete

        for h in HORIZONS:

            y = Y[(task, h)]

            train_rows = np.flatnonzero(
                train_mask & np.isfinite(y)
            )

            test_rows = np.flatnonzero(
                test_mask & np.isfinite(y)
            )

            if len(train_rows) < 1000:
                raise ValueError(
                    f"Too few train rows: {test_day}, "
                    f"{task}, {h}"
                )

            if len(test_rows) < 200:
                raise ValueError(
                    f"Too few test rows: {test_day}, "
                    f"{task}, {h}"
                )

            y_train = y[train_rows]
            y_test = y[test_rows].astype(
                np.float64
            )

            for model_name, cols in (
                model_specs.items()
            ):

                model = lgb.LGBMRegressor(
                    **PARAMS
                )

                train_X = X[np.ix_(
                    train_rows, cols
                )]

                test_X = X[np.ix_(
                    test_rows, cols
                )]

                model.fit(train_X, y_train)

                prediction = model.predict(
                    test_X
                )

                error = y_test - prediction

                daily_results.append({
                    "date": test_day,
                    "task": task,
                    "horizon": h,
                    "model": model_name,
                    "rows": len(test_rows),
                    "sse": float(
                        np.sum(error**2)
                    ),
                    "sae": float(
                        np.sum(np.abs(error))
                    ),
                    "mse": float(
                        np.mean(error**2)
                    ),
                    "mae": float(
                        np.mean(np.abs(error))
                    )
                })

                # Incremental feature importance,
                # normalized within each trained model.

                if model_name != "A_full37":

                    gain = (
                        model.booster_
                        .feature_importance(
                            importance_type="gain"
                        )
                    )

                    total = float(gain.sum())

                    for local_i, global_i in enumerate(
                        cols
                    ):
                        feature = ALL_FEATURES[
                            global_i
                        ]

                        if feature not in [
                            ENT_ASYM,
                            ENT_CHANGE,
                            ENT_INST
                        ]:
                            continue

                        importance_results.append({
                            "date": test_day,
                            "task": task,
                            "horizon": h,
                            "model": model_name,
                            "feature": feature,
                            "gain_share_pct": (
                                100 * float(
                                    gain[local_i]
                                ) / total
                                if total > 0
                                else 0.0
                            )
                        })

                del (
                    model, train_X, test_X,
                    prediction, error
                )

    print("Completed:", test_day)

calendar_exp37b_daily = pd.DataFrame(
    daily_results
)

calendar_exp37b_importance = pd.DataFrame(
    importance_results
)

# ------------------------------------------------------------
# 5. Paired pooled improvements
# ------------------------------------------------------------

COMPARISONS = {
    "signed": {
        "entropy_asym_vs_base": (
            "B_entropy_asym", "A_full37"
        ),
        "entropy_change_vs_base": (
            "C_entropy_change", "A_full37"
        ),
        "both_entropy_vs_base": (
            "D_both_entropy", "A_full37"
        ),
        "both_vs_asym": (
            "D_both_entropy", "B_entropy_asym"
        ),
        "both_vs_change": (
            "D_both_entropy", "C_entropy_change"
        )
    },
    "absret": {
        "entropy_instability_vs_base": (
            "E_entropy_instability",
            "A_full37"
        )
    }
}

summary = []
daily_gains = []

for (task, h), frame in (
    calendar_exp37b_daily.groupby(
        ["task", "horizon"]
    )
):

    pooled = frame.groupby(
        "model"
    )[["sse", "sae"]].sum()

    mse = frame.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    mae = frame.pivot(
        index="date",
        columns="model",
        values="mae"
    )

    for comparison, (candidate, reference) in (
        COMPARISONS[task].items()
    ):

        mse_gain = 100 * (
            1 - mse[candidate] / mse[reference]
        )

        mae_gain = 100 * (
            1 - mae[candidate] / mae[reference]
        )

        pooled_mse = 100 * (
            1
            - pooled.loc[candidate, "sse"]
            / pooled.loc[reference, "sse"]
        )

        pooled_mae = 100 * (
            1
            - pooled.loc[candidate, "sae"]
            / pooled.loc[reference, "sae"]
        )

        summary.append({
            "task": task,
            "horizon": h,
            "comparison": comparison,
            "days": len(mse_gain),
            "pooled_mse_gain_pct": pooled_mse,
            "pooled_mae_gain_pct": pooled_mae,
            "mse_win_days": int(
                (mse_gain > 0).sum()
            ),
            "mae_win_days": int(
                (mae_gain > 0).sum()
            ),
            "median_daily_mse_gain_pct":
                float(mse_gain.median()),
            "worst_daily_mse_gain_pct":
                float(mse_gain.min())
        })

        for day in mse_gain.index:

            daily_gains.append({
                "date": day,
                "task": task,
                "horizon": h,
                "comparison": comparison,
                "mse_gain_pct": float(
                    mse_gain.loc[day]
                ),
                "mae_gain_pct": float(
                    mae_gain.loc[day]
                )
            })

calendar_exp37b_summary = pd.DataFrame(
    summary
)

calendar_exp37b_daily_gains = pd.DataFrame(
    daily_gains
)

# ------------------------------------------------------------
# 6. Additional redundancy diagnostics
#
# Check entropy instability against existing
# activity / volatility features in baseline.
# ------------------------------------------------------------

DIAGNOSTIC_REFS = [
    "update_rate_300s",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "price_move_rate_30s",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "spread_rank_1800s"
]

diagnostics = []

instability = X[:, ix[ENT_INST]]

for ref in DIAGNOSTIC_REFS:

    values = X[:, ix[ref]]

    valid = (
        np.isfinite(instability)
        & np.isfinite(values)
    )

    correlation = pd.Series(
        instability[valid]
    ).corr(
        pd.Series(values[valid]),
        method="spearman"
    )

    diagnostics.append({
        "reference": ref,
        "spearman": correlation,
        "matched_rows": int(valid.sum())
    })

calendar_exp37b_redundancy = pd.DataFrame(
    diagnostics
)

# ------------------------------------------------------------
# 7. Print all results
# ------------------------------------------------------------

print("\n===== 37B FULL MODEL ABLATION =====")

print(
    calendar_exp37b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 37B DAILY MSE GAINS =====")

for comparison in [
    "entropy_asym_vs_base",
    "entropy_change_vs_base",
    "both_entropy_vs_base",
    "entropy_instability_vs_base"
]:

    part = calendar_exp37b_daily_gains.loc[
        calendar_exp37b_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print("\n", comparison)

    print(
        part.pivot(
            index="date",
            columns="horizon",
            values="mse_gain_pct"
        ).round(6).to_string()
    )

print("\n===== 37B DAILY MAE GAINS =====")

for comparison in [
    "entropy_asym_vs_base",
    "entropy_change_vs_base",
    "both_entropy_vs_base",
    "entropy_instability_vs_base"
]:

    part = calendar_exp37b_daily_gains.loc[
        calendar_exp37b_daily_gains[
            "comparison"
        ].eq(comparison)
    ]

    print("\n", comparison)

    print(
        part.pivot(
            index="date",
            columns="horizon",
            values="mae_gain_pct"
        ).round(6).to_string()
    )

print("\n===== 37B FEATURE IMPORTANCE =====")

print(
    calendar_exp37b_importance
    .groupby([
        "task", "horizon",
        "model", "feature"
    ])["gain_share_pct"]
    .agg(["mean", "median"])
    .round(6)
    .to_string()
)

print("\n===== 37B ENTROPY INSTABILITY REDUNDANCY =====")

print(
    calendar_exp37b_redundancy
    .round(6)
    .to_string(index=False)
)

print("\n===== EXPERIMENT 37B COMPLETE =====")
print("Research days:", len(unique_dates))
print(
    "Evaluation days:",
    len(unique_dates) - FIRST_EVAL_DAY
)
print("Baseline features: 37 + 9 controls")
print("New features promoted: 0")
print("Oct 21-23 holdout untouched.")

gc.collect()


===== 37B MATCHED FEATURE SAMPLE =====
Research rows: 459122
Baseline complete: 402979
Signed matched: 402979
Absolute matched: 402979
Signed lost vs baseline: 0
Absolute lost vs baseline: 0
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 37B FULL MODEL ABLATION =====
  task  horizon                  comparison  days  pooled_mse_gain_pct  pooled_mae_gain_pct  mse_win_days  mae_win_days  median_daily_mse_gain_pct  worst_daily_mse_gain_pct
absret       30 entropy_instability_vs_base    11            -0.119631             0.081532             2             6                  -0.126114                 -0.363645
absret       60 entropy_instability_vs_base    11            -0.023354             0.081498             3             8                  -0.194714                 -0.732021
absret     

0

In [30]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# EXPERIMENT 38A — BEST-QUEUE DEPLETION
#
# Required:
#   df_1s
#
# Optional:
#   usdinr_rebuilt_37_research
#
# Outputs:
#   calendar_exp38a_features
#   calendar_exp38a_daily
#   calendar_exp38a_summary
#   calendar_exp38a_redundancy
#
# No holdout access. No automatic feature promotion.
# ============================================================

HORIZONS = [30, 60, 300]

FEATURES = [
    "depletion_pressure_30_38a",
    "depletion_pressure_60_38a",
    "depletion_imbalance_60_38a"
]

REFS = [
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "update_rate_300s",
    "spread_depth_stress_900",
    "quote_retreat_imbalance_30_24a"
]

KEYS = ["symbol", "ts_sec"]

cols = KEYS + [
    "is_observed",
    "bid_p1", "ask_p1",
    "bid_q1", "ask_q1"
]

missing = [c for c in cols if c not in df_1s]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# ------------------------------------------------------------
# 1. Historical research sample
# ------------------------------------------------------------

x = df_1s[cols].copy()

x["symbol"] = x["symbol"].astype(str)

x["ts_sec"] = pd.to_numeric(
    x["ts_sec"], errors="raise"
).astype(np.int64)

ist = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ist.dt.date

clock = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
    - 32400
)

x = x.loc[
    (x["date"].astype(str) <= "2026-10-19")
    & (x["date"].astype(str) != "2026-09-28")
    & clock.between(0, 28799)
].copy()

coverage = x.groupby(
    ["symbol", "date"]
)["ts_sec"].transform("nunique")

x = (
    x.loc[coverage >= 0.97 * 28800]
    .sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True)
)

assert not x.duplicated(KEYS).any()
assert x.groupby(["symbol", "date"]).ngroups == 16

for c in ["bid_p1", "ask_p1", "bid_q1", "ask_q1"]:
    x[c] = pd.to_numeric(
        x[c], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)

N = len(x)
ts = x["ts_sec"].to_numpy(dtype=np.int64)

observed = x["is_observed"].eq(1).to_numpy()

bid_p = x["bid_p1"].to_numpy(dtype=float)
ask_p = x["ask_p1"].to_numpy(dtype=float)
bid_q = x["bid_q1"].to_numpy(dtype=float)
ask_q = x["ask_q1"].to_numpy(dtype=float)

valid = (
    np.isfinite(bid_p)
    & np.isfinite(ask_p)
    & np.isfinite(bid_q)
    & np.isfinite(ask_q)
    & (bid_p > 0)
    & (ask_p >= bid_p)
    & (bid_q >= 0)
    & (ask_q >= 0)
)

groups = x.groupby(
    ["symbol", "date"], sort=False
).indices

print("===== 38A RESEARCH SAMPLE =====")
print("Rows:", N)
print("Sessions:", len(groups))
print("Last date:", x["date"].max())

# ------------------------------------------------------------
# 2. Detect quantity reductions on unchanged best prices
#
# Compare current genuinely observed snapshot against
# the previous genuinely observed valid snapshot,
# within the same trading session.
#
# Skip intervals with large observation gaps (>10 sec)
# rather than assigning an old change to current time.
# ------------------------------------------------------------

bid_reduction = np.zeros(N, dtype=np.float64)
ask_reduction = np.zeros(N, dtype=np.float64)

valid_event = np.zeros(N, dtype=bool)
MAX_GAP = 10

for _, indices in groups.items():

    idx = np.sort(indices)

    genuine = idx[observed[idx] & valid[idx]]

    if len(genuine) < 2:
        continue

    prev_idx = genuine[:-1]
    curr_idx = genuine[1:]

    gap = ts[curr_idx] - ts[prev_idx]

    eligible = (gap > 0) & (gap <= MAX_GAP)

    # Exact unchanged best quote on each side.
    same_bid = (
        bid_p[curr_idx] == bid_p[prev_idx]
    )
    same_ask = (
        ask_p[curr_idx] == ask_p[prev_idx]
    )

    bd = np.maximum(
        bid_q[prev_idx] - bid_q[curr_idx],
        0.0
    )

    ad = np.maximum(
        ask_q[prev_idx] - ask_q[curr_idx],
        0.0
    )

    bid_reduction[curr_idx] = np.where(
        eligible & same_bid, bd, 0.0
    )

    ask_reduction[curr_idx] = np.where(
        eligible & same_ask, ad, 0.0
    )

    valid_event[curr_idx] = eligible

print("\n===== EVENT DIAGNOSTICS =====")
print("Valid observed comparisons:",
      int(valid_event.sum()))
print("Bid depletion events:",
      int((bid_reduction > 0).sum()))
print("Ask depletion events:",
      int((ask_reduction > 0).sum()))
print("Bid depleted quantity:",
      round(bid_reduction.sum(), 2))
print("Ask depleted quantity:",
      round(ask_reduction.sum(), 2))

# ------------------------------------------------------------
# 3. Exact clock-time rolling reduction sums
# ------------------------------------------------------------

def trailing_sum(ts_local, values, window):

    cs = np.concatenate([
        [0.0],
        np.cumsum(values, dtype=np.float64)
    ])

    left = np.searchsorted(
        ts_local,
        ts_local - window + 1,
        side="left"
    )

    return cs[np.arange(len(ts_local)) + 1] - cs[left]

bid30 = np.full(N, np.nan)
ask30 = np.full(N, np.nan)
bid60 = np.full(N, np.nan)
ask60 = np.full(N, np.nan)

for _, indices in groups.items():

    idx = np.sort(indices)
    t = ts[idx]

    b30 = trailing_sum(t, bid_reduction[idx], 30)
    a30 = trailing_sum(t, ask_reduction[idx], 30)
    b60 = trailing_sum(t, bid_reduction[idx], 60)
    a60 = trailing_sum(t, ask_reduction[idx], 60)

    warm30 = t >= t[0] + 29
    warm60 = t >= t[0] + 59

    bid30[idx[warm30]] = b30[warm30]
    ask30[idx[warm30]] = a30[warm30]
    bid60[idx[warm60]] = b60[warm60]
    ask60[idx[warm60]] = a60[warm60]

# ------------------------------------------------------------
# 4. Features
#
# Use a +1 denominator offset because quantities
# may be zero and a near-empty queue can create
# excessive raw ratios.
#
# Winsorization is intentionally NOT fitted here.
# For a deployable model, any fitted clipping
# thresholds must come from training data only.
# ------------------------------------------------------------

with np.errstate(divide="ignore", invalid="ignore"):

    pressure30 = (
        np.log1p(ask30 / (ask_q + 1.0))
        - np.log1p(bid30 / (bid_q + 1.0))
    )

    pressure60 = (
        np.log1p(ask60 / (ask_q + 1.0))
        - np.log1p(bid60 / (bid_q + 1.0))
    )

    imbalance60 = (
        (ask60 - bid60)
        / (ask60 + bid60 + 1e-8)
    )

for a in [pressure30, pressure60, imbalance60]:
    a[~valid] = np.nan
    a[~np.isfinite(a)] = np.nan

calendar_exp38a_features = x[
    ["symbol", "ts_sec", "date"]
].copy()

for name, arr in zip(
    FEATURES,
    [pressure30, pressure60, imbalance60]
):
    calendar_exp38a_features[name] = (
        arr.astype(np.float32)
    )

# ------------------------------------------------------------
# 5. Exact future-return targets, same session
# ------------------------------------------------------------

session_index = pd.MultiIndex.from_frame(
    x[["symbol", "date", "ts_sec"]]
)

mid = np.where(
    valid,
    0.5 * (bid_p + ask_p),
    np.nan
)

for h in HORIZONS:

    future_keys = pd.MultiIndex.from_arrays(
        [
            x["symbol"],
            x["date"],
            ts + h
        ],
        names=["symbol", "date", "ts_sec"]
    )

    pos = session_index.get_indexer(future_keys)

    future_mid = np.full(N, np.nan)

    found = pos >= 0
    future_mid[found] = mid[pos[found]]

    with np.errstate(
        divide="ignore", invalid="ignore"
    ):
        y = 1e4 * np.log(future_mid / mid)

    y[~np.isfinite(y)] = np.nan

    calendar_exp38a_features[
        f"future_return_{h}_38a"
    ] = y.astype(np.float32)

    calendar_exp38a_features[
        f"future_absret_{h}_38a"
    ] = np.abs(y).astype(np.float32)

# ------------------------------------------------------------
# 6. Daily IC
# ------------------------------------------------------------

def calc_ic(a, b):

    ok = np.isfinite(a) & np.isfinite(b)
    count = int(ok.sum())

    if count < 200:
        return np.nan, count

    aa = pd.Series(a[ok])
    bb = pd.Series(b[ok])

    if aa.nunique() < 2 or bb.nunique() < 2:
        return np.nan, count

    return float(
        aa.corr(bb, method="spearman")
    ), count

rows = []

for (symbol, day), frame in (
    calendar_exp38a_features.groupby(
        ["symbol", "date"], sort=False
    )
):
    for task in ["signed", "absret"]:

        for h in HORIZONS:

            target = (
                f"future_return_{h}_38a"
                if task == "signed"
                else f"future_absret_{h}_38a"
            )

            y = frame[target].to_numpy(dtype=float)

            for feature in FEATURES:

                a = frame[feature].to_numpy(dtype=float)

                ic, count = calc_ic(a, y)

                rows.append({
                    "symbol": symbol,
                    "date": day,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": count
                })

calendar_exp38a_daily = pd.DataFrame(rows)

calendar_exp38a_summary = (
    calendar_exp38a_daily.groupby(
        ["task", "horizon", "feature"]
    )
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda z: int((z > 0).sum())
        ),
        negative_days=(
            "ic", lambda z: int((z < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. Redundancy against existing 37-feature baseline
# ------------------------------------------------------------

redundancy_rows = []

if "usdinr_rebuilt_37_research" in globals():

    baseline = usdinr_rebuilt_37_research

    bidx = pd.MultiIndex.from_arrays(
        [
            baseline["symbol"].astype(str),
            pd.to_numeric(
                baseline["ts_sec"],
                errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS
    )

    current_idx = pd.MultiIndex.from_frame(
        calendar_exp38a_features[KEYS]
    )

    if bidx.has_duplicates:
        raise ValueError("Duplicate baseline keys")

    pos = bidx.get_indexer(current_idx)
    matched = pos >= 0

    print("\nBaseline key match:",
          f"{100*matched.mean():.4f}%")

    for reference in REFS:

        if reference not in baseline:
            continue

        original = pd.to_numeric(
            baseline[reference],
            errors="coerce"
        ).to_numpy(dtype=float)

        aligned = np.full(N, np.nan)
        aligned[matched] = original[pos[matched]]

        for feature in FEATURES:

            a = calendar_exp38a_features[
                feature
            ].to_numpy(dtype=float)

            ok = (
                np.isfinite(a)
                & np.isfinite(aligned)
            )

            corr = (
                pd.Series(a[ok]).corr(
                    pd.Series(aligned[ok]),
                    method="spearman"
                )
                if ok.sum() >= 200
                else np.nan
            )

            redundancy_rows.append({
                "feature": feature,
                "reference": reference,
                "spearman": corr
            })

calendar_exp38a_redundancy = pd.DataFrame(
    redundancy_rows
)

# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("\n===== 38A FEATURE COVERAGE =====")

for feature in FEATURES:

    s = calendar_exp38a_features[feature]

    print(
        feature,
        "coverage_pct=", round(100*s.notna().mean(), 4),
        "zero_pct=", round(100*s.eq(0).mean(), 4),
        "std=", round(s.std(), 6),
        "median=", round(s.median(), 6)
    )

print("\n===== 38A SIGNED RETURN IC =====")

print(
    calendar_exp38a_summary.loc[
        calendar_exp38a_summary.task.eq("signed")
    ].round(6).to_string(index=False)
)

print("\n===== 38A ABSOLUTE RETURN IC =====")

print(
    calendar_exp38a_summary.loc[
        calendar_exp38a_summary.task.eq("absret")
    ].round(6).to_string(index=False)
)

print("\n===== 38A REDUNDANCY =====")

if not calendar_exp38a_redundancy.empty:

    print(
        calendar_exp38a_redundancy.pivot(
            index="feature",
            columns="reference",
            values="spearman"
        ).round(5).to_string()
    )

else:
    print("No baseline frame found")

print("\n===== 38A CANDIDATE CORRELATIONS =====")

print(
    calendar_exp38a_features[FEATURES]
    .corr(method="spearman")
    .round(5).to_string()
)

print("\n===== EXPERIMENT 38A COMPLETE =====")
print("Research rows:", N)
print("Sessions:", len(groups))
print("October 21-23 untouched.")
print("Core remains 37 provisional features.")

gc.collect()


===== 38A RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Last date: 2026-10-19

===== EVENT DIAGNOSTICS =====
Valid observed comparisons: 266386
Bid depletion events: 9123
Ask depletion events: 19146
Bid depleted quantity: 1588585.0
Ask depleted quantity: 3625319.0

Baseline key match: 100.0000%

===== 38A FEATURE COVERAGE =====
depletion_pressure_30_38a coverage_pct= 99.8989 zero_pct= 41.1699 std= 1.18461 median= 0.0
depletion_pressure_60_38a coverage_pct= 99.7944 zero_pct= 23.702 std= 1.505858 median= 0.0
depletion_imbalance_60_38a coverage_pct= 99.7944 zero_pct= 24.6 std= 0.743679 median= 0.0

===== 38A SIGNED RETURN IC =====
  task  horizon                    feature  days   mean_ic  median_ic  positive_days  negative_days  median_valid_rows
signed       30 depletion_imbalance_60_38a    16 -0.010685  -0.013888              5             11            28664.0
signed       30  depletion_pressure_30_38a    16  0.006538   0.005647              9              7            28694.0
signe

29

In [31]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import gc

# ============================================================
# EXPERIMENT 38B — DEPLETION PRESSURE FULL-MODEL ABLATION
#
# A: Reconstructed 37-feature baseline + 9 controls
# B: Same baseline + depletion_pressure_60_38a
#
# Task: signed future midprice returns in basis points
# Horizons: 30, 60, 300 seconds
#
# Chronological expanding window:
#   First 5 research dates: initial training
#   Remaining 11 dates: walk-forward evaluation
#
# Training sampled every 10 seconds
# Evaluation at every eligible 1-second observation
#
# Results:
#   calendar_exp38b_summary
#   calendar_exp38b_daily
#   calendar_exp38b_daily_gains
#   calendar_exp38b_importance
#
# No October 21-23 access.
# ============================================================

FEATURE = "depletion_pressure_60_38a"
HORIZONS = [30, 60, 300]
KEYS = ["symbol", "ts_sec"]

CORE31 = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

ADDED6 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

BASE_FEATURES = CORE31 + ADDED6 + CONTROLS9
assert len(BASE_FEATURES) == 46
assert len(set(BASE_FEATURES)) == 46

PARAMS = dict(
    objective="regression",
    n_estimators=180,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=5,
    min_child_samples=200,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

# ------------------------------------------------------------
# 1. Load and align research frames
# ------------------------------------------------------------

base = usdinr_rebuilt_37_research
candidate = calendar_exp38a_features

required_targets = [
    f"future_return_{h}_38a"
    for h in HORIZONS
]

for label, frame, required in [
    ("baseline", base, BASE_FEATURES),
    ("candidate", candidate, [FEATURE] + required_targets)
]:
    missing = [
        c for c in KEYS + required
        if c not in frame.columns
    ]
    if missing:
        raise ValueError(f"{label} missing: {missing}")

research = candidate[
    KEYS + [FEATURE] + required_targets
].copy()

research["symbol"] = research["symbol"].astype(str)
research["ts_sec"] = pd.to_numeric(
    research["ts_sec"], errors="raise"
).astype(np.int64)

research["date"] = pd.to_datetime(
    research["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

research = (
    research.loc[
        (research["date"].astype(str) <= "2026-10-19")
        & (research["date"].astype(str) != "2026-09-28")
    ]
    .sort_values(["date", "symbol", "ts_sec"])
    .reset_index(drop=True)
)

assert len(research) == 459122, (
    f"Unexpected research rows: {len(research)}"
)
assert not research.duplicated(KEYS).any()

base_idx = pd.MultiIndex.from_arrays(
    [
        base["symbol"].astype(str),
        pd.to_numeric(
            base["ts_sec"], errors="raise"
        ).astype(np.int64)
    ],
    names=KEYS
)

research_idx = pd.MultiIndex.from_frame(
    research[KEYS]
)

assert not base_idx.has_duplicates

positions = base_idx.get_indexer(research_idx)

if not (positions >= 0).all():
    raise ValueError("Missing baseline keys")

# ------------------------------------------------------------
# 2. Feature matrices and common sample
# ------------------------------------------------------------

X_base = np.column_stack([
    pd.to_numeric(
        base[col], errors="coerce"
    ).to_numpy(dtype=np.float32)[positions]
    for col in BASE_FEATURES
])

X_candidate = pd.to_numeric(
    research[FEATURE], errors="coerce"
).to_numpy(dtype=np.float32)

X_base[~np.isfinite(X_base)] = np.nan
X_candidate[~np.isfinite(X_candidate)] = np.nan

complete_base = np.isfinite(X_base).all(axis=1)
complete_candidate = np.isfinite(X_candidate)

# Strictly identical rows in both models.
common = complete_base & complete_candidate

X = np.column_stack([
    X_base, X_candidate
]).astype(np.float32)

print("===== 38B MATCHED SAMPLE =====")
print("Research rows:", len(research))
print("Baseline-complete:", int(complete_base.sum()))
print("Common rows:", int(common.sum()))
print(
    "Coverage:",
    round(100 * common.mean(), 4), "%"
)
print(
    "Candidate-related exclusions:",
    int(complete_base.sum() - common.sum())
)

# ------------------------------------------------------------
# 3. Targets and chronological split
# ------------------------------------------------------------

Y = {
    h: pd.to_numeric(
        research[f"future_return_{h}_38a"],
        errors="coerce"
    ).to_numpy(dtype=np.float32)
    for h in HORIZONS
}

dates = research["date"].to_numpy()
ts = research["ts_sec"].to_numpy(dtype=np.int64)

unique_dates = sorted(pd.unique(dates))
assert len(unique_dates) == 16

BASE_COLS = list(range(46))
PLUS_COLS = list(range(47))

# ------------------------------------------------------------
# 4. Chronological walk-forward evaluation
# ------------------------------------------------------------

daily_rows = []
importance_rows = []

for day_i in range(5, len(unique_dates)):

    test_date = unique_dates[day_i]

    train_mask = (
        np.isin(dates, unique_dates[:day_i])
        & common
        & (ts % 10 == 0)
    )

    test_mask = (
        (dates == test_date)
        & common
    )

    for h in HORIZONS:

        y = Y[h]

        train_rows = np.flatnonzero(
            train_mask & np.isfinite(y)
        )

        test_rows = np.flatnonzero(
            test_mask & np.isfinite(y)
        )

        if len(train_rows) < 1000 or len(test_rows) < 200:
            raise ValueError(
                f"Insufficient data: {test_date}, h={h}"
            )

        y_train = y[train_rows]
        y_test = y[test_rows].astype(np.float64)

        for name, cols in [
            ("A_full37", BASE_COLS),
            ("B_depletion60", PLUS_COLS)
        ]:

            model = lgb.LGBMRegressor(**PARAMS)

            train_X = X[np.ix_(train_rows, cols)]
            test_X = X[np.ix_(test_rows, cols)]

            model.fit(train_X, y_train)
            pred = model.predict(test_X)

            err = y_test - pred

            daily_rows.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test_rows),
                "sse": float(np.sum(err**2)),
                "sae": float(np.sum(np.abs(err))),
                "mse": float(np.mean(err**2)),
                "mae": float(np.mean(np.abs(err)))
            })

            if name == "B_depletion60":

                gains = (
                    model.booster_
                    .feature_importance(
                        importance_type="gain"
                    )
                )

                total = float(gains.sum())

                importance_rows.append({
                    "date": test_date,
                    "horizon": h,
                    "feature": FEATURE,
                    "gain_share_pct": (
                        100 * float(gains[-1]) / total
                        if total > 0 else 0.0
                    ),
                    "split_count": int(
                        model.booster_
                        .feature_importance(
                            importance_type="split"
                        )[-1]
                    )
                })

            del model, train_X, test_X, pred, err

    print("Completed:", test_date)

calendar_exp38b_daily = pd.DataFrame(daily_rows)
calendar_exp38b_importance = pd.DataFrame(
    importance_rows
)

# ------------------------------------------------------------
# 5. Paired pooled metrics and daily gains
# ------------------------------------------------------------

summary_rows = []
daily_gain_rows = []

for h, part in calendar_exp38b_daily.groupby("horizon"):

    a = (
        part[part.model == "A_full37"]
        .set_index("date")
        .sort_index()
    )

    b = (
        part[part.model == "B_depletion60"]
        .set_index("date")
        .sort_index()
    )

    assert a.index.equals(b.index)
    assert (a["rows"].to_numpy() ==
            b["rows"].to_numpy()).all()

    pooled_mse_gain = 100 * (
        1 - b["sse"].sum() / a["sse"].sum()
    )

    pooled_mae_gain = 100 * (
        1 - b["sae"].sum() / a["sae"].sum()
    )

    mse_daily = 100 * (
        1 - b["mse"] / a["mse"]
    )

    mae_daily = 100 * (
        1 - b["mae"] / a["mae"]
    )

    summary_rows.append({
        "horizon": h,
        "days": len(a),
        "pooled_mse_gain_pct": pooled_mse_gain,
        "pooled_mae_gain_pct": pooled_mae_gain,
        "mse_win_days": int((mse_daily > 0).sum()),
        "mae_win_days": int((mae_daily > 0).sum()),
        "median_daily_mse_gain_pct":
            float(mse_daily.median()),
        "median_daily_mae_gain_pct":
            float(mae_daily.median()),
        "worst_daily_mse_gain_pct":
            float(mse_daily.min()),
        "worst_daily_mae_gain_pct":
            float(mae_daily.min())
    })

    for day in a.index:
        daily_gain_rows.append({
            "date": day,
            "horizon": h,
            "mse_gain_pct": float(mse_daily.loc[day]),
            "mae_gain_pct": float(mae_daily.loc[day])
        })

calendar_exp38b_summary = pd.DataFrame(
    summary_rows
)

calendar_exp38b_daily_gains = pd.DataFrame(
    daily_gain_rows
)

# ------------------------------------------------------------
# 6. Extra check: early vs late evaluation sessions
#
# Diagnostic only — not separate model selection.
# ------------------------------------------------------------

stability_rows = []

for h, part in calendar_exp38b_daily_gains.groupby(
    "horizon"
):

    part = part.sort_values("date")

    for label, sample in [
        ("early_eval_5", part.iloc[:5]),
        ("late_eval_6", part.iloc[5:])
    ]:
        stability_rows.append({
            "horizon": h,
            "period": label,
            "mean_daily_mse_gain_pct":
                sample["mse_gain_pct"].mean(),
            "mse_win_days":
                int((sample["mse_gain_pct"] > 0).sum()),
            "mean_daily_mae_gain_pct":
                sample["mae_gain_pct"].mean()
        })

calendar_exp38b_stability = pd.DataFrame(
    stability_rows
)

# ------------------------------------------------------------
# 7. Output
# ------------------------------------------------------------

print("\n===== 38B FULL MODEL ABLATION =====")
print(
    calendar_exp38b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 38B DAILY MSE GAINS =====")
print(
    calendar_exp38b_daily_gains.pivot(
        index="date",
        columns="horizon",
        values="mse_gain_pct"
    ).round(6).to_string()
)

print("\n===== 38B DAILY MAE GAINS =====")
print(
    calendar_exp38b_daily_gains.pivot(
        index="date",
        columns="horizon",
        values="mae_gain_pct"
    ).round(6).to_string()
)

print("\n===== 38B FEATURE IMPORTANCE =====")
print(
    calendar_exp38b_importance.groupby("horizon")
    .agg(
        mean_gain_share_pct=("gain_share_pct", "mean"),
        median_gain_share_pct=("gain_share_pct", "median"),
        mean_split_count=("split_count", "mean")
    )
    .round(6)
    .to_string()
)

print("\n===== 38B EARLY VS LATE STABILITY =====")
print(
    calendar_exp38b_stability
    .round(6)
    .to_string(index=False)
)

print("\n===== EXPERIMENT 38B COMPLETE =====")
print("Original baseline: 37 features + 9 controls")
print("Candidate:", FEATURE)
print("Features promoted: 0")
print("October 21-23 untouched.")

gc.collect()


===== 38B MATCHED SAMPLE =====
Research rows: 459122
Baseline-complete: 402979
Common rows: 402979
Coverage: 87.7717 %
Candidate-related exclusions: 0
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 38B FULL MODEL ABLATION =====
 horizon  days  pooled_mse_gain_pct  pooled_mae_gain_pct  mse_win_days  mae_win_days  median_daily_mse_gain_pct  median_daily_mae_gain_pct  worst_daily_mse_gain_pct  worst_daily_mae_gain_pct
      30    11            -0.034755             0.011753             5             5                  -0.034609                  -0.015829                 -0.218984                 -0.249863
      60    11             0.045102             0.038459             5             7                  -0.043436                   0.015489                 -0.281549                 -0.2118

8

In [32]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# EXPERIMENT 39A — WITHIN-QUOTE LIQUIDITY MIGRATION
#
# Research: 16 historical sessions through Oct 19
# No holdout access.
#
# Outputs:
#   calendar_exp39a_features
#   calendar_exp39a_daily
#   calendar_exp39a_summary
#   calendar_exp39a_redundancy
# ============================================================

HORIZONS = [30, 60, 300]
KEYS = ["symbol", "ts_sec"]

FEATURES = [
    "near_migration_asym_60_39a",
    "near_migration_asym_30_39a",
    "migration_activity_asym_60_39a"
]

REFS = [
    "relative_migration_60_31a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "depth_imbalance_l5",
    "depth_imbalance_delta",
    "queue_imbalance_l1",
    "queue_imbalance_delta",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "queue_replenishment_pressure_60s",
    "ofi_l1_sum_60s_signed_log",
    "spread_depth_stress_900"
]

QCOLS = (
    [f"bid_q{i}" for i in range(1, 6)]
    + [f"ask_q{i}" for i in range(1, 6)]
)

NEEDED = KEYS + [
    "is_observed", "bid_p1", "ask_p1"
] + QCOLS

missing = [c for c in NEEDED if c not in df_1s.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

# ------------------------------------------------------------
# 1. Eligible historical research sessions
# ------------------------------------------------------------

x = df_1s[NEEDED].copy()

x["symbol"] = x["symbol"].astype(str)
x["ts_sec"] = pd.to_numeric(
    x["ts_sec"], errors="raise"
).astype(np.int64)

ist = pd.to_datetime(
    x["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata")

x["date"] = ist.dt.date

seconds_after_midnight = (
    ist.dt.hour * 3600
    + ist.dt.minute * 60
    + ist.dt.second
)

x = x.loc[
    (x["date"].astype(str) <= "2026-10-19")
    & (x["date"].astype(str) != "2026-09-28")
    & seconds_after_midnight.between(
        9 * 3600, 17 * 3600 - 1
    )
].copy()

coverage = (
    x.groupby(["symbol", "date"])["ts_sec"]
    .transform("nunique")
)

x = (
    x.loc[coverage >= .97 * 28800]
    .sort_values(["symbol", "date", "ts_sec"])
    .reset_index(drop=True)
)

assert not x.duplicated(KEYS).any()

N = len(x)
groups = x.groupby(
    ["symbol", "date"], sort=False
).indices

assert len(groups) == 16, (
    f"Expected 16 research sessions, got {len(groups)}"
)

assert N == 459122, (
    f"Research rows changed: {N}"
)

for c in NEEDED[2:]:
    if c != "is_observed":
        x[c] = pd.to_numeric(
            x[c], errors="coerce"
        ).replace([np.inf, -np.inf], np.nan)

ts = x["ts_sec"].to_numpy(dtype=np.int64)
observed = x["is_observed"].eq(1).to_numpy()

bid_p = x["bid_p1"].to_numpy(dtype=float)
ask_p = x["ask_p1"].to_numpy(dtype=float)

bq = x[
    [f"bid_q{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

aq = x[
    [f"ask_q{i}" for i in range(1, 6)]
].to_numpy(dtype=np.float64)

valid_bid = (
    np.isfinite(bq).all(axis=1)
    & (bq >= 0).all(axis=1)
    & (bq.sum(axis=1) > 0)
    & np.isfinite(bid_p)
    & (bid_p > 0)
)

valid_ask = (
    np.isfinite(aq).all(axis=1)
    & (aq >= 0).all(axis=1)
    & (aq.sum(axis=1) > 0)
    & np.isfinite(ask_p)
)

valid_book = (
    valid_bid & valid_ask
    & (ask_p >= bid_p)
)

with np.errstate(divide="ignore", invalid="ignore"):
    bid_near = (
        bq[:, :2].sum(axis=1) / bq.sum(axis=1)
    )
    ask_near = (
        aq[:, :2].sum(axis=1) / aq.sum(axis=1)
    )

bid_near[~valid_bid] = np.nan
ask_near[~valid_ask] = np.nan

# ------------------------------------------------------------
# 2. Compare successive genuine observed snapshots
#
# Only count relative near-share changes if:
#   - previous and current are genuine observations
#   - valid quantities and quotes
#   - observed time gap <= 10 seconds
#   - best price unchanged on that side
#
# This measures redistribution at constant touch.
# It does NOT prove that orders moved between levels.
# ------------------------------------------------------------

db = np.zeros(N, dtype=np.float64)
da = np.zeros(N, dtype=np.float64)

bid_eligible = np.zeros(N, dtype=bool)
ask_eligible = np.zeros(N, dtype=bool)

MAX_GAP = 10

for _, indices in groups.items():

    idx = np.sort(indices)

    obs_idx = idx[
        observed[idx] & valid_book[idx]
    ]

    if len(obs_idx) < 2:
        continue

    prev = obs_idx[:-1]
    curr = obs_idx[1:]

    elapsed = ts[curr] - ts[prev]

    recent = (elapsed > 0) & (elapsed <= MAX_GAP)

    unchanged_bid = (
        bid_p[curr] == bid_p[prev]
    )

    unchanged_ask = (
        ask_p[curr] == ask_p[prev]
    )

    eb = recent & unchanged_bid
    ea = recent & unchanged_ask

    bid_eligible[curr] = eb
    ask_eligible[curr] = ea

    db[curr] = np.where(
        eb,
        bid_near[curr] - bid_near[prev],
        0.0
    )

    da[curr] = np.where(
        ea,
        ask_near[curr] - ask_near[prev],
        0.0
    )

# ------------------------------------------------------------
# 3. Exact clock-time trailing sums
# ------------------------------------------------------------

def trailing_sum(ts_local, values, window):

    cs = np.concatenate([
        [0.0],
        np.cumsum(values, dtype=np.float64)
    ])

    left = np.searchsorted(
        ts_local,
        ts_local - window + 1,
        side="left"
    )

    return (
        cs[np.arange(len(ts_local)) + 1]
        - cs[left]
    )

mig30 = np.full(N, np.nan)
mig60 = np.full(N, np.nan)
activity60 = np.full(N, np.nan)

for _, indices in groups.items():

    idx = np.sort(indices)
    t = ts[idx]

    signed_increment = (
        db[idx] - da[idx]
    )

    activity_increment = (
        np.abs(db[idx]) - np.abs(da[idx])
    )

    m30 = trailing_sum(
        t, signed_increment, 30
    )

    m60 = trailing_sum(
        t, signed_increment, 60
    )

    a60 = trailing_sum(
        t, activity_increment, 60
    )

    warm30 = t >= t[0] + 29
    warm60 = t >= t[0] + 59

    mig30[idx[warm30]] = m30[warm30]
    mig60[idx[warm60]] = m60[warm60]
    activity60[idx[warm60]] = a60[warm60]

for arr in [mig30, mig60, activity60]:
    arr[~valid_book] = np.nan
    arr[~np.isfinite(arr)] = np.nan

calendar_exp39a_features = x[
    ["symbol", "ts_sec", "date"]
].copy()

for col, values in zip(
    FEATURES, [mig60, mig30, activity60]
):
    calendar_exp39a_features[col] = (
        values.astype(np.float32)
    )

# ------------------------------------------------------------
# 4. Exact future-return targets
# ------------------------------------------------------------

session_index = pd.MultiIndex.from_frame(
    x[["symbol", "date", "ts_sec"]]
)

mid = np.where(
    valid_book,
    (bid_p + ask_p) / 2.0,
    np.nan
)

for h in HORIZONS:

    future_index = pd.MultiIndex.from_arrays(
        [
            x["symbol"],
            x["date"],
            ts + h
        ],
        names=["symbol", "date", "ts_sec"]
    )

    pos = session_index.get_indexer(
        future_index
    )

    future_mid = np.full(N, np.nan)
    found = pos >= 0

    future_mid[found] = mid[pos[found]]

    with np.errstate(divide="ignore", invalid="ignore"):
        y = 1e4 * np.log(future_mid / mid)

    y[~np.isfinite(y)] = np.nan

    calendar_exp39a_features[
        f"future_return_{h}_39a"
    ] = y.astype(np.float32)

    calendar_exp39a_features[
        f"future_absret_{h}_39a"
    ] = np.abs(y).astype(np.float32)

# ------------------------------------------------------------
# 5. Daily Spearman IC
# ------------------------------------------------------------

def daily_ic(a, b):

    ok = np.isfinite(a) & np.isfinite(b)
    n = int(ok.sum())

    if n < 200:
        return np.nan, n

    aa = pd.Series(a[ok])
    bb = pd.Series(b[ok])

    if aa.nunique() < 2 or bb.nunique() < 2:
        return np.nan, n

    return float(
        aa.corr(bb, method="spearman")
    ), n

rows = []

for (symbol, day), frame in (
    calendar_exp39a_features.groupby(
        ["symbol", "date"], sort=False
    )
):

    for task in ["signed", "absret"]:

        for h in HORIZONS:

            target = (
                f"future_return_{h}_39a"
                if task == "signed"
                else f"future_absret_{h}_39a"
            )

            y = frame[target].to_numpy(dtype=float)

            for feature in FEATURES:

                a = frame[feature].to_numpy(dtype=float)
                ic, n = daily_ic(a, y)

                rows.append({
                    "symbol": symbol,
                    "date": day,
                    "task": task,
                    "horizon": h,
                    "feature": feature,
                    "ic": ic,
                    "valid_rows": n
                })

calendar_exp39a_daily = pd.DataFrame(rows)

calendar_exp39a_summary = (
    calendar_exp39a_daily
    .groupby(["task", "horizon", "feature"])
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic", lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic", lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 6. Compare against existing baseline
# ------------------------------------------------------------

redundancy_rows = []

if "usdinr_rebuilt_37_research" in globals():

    baseline = usdinr_rebuilt_37_research

    baseline_index = pd.MultiIndex.from_arrays(
        [
            baseline["symbol"].astype(str),
            pd.to_numeric(
                baseline["ts_sec"], errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS
    )

    target_index = pd.MultiIndex.from_frame(
        calendar_exp39a_features[KEYS]
    )

    assert not baseline_index.has_duplicates

    pos = baseline_index.get_indexer(
        target_index
    )

    found = pos >= 0

    print(
        "\nBaseline key match:",
        f"{100 * found.mean():.4f}%"
    )

    for reference in REFS:

        if reference not in baseline.columns:
            continue

        original = pd.to_numeric(
            baseline[reference],
            errors="coerce"
        ).to_numpy(dtype=float)

        aligned = np.full(N, np.nan)

        aligned[found] = original[pos[found]]

        for feature in FEATURES:

            current = calendar_exp39a_features[
                feature
            ].to_numpy(dtype=float)

            ok = (
                np.isfinite(current)
                & np.isfinite(aligned)
            )

            corr = (
                pd.Series(current[ok]).corr(
                    pd.Series(aligned[ok]),
                    method="spearman"
                )
                if ok.sum() >= 200
                else np.nan
            )

            redundancy_rows.append({
                "feature": feature,
                "reference": reference,
                "spearman": corr
            })

calendar_exp39a_redundancy = pd.DataFrame(
    redundancy_rows
)

# ------------------------------------------------------------
# 7. Diagnostics
# ------------------------------------------------------------

print("\n===== 39A EVENT DIAGNOSTICS =====")
print("Bid migration-eligible updates:",
      int(bid_eligible.sum()))
print("Ask migration-eligible updates:",
      int(ask_eligible.sum()))
print("Nonzero bid redistribution:",
      int((db != 0).sum()))
print("Nonzero ask redistribution:",
      int((da != 0).sum()))

print("\n===== 39A FEATURE COVERAGE =====")

for feature in FEATURES:

    s = calendar_exp39a_features[feature]

    print(
        feature,
        "valid_pct=", round(100*s.notna().mean(), 4),
        "zero_pct=", round(100*s.eq(0).mean(), 4),
        "std=", round(s.std(), 6),
        "median=", round(s.median(), 6)
    )

print("\n===== 39A SIGNED RETURN IC =====")

print(
    calendar_exp39a_summary.loc[
        calendar_exp39a_summary.task.eq("signed")
    ].round(6).to_string(index=False)
)

print("\n===== 39A ABSOLUTE RETURN IC =====")

print(
    calendar_exp39a_summary.loc[
        calendar_exp39a_summary.task.eq("absret")
    ].round(6).to_string(index=False)
)

print("\n===== 39A REDUNDANCY =====")

if not calendar_exp39a_redundancy.empty:

    print(
        calendar_exp39a_redundancy.pivot(
            index="feature",
            columns="reference",
            values="spearman"
        ).round(5).to_string()
    )

else:
    print("No reconstructed baseline available")

print("\n===== 39A CANDIDATE CORRELATIONS =====")

print(
    calendar_exp39a_features[FEATURES]
    .corr(method="spearman")
    .round(5).to_string()
)

print("\n===== EXPERIMENT 39A COMPLETE =====")
print("Research rows:", N)
print("Sessions:", len(groups))
print("October 21-23 excluded.")
print("Core remains 37 provisional features.")

gc.collect()



Baseline key match: 100.0000%

===== 39A EVENT DIAGNOSTICS =====
Bid migration-eligible updates: 248616
Ask migration-eligible updates: 242914
Nonzero bid redistribution: 58961
Nonzero ask redistribution: 120536

===== 39A FEATURE COVERAGE =====
near_migration_asym_60_39a valid_pct= 99.7944 zero_pct= 5.5075 std= 0.367661 median= 0.0
near_migration_asym_30_39a valid_pct= 99.8989 zero_pct= 12.2127 std= 0.273646 median= 0.0
migration_activity_asym_60_39a valid_pct= 99.7944 zero_pct= 4.7112 std= 2.163026 median= -0.171945

===== 39A SIGNED RETURN IC =====
  task  horizon                        feature  days  mean_ic  median_ic  positive_days  negative_days  median_valid_rows
signed       30 migration_activity_asym_60_39a    16 0.056057   0.063269             15              1            28664.0
signed       30     near_migration_asym_30_39a    16 0.024044   0.032069             14              2            28694.0
signed       30     near_migration_asym_60_39a    16 0.022923   0.031055   

27

In [33]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import gc

# ============================================================
# EXPERIMENT 39B — MIGRATION ACTIVITY ABLATION
#
# A: 37 engineered features + 9 controls
# B: A + migration_activity_asym_60_39a
#
# Signed returns: 30s, 60s, 300s
# 16 development sessions, 11 evaluation sessions
# October 21-23 excluded.
# ============================================================

FEATURE_39B = "migration_activity_asym_60_39a"
HORIZONS_39B = [30, 60, 300]
KEYS_39B = ["symbol", "ts_sec"]

ADDED6_39B = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9_39B = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

# Use original 31 saved column definitions.
CORE31_39B = [
    c for c in core_31.columns
    if c not in KEYS_39B
]

BASE_39B = (
    CORE31_39B + ADDED6_39B + CONTROLS9_39B
)

assert len(CORE31_39B) == 31
assert len(BASE_39B) == 46
assert len(set(BASE_39B)) == 46

PARAMS_39B = dict(
    objective="regression",
    n_estimators=180,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=5,
    min_child_samples=200,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

base = usdinr_rebuilt_37_research
source = calendar_exp39a_features

targets = [
    f"future_return_{h}_39a"
    for h in HORIZONS_39B
]

for label, frame, columns in [
    ("baseline", base, BASE_39B),
    ("candidate", source, [FEATURE_39B] + targets)
]:
    missing = [
        c for c in KEYS_39B + columns
        if c not in frame.columns
    ]
    if missing:
        raise ValueError(f"{label} missing: {missing}")

# ------------------------------------------------------------
# 1. Research dates only
# ------------------------------------------------------------

research = source[
    KEYS_39B + [FEATURE_39B] + targets
].copy()

research["symbol"] = research["symbol"].astype(str)
research["ts_sec"] = pd.to_numeric(
    research["ts_sec"], errors="raise"
).astype(np.int64)

research["date"] = pd.to_datetime(
    research["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

research = (
    research.loc[
        (research["date"].astype(str) <= "2026-10-19")
        & (research["date"].astype(str) != "2026-09-28")
    ]
    .sort_values(["date", "symbol", "ts_sec"])
    .reset_index(drop=True)
)

assert len(research) == 459122
assert not research.duplicated(KEYS_39B).any()

dates = research["date"].to_numpy()
ts = research["ts_sec"].to_numpy(dtype=np.int64)

unique_dates = sorted(pd.unique(dates))
assert len(unique_dates) == 16

# ------------------------------------------------------------
# 2. Exact alignment with reconstructed baseline
# ------------------------------------------------------------

base_index = pd.MultiIndex.from_arrays(
    [
        base["symbol"].astype(str),
        pd.to_numeric(
            base["ts_sec"], errors="raise"
        ).astype(np.int64)
    ],
    names=KEYS_39B
)

research_index = pd.MultiIndex.from_frame(
    research[KEYS_39B]
)

assert not base_index.has_duplicates

positions = base_index.get_indexer(research_index)
assert (positions >= 0).all()

X_base = np.column_stack([
    pd.to_numeric(
        base[c], errors="coerce"
    ).to_numpy(dtype=np.float32)[positions]
    for c in BASE_39B
])

X_extra = pd.to_numeric(
    research[FEATURE_39B],
    errors="coerce"
).to_numpy(dtype=np.float32)

X_base[~np.isfinite(X_base)] = np.nan
X_extra[~np.isfinite(X_extra)] = np.nan

complete_base = np.isfinite(X_base).all(axis=1)
common = complete_base & np.isfinite(X_extra)

X = np.column_stack(
    [X_base, X_extra]
).astype(np.float32)

Y = {
    h: pd.to_numeric(
        research[f"future_return_{h}_39a"],
        errors="coerce"
    ).to_numpy(dtype=np.float32)
    for h in HORIZONS_39B
}

print("===== 39B MATCHED SAMPLE =====")
print("Research rows:", len(research))
print("Baseline-complete:", int(complete_base.sum()))
print("Candidate-complete:", int(common.sum()))
print(
    "Candidate-related exclusions:",
    int(complete_base.sum() - common.sum())
)
print(
    "Common coverage:",
    f"{100 * common.mean():.4f}%"
)

# ------------------------------------------------------------
# 3. Chronological expanding-window LightGBM
# ------------------------------------------------------------

daily_records = []
importance_records = []

models = {
    "A_full37": list(range(46)),
    "B_migration_activity": list(range(47))
}

for day_i in range(5, len(unique_dates)):

    test_date = unique_dates[day_i]

    train_mask = (
        np.isin(dates, unique_dates[:day_i])
        & common
        & (ts % 10 == 0)
    )

    test_mask = (
        (dates == test_date)
        & common
    )

    for h in HORIZONS_39B:

        y = Y[h]

        train_rows = np.flatnonzero(
            train_mask & np.isfinite(y)
        )
        test_rows = np.flatnonzero(
            test_mask & np.isfinite(y)
        )

        if len(train_rows) < 1000 or len(test_rows) < 200:
            raise ValueError(
                f"Insufficient rows: {test_date}, h={h}"
            )

        y_train = y[train_rows]
        y_test = y[test_rows].astype(np.float64)

        for name, cols in models.items():

            model = lgb.LGBMRegressor(**PARAMS_39B)

            train_X = X[np.ix_(train_rows, cols)]
            test_X = X[np.ix_(test_rows, cols)]

            model.fit(train_X, y_train)

            pred = model.predict(test_X)
            error = y_test - pred

            daily_records.append({
                "date": test_date,
                "horizon": h,
                "model": name,
                "rows": len(test_rows),
                "sse": float(np.sum(error**2)),
                "sae": float(np.sum(np.abs(error))),
                "mse": float(np.mean(error**2)),
                "mae": float(np.mean(np.abs(error)))
            })

            if name == "B_migration_activity":

                gain = model.booster_.feature_importance(
                    importance_type="gain"
                )

                splits = model.booster_.feature_importance(
                    importance_type="split"
                )

                importance_records.append({
                    "date": test_date,
                    "horizon": h,
                    "gain_share_pct": (
                        100 * float(gain[-1]) / gain.sum()
                        if gain.sum() > 0 else 0.0
                    ),
                    "split_count": int(splits[-1])
                })

            del model, train_X, test_X, pred, error

    print("Completed:", test_date)

calendar_exp39b_daily = pd.DataFrame(daily_records)
calendar_exp39b_importance = pd.DataFrame(
    importance_records
)

# ------------------------------------------------------------
# 4. Paired pooled improvements and daily gains
# ------------------------------------------------------------

summary_records = []
daily_gain_records = []

for h, frame in calendar_exp39b_daily.groupby("horizon"):

    a = (
        frame.loc[frame.model == "A_full37"]
        .set_index("date").sort_index()
    )

    b = (
        frame.loc[frame.model == "B_migration_activity"]
        .set_index("date").sort_index()
    )

    assert a.index.equals(b.index)
    assert np.array_equal(
        a["rows"].to_numpy(),
        b["rows"].to_numpy()
    )

    mse_daily = 100 * (
        1 - b["mse"] / a["mse"]
    )

    mae_daily = 100 * (
        1 - b["mae"] / a["mae"]
    )

    summary_records.append({
        "horizon": h,
        "days": len(a),
        "pooled_mse_gain_pct":
            100 * (1 - b["sse"].sum() / a["sse"].sum()),
        "pooled_mae_gain_pct":
            100 * (1 - b["sae"].sum() / a["sae"].sum()),
        "mse_win_days": int((mse_daily > 0).sum()),
        "mae_win_days": int((mae_daily > 0).sum()),
        "median_daily_mse_gain_pct":
            float(mse_daily.median()),
        "worst_daily_mse_gain_pct":
            float(mse_daily.min()),
        "median_daily_mae_gain_pct":
            float(mae_daily.median()),
        "worst_daily_mae_gain_pct":
            float(mae_daily.min())
    })

    for date in a.index:
        daily_gain_records.append({
            "date": date,
            "horizon": h,
            "mse_gain_pct": float(mse_daily.loc[date]),
            "mae_gain_pct": float(mae_daily.loc[date])
        })

calendar_exp39b_summary = pd.DataFrame(
    summary_records
)

calendar_exp39b_daily_gains = pd.DataFrame(
    daily_gain_records
)

# ------------------------------------------------------------
# 5. Early versus late evaluation stability
# ------------------------------------------------------------

stability = []

for h, part in calendar_exp39b_daily_gains.groupby(
    "horizon"
):

    part = part.sort_values("date")

    for label, subset in [
        ("early_5", part.iloc[:5]),
        ("late_6", part.iloc[5:])
    ]:
        stability.append({
            "horizon": h,
            "period": label,
            "mean_mse_gain_pct":
                subset["mse_gain_pct"].mean(),
            "mse_win_days":
                int((subset["mse_gain_pct"] > 0).sum()),
            "mean_mae_gain_pct":
                subset["mae_gain_pct"].mean()
        })

calendar_exp39b_stability = pd.DataFrame(stability)

# ------------------------------------------------------------
# 6. Activity-normalization diagnostic
#
# Measures whether migration activity is related to
# unequal numbers of eligible observed updates.
#
# Requires genuine-event arrays still in memory
# from Experiment 39A.
# No future information or extra model selection.
# ------------------------------------------------------------

if all(
    name in globals()
    for name in [
        "bid_eligible", "ask_eligible",
        "db", "da", "x"
    ]
) and len(bid_eligible) == len(research):

    # 39A 'x' was ordered by symbol/date/timestamp,
    # while this research frame is date/symbol/time.
    # Therefore, do not assume positional alignment.
    # Align all event arrays explicitly by keys.

    event_frame = x[
        ["symbol", "ts_sec"]
    ].copy()

    event_frame["bid_elig"] = bid_eligible.astype(float)
    event_frame["ask_elig"] = ask_eligible.astype(float)
    event_frame["bid_activity"] = np.abs(db)
    event_frame["ask_activity"] = np.abs(da)

    source_idx = pd.MultiIndex.from_arrays(
        [
            event_frame["symbol"].astype(str),
            pd.to_numeric(
                event_frame["ts_sec"],
                errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS_39B
    )

    loc = source_idx.get_indexer(research_index)

    if (loc >= 0).all():

        event = event_frame.iloc[loc]

        b_elig = event["bid_elig"].to_numpy()
        a_elig = event["ask_elig"].to_numpy()

        print("\n===== 39B UPDATE OPPORTUNITIES =====")
        print("Bid eligible:", int(b_elig.sum()))
        print("Ask eligible:", int(a_elig.sum()))
        print(
            "Bid nonzero share among eligible:",
            round(
                100 * (
                    (event["bid_activity"].to_numpy() > 0)
                    & (b_elig > 0)
                ).sum() / max(b_elig.sum(), 1),
                4
            ), "%"
        )
        print(
            "Ask nonzero share among eligible:",
            round(
                100 * (
                    (event["ask_activity"].to_numpy() > 0)
                    & (a_elig > 0)
                ).sum() / max(a_elig.sum(), 1),
                4
            ), "%"
        )

        # Only descriptive diagnostics are shown.
        # A fully normalized 60s activity feature
        # would require rolling numerator and
        # opportunity counts before model comparison.

        del event, event_frame

# ------------------------------------------------------------
# 7. Results
# ------------------------------------------------------------

print("\n===== 39B FULL MODEL ABLATION =====")

print(
    calendar_exp39b_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 39B DAILY MSE GAINS =====")

print(
    calendar_exp39b_daily_gains.pivot(
        index="date",
        columns="horizon",
        values="mse_gain_pct"
    ).round(6).to_string()
)

print("\n===== 39B DAILY MAE GAINS =====")

print(
    calendar_exp39b_daily_gains.pivot(
        index="date",
        columns="horizon",
        values="mae_gain_pct"
    ).round(6).to_string()
)

print("\n===== 39B FEATURE IMPORTANCE =====")

print(
    calendar_exp39b_importance.groupby("horizon")
    .agg(
        mean_gain_share_pct=("gain_share_pct", "mean"),
        median_gain_share_pct=("gain_share_pct", "median"),
        mean_split_count=("split_count", "mean")
    )
    .round(6)
    .to_string()
)

print("\n===== 39B EARLY VS LATE STABILITY =====")

print(
    calendar_exp39b_stability
    .round(6)
    .to_string(index=False)
)

print("\n===== EXPERIMENT 39B COMPLETE =====")
print("Core features: 37 provisional")
print("Challenger:", FEATURE_39B)
print("Automatic promotion: None")
print("Oct 21-23 holdout untouched.")

gc.collect()


===== 39B MATCHED SAMPLE =====
Research rows: 459122
Baseline-complete: 402979
Candidate-complete: 402979
Candidate-related exclusions: 0
Common coverage: 87.7717%
Completed: 2026-10-05
Completed: 2026-10-06
Completed: 2026-10-07
Completed: 2026-10-08
Completed: 2026-10-09
Completed: 2026-10-12
Completed: 2026-10-13
Completed: 2026-10-14
Completed: 2026-10-15
Completed: 2026-10-16
Completed: 2026-10-19

===== 39B UPDATE OPPORTUNITIES =====
Bid eligible: 248616
Ask eligible: 242914
Bid nonzero share among eligible: 23.7157 %
Ask nonzero share among eligible: 49.6209 %

===== 39B FULL MODEL ABLATION =====
 horizon  days  pooled_mse_gain_pct  pooled_mae_gain_pct  mse_win_days  mae_win_days  median_daily_mse_gain_pct  worst_daily_mse_gain_pct  median_daily_mae_gain_pct  worst_daily_mae_gain_pct
      30    11             0.001489            -0.063842             5             6                  -0.026233                 -0.224383                   0.032863                 -1.010480
      6

8

In [34]:

import numpy as np
import pandas as pd
import gc

# ============================================================
# EXPERIMENT 40A — NONLINEAR OFI / LIQUIDITY INTERACTIONS
#
# Features:
#  1. ofi_opposing_l1_40a
#  2. ofi_queue_confirmation_40a
#  3. ofi_opposing_l5_40a
#
# Outputs:
#  calendar_exp40a_features
#  calendar_exp40a_daily
#  calendar_exp40a_summary
#  calendar_exp40a_redundancy
#
# Historical development sessions ONLY.
# Oct 21-23 holdout never used.
# ============================================================

HORIZONS_40A = [30, 60, 300]
KEYS_40A = ["symbol", "ts_sec"]

FEATURES_40A = [
    "ofi_opposing_l1_40a",
    "ofi_queue_confirmation_40a",
    "ofi_opposing_l5_40a"
]

REFS_40A = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "ofi_disagreement_weighted_60",
    "spread_depth_stress_900",
    "log_total_depth_28a",
    "spread_bps_28a"
]

FLOW_COL = "ofi_l1_sum_60s_signed_log"
QUEUE_COL = "queue_imbalance_l1"

base = usdinr_rebuilt_37_research
old = calendar_exp39a_features

raw_cols = KEYS_40A + [
    f"{side}_q{i}"
    for side in ["bid", "ask"]
    for i in range(1, 6)
]

for label, frame, needed in [
    (
        "raw",
        df_1s,
        raw_cols
    ),
    (
        "baseline",
        base,
        KEYS_40A + REFS_40A
    ),
    (
        "targets",
        old,
        KEYS_40A + [
            f"future_return_{h}_39a"
            for h in HORIZONS_40A
        ]
    )
]:
    missing = [
        c for c in needed
        if c not in frame.columns
    ]
    if missing:
        raise ValueError(
            f"{label} missing {missing}"
        )

# ------------------------------------------------------------
# 1. Use the already-established research sample
# ------------------------------------------------------------

research = old[
    KEYS_40A
    + [
        f"future_return_{h}_39a"
        for h in HORIZONS_40A
    ]
].copy()

research["symbol"] = research[
    "symbol"
].astype(str)

research["ts_sec"] = pd.to_numeric(
    research["ts_sec"],
    errors="raise"
).astype(np.int64)

dates = pd.to_datetime(
    research["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

research["date"] = dates

research = research.loc[
    (research["date"].astype(str) <= "2026-10-19")
    & (research["date"].astype(str) != "2026-09-28")
].sort_values(
    ["date", "symbol", "ts_sec"]
).reset_index(drop=True)

assert len(research) == 459122
assert not research.duplicated(KEYS_40A).any()

assert (
    research.groupby(
        ["symbol", "date"]
    ).ngroups == 16
)

# ------------------------------------------------------------
# 2. Align baseline and raw snapshots by exact keys
# ------------------------------------------------------------

def make_index_40a(frame):
    return pd.MultiIndex.from_arrays(
        [
            frame["symbol"].astype(str),
            pd.to_numeric(
                frame["ts_sec"],
                errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS_40A
    )

research_idx = make_index_40a(research)
base_idx = make_index_40a(base)
raw_idx = make_index_40a(df_1s)

if base_idx.has_duplicates or raw_idx.has_duplicates:
    raise ValueError(
        "Source data contain duplicate keys"
    )

base_pos = base_idx.get_indexer(
    research_idx
)

raw_pos = raw_idx.get_indexer(
    research_idx
)

if not (base_pos >= 0).all():
    raise ValueError("Baseline alignment failed")

if not (raw_pos >= 0).all():
    raise ValueError("Raw-book alignment failed")

N = len(research)

print("===== 40A DATA ALIGNMENT =====")
print("Research rows:", N)
print("Research dates:", research["date"].nunique())
print(
    "Baseline alignment:",
    round(100 * (base_pos >= 0).mean(), 4), "%"
)
print(
    "Raw alignment:",
    round(100 * (raw_pos >= 0).mean(), 4), "%"
)

# ------------------------------------------------------------
# 3. Recover OFI, queue imbalance and L5 quantity
# ------------------------------------------------------------

F = pd.to_numeric(
    base[FLOW_COL],
    errors="coerce"
).to_numpy(dtype=np.float64)[base_pos]

I = pd.to_numeric(
    base[QUEUE_COL],
    errors="coerce"
).to_numpy(dtype=np.float64)[base_pos]

BQ = np.column_stack([
    pd.to_numeric(
        df_1s[f"bid_q{i}"],
        errors="coerce"
    ).to_numpy(dtype=np.float64)[raw_pos]
    for i in range(1, 6)
])

AQ = np.column_stack([
    pd.to_numeric(
        df_1s[f"ask_q{i}"],
        errors="coerce"
    ).to_numpy(dtype=np.float64)[raw_pos]
    for i in range(1, 6)
])

valid_book = (
    np.isfinite(BQ).all(axis=1)
    & np.isfinite(AQ).all(axis=1)
    & (BQ >= 0).all(axis=1)
    & (AQ >= 0).all(axis=1)
)

valid_inputs = (
    np.isfinite(F)
    & np.isfinite(I)
    & valid_book
)

b1 = BQ[:, 0]
a1 = AQ[:, 0]

b5 = BQ.sum(axis=1)
a5 = AQ.sum(axis=1)

# Buying pressure faces ask depth.
# Selling pressure faces bid depth.
# Zero flow yields a zero signed pressure.

opp_l1 = np.where(
    F > 0, a1, b1
)

opp_l5 = np.where(
    F > 0, a5, b5
)

with np.errstate(
    divide="ignore",
    invalid="ignore",
    over="ignore"
):
    ofi_opp1 = F / np.sqrt(
        1.0 + opp_l1
    )

    ofi_opp5 = F / np.sqrt(
        1.0 + opp_l5
    )

    ofi_confirmation = F * I

for arr in [
    ofi_opp1,
    ofi_opp5,
    ofi_confirmation
]:
    arr[~valid_inputs] = np.nan
    arr[~np.isfinite(arr)] = np.nan

# ------------------------------------------------------------
# 4. Store compact feature frame
# ------------------------------------------------------------

calendar_exp40a_features = research[
    ["symbol", "ts_sec", "date"]
].copy()

arrays_40a = [
    ofi_opp1,
    ofi_confirmation,
    ofi_opp5
]

for feature, arr in zip(
    FEATURES_40A,
    arrays_40a
):
    calendar_exp40a_features[
        feature
    ] = arr.astype(np.float32)

for h in HORIZONS_40A:
    calendar_exp40a_features[
        f"future_return_{h}_40a"
    ] = pd.to_numeric(
        research[
            f"future_return_{h}_39a"
        ],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

# ------------------------------------------------------------
# 5. Daily IC calculations
# ------------------------------------------------------------

def ic_40a(x, y):

    good = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    count = int(good.sum())

    if count < 200:
        return np.nan, count

    a = pd.Series(x[good])
    b = pd.Series(y[good])

    if (
        a.nunique() < 2
        or b.nunique() < 2
    ):
        return np.nan, count

    return (
        float(
            a.corr(
                b,
                method="spearman"
            )
        ),
        count
    )

rows = []

for (symbol, day), frame in (
    calendar_exp40a_features.groupby(
        ["symbol", "date"],
        sort=False
    )
):

    for h in HORIZONS_40A:

        y = frame[
            f"future_return_{h}_40a"
        ].to_numpy(dtype=float)

        for feature in FEATURES_40A:

            values = frame[
                feature
            ].to_numpy(dtype=float)

            ic, n = ic_40a(values, y)

            rows.append({
                "symbol": symbol,
                "date": day,
                "horizon": h,
                "feature": feature,
                "ic": ic,
                "valid_rows": n
            })

calendar_exp40a_daily = pd.DataFrame(rows)

calendar_exp40a_summary = (
    calendar_exp40a_daily
    .groupby(
        ["horizon", "feature"]
    )
    .agg(
        days=("ic", "count"),
        mean_ic=("ic", "mean"),
        median_ic=("ic", "median"),
        positive_days=(
            "ic",
            lambda s: int((s > 0).sum())
        ),
        negative_days=(
            "ic",
            lambda s: int((s < 0).sum())
        ),
        median_valid_rows=(
            "valid_rows", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 6. Redundancy against baseline
# ------------------------------------------------------------

redundancy = []

for ref in REFS_40A:

    ref_values = pd.to_numeric(
        base[ref],
        errors="coerce"
    ).to_numpy(dtype=np.float64)[base_pos]

    for feature in FEATURES_40A:

        v = calendar_exp40a_features[
            feature
        ].to_numpy(dtype=np.float64)

        good = (
            np.isfinite(v)
            & np.isfinite(ref_values)
        )

        corr = (
            pd.Series(v[good]).corr(
                pd.Series(
                    ref_values[good]
                ),
                method="spearman"
            )
            if good.sum() >= 200
            else np.nan
        )

        redundancy.append({
            "feature": feature,
            "reference": ref,
            "spearman": corr
        })

calendar_exp40a_redundancy = pd.DataFrame(
    redundancy
)

# ------------------------------------------------------------
# 7. Print diagnostics
# ------------------------------------------------------------

print("\n===== 40A FEATURE DISTRIBUTIONS =====")

for feature in FEATURES_40A:

    s = calendar_exp40a_features[
        feature
    ]

    print(
        feature,
        "coverage_pct=",
        round(100 * s.notna().mean(), 4),
        "zero_pct=",
        round(100 * s.eq(0).mean(), 4),
        "std=",
        round(s.std(), 6),
        "median=",
        round(s.median(), 6),
        "p01=",
        round(s.quantile(.01), 6),
        "p99=",
        round(s.quantile(.99), 6)
    )

print("\n===== 40A SIGNED RETURN IC =====")

print(
    calendar_exp40a_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 40A REDUNDANCY =====")

print(
    calendar_exp40a_redundancy
    .pivot(
        index="feature",
        columns="reference",
        values="spearman"
    )
    .round(5)
    .to_string()
)

print("\n===== 40A CANDIDATE CORRELATIONS =====")

print(
    calendar_exp40a_features[
        FEATURES_40A
    ]
    .corr(method="spearman")
    .round(5)
    .to_string()
)

print("\n===== EXPERIMENT 40A COMPLETE =====")
print("Research rows:", N)
print("Research sessions: 16")
print("New features promoted: 0")
print("Core remains 37 provisional features")
print("Oct 21-23 holdout untouched")

del BQ, AQ
gc.collect()


===== 40A DATA ALIGNMENT =====
Research rows: 459122
Research dates: 16
Baseline alignment: 100.0 %
Raw alignment: 100.0 %

===== 40A FEATURE DISTRIBUTIONS =====
ofi_opposing_l1_40a coverage_pct= 99.9303 zero_pct= 7.9979 std= 1.136911 median= 0.0 p01= -3.371556 p99= 3.65927
ofi_queue_confirmation_40a coverage_pct= 99.9303 zero_pct= 8.7582 std= 3.916124 median= 0.927395 p01= -6.921582 p99= 7.84092
ofi_opposing_l5_40a coverage_pct= 99.9303 zero_pct= 7.9979 std= 0.190528 median= 0.0 p01= -0.395353 p99= 0.457726

===== 40A SIGNED RETURN IC =====
 horizon                    feature  days  mean_ic  median_ic  positive_days  negative_days  median_valid_rows
      30        ofi_opposing_l1_40a    16 0.067299   0.071834             14              2            28703.0
      30        ofi_opposing_l5_40a    16 0.055183   0.064144             13              3            28703.0
      30 ofi_queue_confirmation_40a    16 0.020631   0.026752             13              3            28703.0
      60

0

In [35]:

import numpy as np
import pandas as pd
import lightgbm as lgb
import gc

# ============================================================
# EXPERIMENT 41 — FINAL COMBINED-CHALLENGER ABLATION
#
# Signed models:
#   A_full37
#   B_migration
#   C_other_four
#   D_all_five
#
# Absolute-return models:
#   A_full37
#   E_burstiness
#
# Research only: 16 sessions through 2026-10-19.
# First 5 chronological sessions initialize training.
# Next 11 sessions evaluated with expanding training.
#
# All candidate NaNs retained, LightGBM handles them.
# Baseline-complete rows define the common sample.
#
# October 21–23 never accessed.
#
# Outputs:
#   calendar_exp41_summary
#   calendar_exp41_daily
#   calendar_exp41_daily_gains
#   calendar_exp41_importance
#   calendar_exp41_candidate_diagnostics
# ============================================================

KEYS_41 = ["symbol", "ts_sec"]
HORIZONS_41 = [30, 60, 300]

# ------------------------------------------------------------
# 1. Feature registry
# ------------------------------------------------------------

CORE31_41 = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

ADDED6_41 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9_41 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

BASE_41 = CORE31_41 + ADDED6_41 + CONTROLS9_41

assert len(BASE_41) == 46
assert len(set(BASE_41)) == 46

RECOVERY_41 = "recovery_pressure_34b"
ASK_AGE_41 = "ask_shock_age_34b"
SIDE_41 = "side_update_imbalance_60_36a"
ENTROPY_41 = "entropy_change_asym_60_37a"
MIGRATION_41 = "migration_activity_asym_60_39a"
BURSTINESS_41 = "event_burstiness_60_36a"

DIRECTIONAL_41 = [
    RECOVERY_41,
    ASK_AGE_41,
    SIDE_41,
    ENTROPY_41,
    MIGRATION_41
]

ALL_CANDIDATES_41 = (
    DIRECTIONAL_41 + [BURSTINESS_41]
)

PARAMS_41 = dict(
    objective="regression",
    n_estimators=180,
    learning_rate=0.035,
    num_leaves=15,
    max_depth=5,
    min_child_samples=200,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=1,
    verbosity=-1
)

# ------------------------------------------------------------
# 2. Source DataFrames
# ------------------------------------------------------------

required_frames = [
    "usdinr_rebuilt_37_research",
    "calendar_exp34b_features",
    "calendar_exp36a_features",
    "calendar_exp37a_features",
    "calendar_exp39a_features"
]

missing_frames = [
    name for name in required_frames
    if name not in globals()
]

if missing_frames:
    raise ValueError(
        "Missing research DataFrames: "
        + str(missing_frames)
        + ". Restore their earlier saved Parquets "
          "or rerun the corresponding experiment."
    )

base_source = usdinr_rebuilt_37_research

FEATURE_SOURCES_41 = [
    (calendar_exp34b_features,
     [RECOVERY_41, ASK_AGE_41]),

    (calendar_exp36a_features,
     [SIDE_41, BURSTINESS_41]),

    (calendar_exp37a_features,
     [ENTROPY_41]),

    (calendar_exp39a_features,
     [MIGRATION_41])
]

TARGET_SOURCE_41 = calendar_exp39a_features

# ------------------------------------------------------------
# 3. Research keys — never construct from holdout
# ------------------------------------------------------------

target_columns = [
    f"future_{kind}_{h}_39a"
    for kind in ["return", "absret"]
    for h in HORIZONS_41
]

required_target_cols = KEYS_41 + target_columns

missing = [
    c for c in required_target_cols
    if c not in TARGET_SOURCE_41
]

if missing:
    raise ValueError(
        f"Target source missing: {missing}"
    )

research = TARGET_SOURCE_41[
    required_target_cols
].copy()

research["symbol"] = research[
    "symbol"
].astype(str)

research["ts_sec"] = pd.to_numeric(
    research["ts_sec"], errors="raise"
).astype(np.int64)

research["date"] = pd.to_datetime(
    research["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

research = (
    research.loc[
        (research["date"].astype(str) <= "2026-10-19")
        & (research["date"].astype(str) != "2026-09-28")
    ]
    .sort_values(["date", "symbol", "ts_sec"])
    .reset_index(drop=True)
)

assert len(research) == 459122
assert not research.duplicated(KEYS_41).any()

unique_dates = sorted(
    research["date"].unique()
)

assert len(unique_dates) == 16

N = len(research)
dates = research["date"].to_numpy()
ts = research["ts_sec"].to_numpy(
    dtype=np.int64
)

# ------------------------------------------------------------
# 4. Exact key alignment — no positional assumptions
# ------------------------------------------------------------

def keyed_index_41(frame):

    return pd.MultiIndex.from_arrays(
        [
            frame["symbol"].astype(str),
            pd.to_numeric(
                frame["ts_sec"], errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS_41
    )

research_idx = keyed_index_41(research)

def align_columns_41(source, columns):

    missing = [
        c for c in KEYS_41 + columns
        if c not in source.columns
    ]

    if missing:
        raise ValueError(
            f"Source missing: {missing}"
        )

    source_idx = keyed_index_41(source)

    if source_idx.has_duplicates:
        raise ValueError(
            "Duplicate keys in source"
        )

    positions = source_idx.get_indexer(
        research_idx
    )

    # Allow missing source keys: represented as NaN.
    aligned = np.full(
        (N, len(columns)),
        np.nan,
        dtype=np.float32
    )

    valid = positions >= 0

    if valid.any():
        for j, col in enumerate(columns):

            values = pd.to_numeric(
                source[col], errors="coerce"
            ).to_numpy(dtype=np.float32)

            aligned[valid, j] = values[
                positions[valid]
            ]

    aligned[~np.isfinite(aligned)] = np.nan

    return aligned, valid

X_base, baseline_matched = align_columns_41(
    base_source, BASE_41
)

if not baseline_matched.all():
    raise ValueError(
        "Baseline research-key alignment incomplete"
    )

candidate_arrays = []
source_coverage = {}

for source, cols in FEATURE_SOURCES_41:

    aligned, matched = align_columns_41(
        source, cols
    )

    for j, name in enumerate(cols):
        source_coverage[name] = (
            int(matched.sum()),
            int(np.isfinite(
                aligned[:, j]
            ).sum())
        )

    candidate_arrays.append(aligned)

X_extra = np.column_stack(
    candidate_arrays
).astype(np.float32)

assert X_extra.shape[1] == 6

# Order from FEATURE_SOURCES_41:
# recovery, ask_age, side, burstiness,
# entropy, migration

EXTRA_ORDER_41 = [
    RECOVERY_41,
    ASK_AGE_41,
    SIDE_41,
    BURSTINESS_41,
    ENTROPY_41,
    MIGRATION_41
]

ALL_FEATURES_41 = BASE_41 + EXTRA_ORDER_41

X = np.column_stack(
    [X_base, X_extra]
).astype(np.float32)

name_to_index = {
    name: i
    for i, name in enumerate(ALL_FEATURES_41)
}

complete_baseline = np.isfinite(
    X_base
).all(axis=1)

# IMPORTANT:
# No candidate-completeness filtering.
# Missing age values are meaningful information.
common_mask = complete_baseline

print("===== 41 SAMPLE DIAGNOSTICS =====")
print("Research rows:", N)
print("Research sessions:", len(unique_dates))
print(
    "Baseline-complete rows:",
    int(complete_baseline.sum())
)
print(
    "Baseline coverage:",
    round(100 * complete_baseline.mean(), 4), "%"
)

# Candidate coverage
diag = []

for feature in ALL_CANDIDATES_41:

    idx = name_to_index[feature]

    vals = X[:, idx]

    diag.append({
        "feature": feature,
        "matched_source_rows":
            source_coverage[feature][0],
        "finite_rows":
            int(np.isfinite(vals).sum()),
        "finite_pct":
            100 * np.isfinite(vals).mean(),
        "missing_pct_within_baseline":
            100 * (
                ~np.isfinite(
                    vals[complete_baseline]
                )
            ).mean(),
        "std":
            float(np.nanstd(vals))
    })

calendar_exp41_candidate_diagnostics = (
    pd.DataFrame(diag)
)

print(
    "\n===== 41 CANDIDATE COVERAGE ====="
)
print(
    calendar_exp41_candidate_diagnostics
    .round(5)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 5. Models
# ------------------------------------------------------------

A = list(range(46))

other_four = [
    RECOVERY_41,
    ASK_AGE_41,
    SIDE_41,
    ENTROPY_41
]

MODELS_41 = {
    "signed": {
        "A_full37": A,
        "B_migration": A + [
            name_to_index[MIGRATION_41]
        ],
        "C_other_four": A + [
            name_to_index[c]
            for c in other_four
        ],
        "D_all_five": A + [
            name_to_index[c]
            for c in DIRECTIONAL_41
        ]
    },

    "absret": {
        "A_full37": A,
        "E_burstiness": A + [
            name_to_index[BURSTINESS_41]
        ]
    }
}

Y = {}

for task in ["signed", "absret"]:

    prefix = (
        "return"
        if task == "signed"
        else "absret"
    )

    for h in HORIZONS_41:

        Y[(task, h)] = pd.to_numeric(
            research[
                f"future_{prefix}_{h}_39a"
            ],
            errors="coerce"
        ).to_numpy(dtype=np.float32)

# ------------------------------------------------------------
# 6. Expanding chronological walk-forward
# ------------------------------------------------------------

daily_records = []
importance_records = []

for day_number in range(
    5, len(unique_dates)
):

    test_day = unique_dates[day_number]

    historical = np.isin(
        dates,
        unique_dates[:day_number]
    )

    train_base = (
        historical
        & common_mask
        & (ts % 10 == 0)
    )

    test_base = (
        (dates == test_day)
        & common_mask
    )

    for task, model_specs in MODELS_41.items():

        for h in HORIZONS_41:

            y = Y[(task, h)]

            train_rows = np.flatnonzero(
                train_base
                & np.isfinite(y)
            )

            test_rows = np.flatnonzero(
                test_base
                & np.isfinite(y)
            )

            if (
                len(train_rows) < 1000
                or len(test_rows) < 200
            ):
                raise ValueError(
                    f"Insufficient rows: "
                    f"{test_day}, {task}, h={h}"
                )

            y_train = y[train_rows]
            y_test = y[test_rows].astype(
                np.float64
            )

            for model_name, selected in (
                model_specs.items()
            ):

                train_X = X[np.ix_(
                    train_rows, selected
                )]

                test_X = X[np.ix_(
                    test_rows, selected
                )]

                model = lgb.LGBMRegressor(
                    **PARAMS_41
                )

                model.fit(
                    train_X,
                    y_train
                )

                prediction = model.predict(
                    test_X
                )

                error = y_test - prediction

                daily_records.append({
                    "date": test_day,
                    "task": task,
                    "horizon": h,
                    "model": model_name,
                    "rows": len(test_rows),
                    "sse": float(
                        np.sum(error**2)
                    ),
                    "sae": float(
                        np.sum(np.abs(error))
                    ),
                    "mse": float(
                        np.mean(error**2)
                    ),
                    "mae": float(
                        np.mean(np.abs(error))
                    )
                })

                if model_name != "A_full37":

                    gains = model.booster_.feature_importance(
                        importance_type="gain"
                    )

                    splits = model.booster_.feature_importance(
                        importance_type="split"
                    )

                    total_gain = float(
                        gains.sum()
                    )

                    for local_i, global_i in enumerate(
                        selected
                    ):

                        feature = ALL_FEATURES_41[
                            global_i
                        ]

                        if feature not in ALL_CANDIDATES_41:
                            continue

                        importance_records.append({
                            "date": test_day,
                            "task": task,
                            "horizon": h,
                            "model": model_name,
                            "feature": feature,
                            "gain_share_pct": (
                                100 * float(
                                    gains[local_i]
                                ) / total_gain
                                if total_gain > 0
                                else 0.0
                            ),
                            "split_count": int(
                                splits[local_i]
                            )
                        })

                del (
                    model,
                    train_X,
                    test_X,
                    prediction,
                    error
                )

    print("Completed:", test_day)

calendar_exp41_daily = pd.DataFrame(
    daily_records
)

calendar_exp41_importance = pd.DataFrame(
    importance_records
)

# ------------------------------------------------------------
# 7. Paired comparisons
# ------------------------------------------------------------

COMPARISONS_41 = {
    "signed": {
        "migration_vs_baseline": (
            "B_migration",
            "A_full37"
        ),
        "other_four_vs_baseline": (
            "C_other_four",
            "A_full37"
        ),
        "all_five_vs_baseline": (
            "D_all_five",
            "A_full37"
        ),
        "all_five_vs_migration": (
            "D_all_five",
            "B_migration"
        ),
        "migration_increment_to_four": (
            "D_all_five",
            "C_other_four"
        )
    },

    "absret": {
        "burstiness_vs_baseline": (
            "E_burstiness",
            "A_full37"
        )
    }
}

summary = []
daily_gains = []

for (task, h), frame in (
    calendar_exp41_daily.groupby(
        ["task", "horizon"]
    )
):

    pooled = frame.groupby(
        "model"
    )[["sse", "sae"]].sum()

    mse = frame.pivot(
        index="date",
        columns="model",
        values="mse"
    )

    mae = frame.pivot(
        index="date",
        columns="model",
        values="mae"
    )

    nrows = frame.pivot(
        index="date",
        columns="model",
        values="rows"
    )

    assert (
        nrows.nunique(axis=1) == 1
    ).all(), "Unmatched evaluation sample"

    for comparison, (
        candidate,
        reference
    ) in COMPARISONS_41[task].items():

        d_mse = 100 * (
            1
            - mse[candidate]
            / mse[reference]
        )

        d_mae = 100 * (
            1
            - mae[candidate]
            / mae[reference]
        )

        pooled_mse = 100 * (
            1
            - pooled.loc[
                candidate, "sse"
            ]
            / pooled.loc[
                reference, "sse"
            ]
        )

        pooled_mae = 100 * (
            1
            - pooled.loc[
                candidate, "sae"
            ]
            / pooled.loc[
                reference, "sae"
            ]
        )

        summary.append({
            "task": task,
            "horizon": h,
            "comparison": comparison,
            "days": len(d_mse),
            "pooled_mse_gain_pct":
                pooled_mse,
            "pooled_mae_gain_pct":
                pooled_mae,
            "mse_win_days":
                int((d_mse > 0).sum()),
            "mae_win_days":
                int((d_mae > 0).sum()),
            "median_daily_mse_gain_pct":
                float(d_mse.median()),
            "worst_daily_mse_gain_pct":
                float(d_mse.min()),
            "median_daily_mae_gain_pct":
                float(d_mae.median()),
            "worst_daily_mae_gain_pct":
                float(d_mae.min())
        })

        for day in d_mse.index:

            daily_gains.append({
                "date": day,
                "task": task,
                "horizon": h,
                "comparison": comparison,
                "mse_gain_pct":
                    float(d_mse.loc[day]),
                "mae_gain_pct":
                    float(d_mae.loc[day])
            })

calendar_exp41_summary = pd.DataFrame(
    summary
)

calendar_exp41_daily_gains = pd.DataFrame(
    daily_gains
)

# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("\n===== 41 COMBINED ABLATION =====")

print(
    calendar_exp41_summary
    .round(6)
    .to_string(index=False)
)

print("\n===== 41 DAILY MSE GAINS =====")

for comp in [
    "migration_vs_baseline",
    "all_five_vs_baseline",
    "migration_increment_to_four",
    "burstiness_vs_baseline"
]:

    part = calendar_exp41_daily_gains.loc[
        calendar_exp41_daily_gains[
            "comparison"
        ].eq(comp)
    ]

    print("\n", comp)

    print(
        part.pivot(
            index="date",
            columns="horizon",
            values="mse_gain_pct"
        )
        .round(5)
        .to_string()
    )

print("\n===== 41 DAILY MAE GAINS =====")

for comp in [
    "all_five_vs_baseline",
    "migration_increment_to_four",
    "burstiness_vs_baseline"
]:

    part = calendar_exp41_daily_gains.loc[
        calendar_exp41_daily_gains[
            "comparison"
        ].eq(comp)
    ]

    print("\n", comp)

    print(
        part.pivot(
            index="date",
            columns="horizon",
            values="mae_gain_pct"
        )
        .round(5)
        .to_string()
    )

print("\n===== 41 FEATURE IMPORTANCE =====")

print(
    calendar_exp41_importance
    .groupby(
        [
            "task",
            "horizon",
            "model",
            "feature"
        ]
    )[
        ["gain_share_pct", "split_count"]
    ]
    .mean()
    .round(5)
    .to_string()
)

print("\n===== 41 CANDIDATE COVERAGE =====")

print(
    calendar_exp41_candidate_diagnostics
    .round(5)
    .to_string(index=False)
)

print("\n===== EXPERIMENT 41 COMPLETE =====")
print("Baseline: 37 engineered + 9 controls")
print("Directional challengers:", len(DIRECTIONAL_41))
print("Volatility challengers: 1")
print("Evaluation days:", len(unique_dates) - 5)
print("Automatically promoted: 0")
print("October 21–23: untouched")

gc.collect()


===== 41 SAMPLE DIAGNOSTICS =====
Research rows: 459122
Research sessions: 16
Baseline-complete rows: 402979
Baseline coverage: 87.7717 %

===== 41 CANDIDATE COVERAGE =====
                       feature  matched_source_rows  finite_rows  finite_pct  missing_pct_within_baseline      std
         recovery_pressure_34b               459122       459122   100.00000                      0.00000  0.82808
             ask_shock_age_34b               459122       304438    66.30874                     31.99149 16.30823
  side_update_imbalance_60_36a               459122       458178    99.79439                      0.00000  0.57994
    entropy_change_asym_60_37a               459122       458162    99.79091                      0.00000  0.21998
migration_activity_asym_60_39a               459122       458178    99.79439                      0.00000  2.16302
       event_burstiness_60_36a               459122       458178    99.79439                      0.00000  0.92313
Completed: 2026-10-05


0

In [36]:

import numpy as np
import pandas as pd
import json
import hashlib
from pathlib import Path

# ============================================================
# USDINR FINAL FEATURE FREEZE — RESEARCH V2
#
# REQUIRED:
#   core_31
#   usdinr_rebuilt_37_research
#   calendar_exp34b_features
#   calendar_exp36a_features
#   calendar_exp39a_features
#
# OUTPUTS:
#   usdinr_baseline_v2_46inputs.parquet
#   usdinr_challenger_v2_50inputs.parquet
#   usdinr_feature_manifest_v2.json
#
# Only original 16 research sessions (<= Oct 19).
# Do not include October 21–23 holdout.
#
# This is a registry/data-freeze step, not a
# fresh end-to-end computation from raw L5 data.
# ============================================================

OUTDIR = Path("usdinr_final_v2")
OUTDIR.mkdir(parents=True, exist_ok=True)

KEYS = ["symbol", "ts_sec"]

CORE31 = [
    "ofi_l1_sum_60s_signed_log",
    "ofi_rate_diff_30_300_sl",
    "ofi_activity_60s",
    "ofi_activity_300s",
    "update_rate_300s",
    "price_move_rate_30s",
    "impact_lambda_1800s",
    "impact_r2_1800s",
    "ofi_response_gap_z",
    "queue_imbalance_l1",
    "depth_imbalance_l5",
    "queue_replenishment_pressure_60s",
    "queue_imbalance_delta",
    "depth_imbalance_delta",
    "spread_rank_1800s",
    "price_mom_10s_new",
    "price_mom_30s_new",
    "price_mom_60s_new",
    "log_rv_300s_new",
    "rv_ratio_60_300_new",
    "ret_acf_lag1_300s",
    "log_vr_5_300s",
    "log_vr_30_300s",
    "semivar_imbalance_300s",
    "semivar_imbalance_900s",
    "sign_persistence_900s",
    "jump_ratio_300s",
    "range_position_300s",
    "spread_depth_stress_900",
    "recovery_available_300",
    "ofi_disagreement_weighted_60"
]

REBUILT6 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a"
]

CONTROLS9 = [
    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds"
]

CHALLENGERS4 = [
    "recovery_pressure_34b",
    "ask_shock_age_34b",
    "side_update_imbalance_60_36a",
    "migration_activity_asym_60_39a"
]

BASE_INPUTS = CORE31 + REBUILT6 + CONTROLS9
CHALLENGER_INPUTS = BASE_INPUTS + CHALLENGERS4

assert len(CORE31) == 31
assert len(REBUILT6) == 6
assert len(CONTROLS9) == 9
assert len(CHALLENGERS4) == 4

assert len(BASE_INPUTS) == 46
assert len(CHALLENGER_INPUTS) == 50
assert len(set(CHALLENGER_INPUTS)) == 50

# ------------------------------------------------------------
# 1. Source availability checks
# ------------------------------------------------------------

REQUIRED_FRAMES = [
    "core_31",
    "usdinr_rebuilt_37_research",
    "calendar_exp34b_features",
    "calendar_exp36a_features",
    "calendar_exp39a_features"
]

missing_frames = [
    name for name in REQUIRED_FRAMES
    if name not in globals()
]

if missing_frames:
    raise ValueError(
        f"Missing frames: {missing_frames}. "
        "Load saved outputs or rerun their "
        "feature-construction cells."
    )

SOURCES = {
    "original31": core_31,
    "rebuilt": usdinr_rebuilt_37_research,
    "recovery": calendar_exp34b_features,
    "events": calendar_exp36a_features,
    "migration": calendar_exp39a_features
}

SOURCE_GROUPS = {
    "original31": CORE31,
    "rebuilt": REBUILT6 + CONTROLS9,
    "recovery": [
        "recovery_pressure_34b",
        "ask_shock_age_34b"
    ],
    "events": [
        "side_update_imbalance_60_36a"
    ],
    "migration": [
        "migration_activity_asym_60_39a"
    ]
}

# ------------------------------------------------------------
# 2. Freeze the exact research-key universe
#
# IMPORTANT: keys come from the already restricted
# research frame, not the full raw data.
# ------------------------------------------------------------

base_source = SOURCES["rebuilt"]

needed = KEYS + REBUILT6 + CONTROLS9

missing = [
    c for c in needed
    if c not in base_source.columns
]

if missing:
    raise ValueError(
        "Rebuilt frame missing: "
        + str(missing)
    )

research = base_source[KEYS].copy()

research["symbol"] = research[
    "symbol"
].astype(str)

research["ts_sec"] = pd.to_numeric(
    research["ts_sec"],
    errors="raise"
).astype(np.int64)

research["date"] = pd.to_datetime(
    research["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert(
    "Asia/Kolkata"
).dt.date

research = (
    research.loc[
        (research["date"].astype(str) <= "2026-10-19")
        & (research["date"].astype(str) != "2026-09-28")
    ]
    .sort_values(["date", "symbol", "ts_sec"])
    .reset_index(drop=True)
)

assert len(research) == 459122, (
    f"Expected 459122 research rows, "
    f"found {len(research)}"
)

assert not research.duplicated(KEYS).any()

sessions = research.groupby(
    ["symbol", "date"]
).ngroups

assert sessions == 16

# ------------------------------------------------------------
# 3. Exact keyed feature alignment
# ------------------------------------------------------------

def make_index(frame):
    return pd.MultiIndex.from_arrays(
        [
            frame["symbol"].astype(str),
            pd.to_numeric(
                frame["ts_sec"],
                errors="raise"
            ).astype(np.int64)
        ],
        names=KEYS
    )

target_index = make_index(research)

def align_source(frame, columns, label):

    missing = [
        c for c in KEYS + columns
        if c not in frame.columns
    ]

    if missing:
        raise ValueError(
            f"{label} missing columns: {missing}"
        )

    source_index = make_index(frame)

    if source_index.has_duplicates:
        raise ValueError(
            f"{label} has duplicate keys"
        )

    positions = source_index.get_indexer(
        target_index
    )

    if not (positions >= 0).all():
        raise ValueError(
            f"{label}: "
            f"{int((positions < 0).sum())} "
            "research keys missing"
        )

    result = {}

    for col in columns:
        values = pd.to_numeric(
            frame[col],
            errors="raise"
        ).to_numpy(dtype=np.float32)

        aligned = values[positions].copy()

        aligned[
            ~np.isfinite(aligned)
        ] = np.nan

        result[col] = aligned

    return pd.DataFrame(result)

feature_parts = {}

for source_name, columns in SOURCE_GROUPS.items():

    feature_parts[source_name] = align_source(
        SOURCES[source_name],
        columns,
        source_name
    )

# ------------------------------------------------------------
# 4. Construct frozen datasets
# ------------------------------------------------------------

features = pd.concat(
    [
        feature_parts["original31"],
        feature_parts["rebuilt"],
        feature_parts["recovery"],
        feature_parts["events"],
        feature_parts["migration"]
    ],
    axis=1
)

assert list(features.columns) == (
    CORE31
    + REBUILT6
    + CONTROLS9
    + CHALLENGERS4
)

frozen = pd.concat(
    [
        research[["symbol", "ts_sec", "date"]],
        features
    ],
    axis=1
)

# Preserve original historical research columns,
# including NaNs. Never substitute zeros for
# ask_shock_age when no shock was observed.

baseline = frozen[
    ["symbol", "ts_sec", "date"] + BASE_INPUTS
].copy()

challenger = frozen[
    ["symbol", "ts_sec", "date"] + CHALLENGER_INPUTS
].copy()

assert len(baseline) == len(challenger) == 459122

assert len(baseline.columns) == 49
assert len(challenger.columns) == 53

# ------------------------------------------------------------
# 5. Core data-quality checks
# ------------------------------------------------------------

# Compare frozen original 31 with source values.
assert np.array_equal(
    baseline[CORE31].isna().to_numpy(),
    feature_parts["original31"][CORE31].isna().to_numpy()
)

for col in CORE31:
    a = baseline[col].to_numpy()
    b = feature_parts["original31"][col].to_numpy()

    if not np.array_equal(
        a, b, equal_nan=True
    ):
        raise ValueError(
            f"Original 31 preservation failed: {col}"
        )

# Dataset contains historical observations only.
assert str(baseline["date"].max()) <= "2026-10-19"

# No target columns in input schema.
assert not any(
    c.startswith(("future_", "target_", "label_"))
    for c in CHALLENGER_INPUTS
)

# Both datasets have exactly matching keys.
assert baseline[KEYS].equals(
    challenger[KEYS]
)

# Validate known bounds.
BOUNDED = {
    "relative_migration_60_31a": (-1, 1),
    "queue_imbalance_28a": (-1, 1),
    "depth_imbalance_28a": (-1, 1),
    "side_update_imbalance_60_36a": (-1, 1)
}

for col, (lo, hi) in BOUNDED.items():

    v = challenger[col].dropna()

    if not v.between(
        lo - 1e-5,
        hi + 1e-5
    ).all():
        raise ValueError(
            f"Bounds check failed: {col}"
        )

age = challenger[
    "ask_shock_age_34b"
].dropna()

assert age.between(0, 60).all()

# ------------------------------------------------------------
# 6. Model sample diagnostics
# ------------------------------------------------------------

base_complete = np.isfinite(
    baseline[
        BASE_INPUTS
    ].to_numpy(dtype=np.float32)
).all(axis=1)

print("===== FINAL FEATURE FREEZE =====")
print("Research rows:", len(baseline))
print("Research sessions:", sessions)
print("First date:", research["date"].min())
print("Last date:", research["date"].max())
print("Original features:", len(CORE31))
print("Reconstructed features:", len(REBUILT6))
print("Controls:", len(CONTROLS9))
print("Challengers:", len(CHALLENGERS4))

print("\nBaseline inputs:", len(BASE_INPUTS))
print("Challenger inputs:", len(CHALLENGER_INPUTS))

print("\n===== COVERAGE =====")

coverage_rows = []

for name in CHALLENGER_INPUTS:

    s = challenger[name]

    coverage_rows.append({
        "feature": name,
        "coverage_pct": 100 * s.notna().mean(),
        "missing_pct": 100 * s.isna().mean(),
        "std": s.std()
    })

coverage_table = pd.DataFrame(
    coverage_rows
)

print(
    coverage_table.loc[
        coverage_table["feature"].isin(
            CHALLENGERS4
        )
    ]
    .round(5)
    .to_string(index=False)
)

print(
    "\nBaseline-complete rows:",
    int(base_complete.sum())
)

print(
    "Baseline-complete coverage:",
    round(
        100 * base_complete.mean(),
        5
    ), "%"
)

# ------------------------------------------------------------
# 7. Provenance and model schema
# ------------------------------------------------------------

def schema_digest(columns):
    serialized = json.dumps(
        columns,
        separators=(",", ":")
    )
    return hashlib.sha256(
        serialized.encode("utf-8")
    ).hexdigest()

manifest = {
    "version": "research_v2",
    "instrument": "USDINR futures",
    "time_resolution": "1_second",
    "session_timezone": "Asia/Kolkata",
    "research_end_date": "2026-10-19",
    "holdout_dates_reserved": [
        "2026-10-21",
        "2026-10-22",
        "2026-10-23"
    ],
    "research_rows": int(len(research)),
    "research_sessions": int(sessions),

    "baseline_engineered_count": 37,
    "baseline_input_count": 46,
    "challenger_engineered_count": 41,
    "challenger_input_count": 50,

    "original31": CORE31,
    "reconstructed6": REBUILT6,
    "controls9": CONTROLS9,
    "challengers4": CHALLENGERS4,

    "baseline_columns": BASE_INPUTS,
    "challenger_columns": CHALLENGER_INPUTS,

    "baseline_schema_sha256":
        schema_digest(BASE_INPUTS),

    "challenger_schema_sha256":
        schema_digest(CHALLENGER_INPUTS),

    "baseline_complete_rows":
        int(base_complete.sum()),

    "baseline_complete_coverage_pct":
        float(100 * base_complete.mean()),

    "formula_provenance": {
        "original31":
            "Values from original 31-feature parquet; "
            "underlying source implementations must "
            "be retained for future raw-data rebuilds.",

        "reconstructed6":
            "Research V2 reconstructed definitions. "
            "Exact original implementation parity "
            "is not established for all features.",

        "controls9":
            "Research V2 reconstructed controls; "
            "historical formula parity unverified.",

        "challengers4":
            "Calculated using historical experiment "
            "34B, 36A and 39A feature constructors."
    },

    "missing_value_policy": {
        "model": "LightGBM native NaN support",
        "ask_shock_age_34b":
            "NaN means no active qualifying ask shock",
        "no_global_imputation": True
    },

    "production_status":
        "RESEARCH FREEZE — NOT YET LIVE READY",

    "limitations": [
        "Some historical feature definitions "
        "were reconstructed.",
        "Holdout features must be generated "
        "with frozen causal implementations.",
        "No transaction cost or fill simulation "
        "has been completed.",
        "No independent holdout performance "
        "has been measured."
    ]
}

# ------------------------------------------------------------
# 8. Save frozen artifacts
# ------------------------------------------------------------

baseline_path = (
    OUTDIR / "usdinr_baseline_v2_46inputs.parquet"
)

challenger_path = (
    OUTDIR / "usdinr_challenger_v2_50inputs.parquet"
)

manifest_path = (
    OUTDIR / "usdinr_feature_manifest_v2.json"
)

baseline.to_parquet(
    baseline_path,
    index=False
)

challenger.to_parquet(
    challenger_path,
    index=False
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        manifest,
        f,
        indent=2
    )

# Reload and verify saved schemas.
check_base = pd.read_parquet(
    baseline_path
)

check_challenger = pd.read_parquet(
    challenger_path
)

assert list(check_base.columns) == (
    ["symbol", "ts_sec", "date"] + BASE_INPUTS
)

assert list(check_challenger.columns) == (
    ["symbol", "ts_sec", "date"] + CHALLENGER_INPUTS
)

assert len(check_base) == len(check_challenger)

del check_base, check_challenger

print("\n===== SAVED FILES =====")

for path in [
    baseline_path,
    challenger_path,
    manifest_path
]:
    print(
        str(path.resolve()),
        "bytes=", path.stat().st_size
    )

print("\n===== FREEZE COMPLETE =====")
print("Baseline: 37 engineered + 9 controls")
print("Challenger: 41 engineered + 9 controls")
print("Original 31 preserved")
print("Target columns excluded")
print("Historical research dates only")
print("Holdout untouched")
print("Ready for versioned model development")

===== FINAL FEATURE FREEZE =====
Research rows: 459122
Research sessions: 16
First date: 2026-09-23
Last date: 2026-10-19
Original features: 31
Reconstructed features: 6
Controls: 9
Challengers: 4

Baseline inputs: 46
Challenger inputs: 50

===== COVERAGE =====
                       feature  coverage_pct  missing_pct      std
         recovery_pressure_34b     100.00000      0.00000  0.82808
             ask_shock_age_34b      66.30874     33.69126 16.30826
  side_update_imbalance_60_36a      99.79439      0.20561  0.57994
migration_activity_asym_60_39a      99.79439      0.20561  2.16303

Baseline-complete rows: 402979
Baseline-complete coverage: 87.77166 %

===== SAVED FILES =====
C:\projects\hedge_fund\research\notebooks\usdinr_final_v2\usdinr_baseline_v2_46inputs.parquet bytes= 36553542
C:\projects\hedge_fund\research\notebooks\usdinr_final_v2\usdinr_challenger_v2_50inputs.parquet bytes= 39524428
C:\projects\hedge_fund\research\notebooks\usdinr_final_v2\usdinr_feature_manifest_v2.

In [37]:

import pandas as pd

required = {
    "Raw 1-second L5 data": "df_1s",
    "Original 31 features": "core_31",
    "Reconstructed baseline": "usdinr_rebuilt_37_research",
    "Recovery features": "calendar_exp34b_features",
    "Event features": "calendar_exp36a_features",
    "Migration features": "calendar_exp39a_features",
}

print("===== PIPELINE SOURCE AUDIT =====")

for label, variable in required.items():
    if variable in globals():
        obj = globals()[variable]
        if isinstance(obj, pd.DataFrame):
            print(
                f"{label}: FOUND | "
                f"shape={obj.shape}"
            )
        else:
            print(
                f"{label}: FOUND | "
                f"type={type(obj).__name__}"
            )
    else:
        print(f"{label}: MISSING")

print("\n===== SAVED FEATURE DATASETS =====")

for name in [
    "usdinr_baseline_v2_46inputs.parquet",
    "usdinr_challenger_v2_50inputs.parquet",
    "usdinr_feature_manifest_v2.json"
]:
    from pathlib import Path
    path = Path("usdinr_final_v2") / name
    print(
        name,
        "FOUND" if path.exists() else "NOT FOUND"
    )


===== PIPELINE SOURCE AUDIT =====
Raw 1-second L5 data: FOUND | shape=(639440, 65)
Original 31 features: FOUND | shape=(524325, 33)
Reconstructed baseline: FOUND | shape=(459122, 48)
Recovery features: FOUND | shape=(459122, 14)
Event features: FOUND | shape=(459122, 13)
Migration features: FOUND | shape=(459122, 12)

===== SAVED FEATURE DATASETS =====
usdinr_baseline_v2_46inputs.parquet FOUND
usdinr_challenger_v2_50inputs.parquet FOUND
usdinr_feature_manifest_v2.json FOUND


In [38]:

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("usdinr_final_v2")

baseline = pd.read_parquet(
    ROOT / "usdinr_baseline_v2_46inputs.parquet"
)
challenger = pd.read_parquet(
    ROOT / "usdinr_challenger_v2_50inputs.parquet"
)

manifest = pd.read_json(
    ROOT / "usdinr_feature_manifest_v2.json",
    typ="series"
)

KEYS = ["symbol", "ts_sec"]

sources = {
    "Original31": (
        core_31,
        manifest["original31"]
    ),
    "Reconstructed6": (
        usdinr_rebuilt_37_research,
        manifest["reconstructed6"]
    ),
    "Controls9": (
        usdinr_rebuilt_37_research,
        manifest["controls9"]
    ),
    "Recovery34B": (
        calendar_exp34b_features,
        [
            "recovery_pressure_34b",
            "ask_shock_age_34b"
        ]
    ),
    "Events36A": (
        calendar_exp36a_features,
        ["side_update_imbalance_60_36a"]
    ),
    "Migration39A": (
        calendar_exp39a_features,
        ["migration_activity_asym_60_39a"]
    )
}

ref = challenger.set_index(KEYS)

records = []

for group, (source, columns) in sources.items():

    src = source.copy()

    src["symbol"] = src["symbol"].astype(str)
    src["ts_sec"] = src["ts_sec"].astype("int64")

    src = src.set_index(KEYS)

    if src.index.has_duplicates:
        raise ValueError(
            f"Duplicate keys in {group}"
        )

    aligned = src.reindex(ref.index)

    for col in columns:

        a = pd.to_numeric(
            ref[col], errors="coerce"
        ).to_numpy(dtype=float)

        b = pd.to_numeric(
            aligned[col], errors="coerce"
        ).to_numpy(dtype=float)

        both_finite = (
            np.isfinite(a)
            & np.isfinite(b)
        )

        diff = np.abs(
            a[both_finite] - b[both_finite]
        )

        missing_mismatch = int(
            (np.isnan(a) != np.isnan(b)).sum()
        )

        records.append({
            "group": group,
            "feature": col,
            "finite_pairs": int(both_finite.sum()),
            "missing_mismatch": missing_mismatch,
            "max_abs_diff": (
                float(diff.max())
                if len(diff) else np.nan
            ),
            "mean_abs_diff": (
                float(diff.mean())
                if len(diff) else np.nan
            )
        })

audit = pd.DataFrame(records)

print("===== RESEARCH FEATURE PARITY =====")
print("Baseline rows:", len(baseline))
print("Challenger rows:", len(challenger))
print("Audited features:", len(audit))

print("\n===== GROUP SUMMARY =====")

print(
    audit.groupby("group").agg(
        features=("feature", "count"),
        missing_mismatches=(
            "missing_mismatch", "sum"
        ),
        largest_abs_diff=(
            "max_abs_diff", "max"
        )
    ).round(10).to_string()
)

print("\n===== FEATURES WITH DIFFERENCES =====")

problems = audit.loc[
    (audit["missing_mismatch"] > 0)
    | (audit["max_abs_diff"] > 1e-6)
]

if len(problems):
    print(
        problems.round(10).to_string(index=False)
    )
else:
    print("No numerical or missingness mismatches.")

print("\n===== FROZEN SCHEMA =====")
print("Baseline model inputs:", len(baseline.columns) - 3)
print("Challenger model inputs:", len(challenger.columns) - 3)

print("\n===== AUDIT COMPLETE =====")


===== RESEARCH FEATURE PARITY =====
Baseline rows: 459122
Challenger rows: 459122
Audited features: 50

===== GROUP SUMMARY =====
                features  missing_mismatches  largest_abs_diff
group                                                         
Controls9              9                   0          0.000000
Events36A              1                   0          0.000000
Migration39A           1                   0          0.000000
Original31            31                   0          0.000003
Reconstructed6         6                   0          0.000000
Recovery34B            2                   0          0.000000

===== FEATURES WITH DIFFERENCES =====
     group                          feature  finite_pairs  missing_mismatch  max_abs_diff  mean_abs_diff
Original31 queue_replenishment_pressure_60s        458818                 0      0.000003   4.290000e-08

===== FROZEN SCHEMA =====
Baseline model inputs: 46
Challenger model inputs: 50

===== AUDIT COMPLETE =====


In [39]:

import inspect
import types
from pathlib import Path

print("===== FEATURE PIPELINE DISCOVERY =====")

# 1. Find currently defined feature-building functions.
keywords = [
    "feature", "ofi", "queue", "depth",
    "volatility", "momentum", "impact",
    "migration", "recovery", "build",
    "compute", "reconstruct"
]

functions_found = []

for name, obj in list(globals().items()):
    if not isinstance(obj, types.FunctionType):
        continue

    if any(k in name.lower() for k in keywords):
        try:
            source = inspect.getsource(obj)
            lines = len(source.splitlines())
        except (OSError, TypeError):
            lines = "unavailable"

        functions_found.append((name, lines))

print("\n===== AVAILABLE FUNCTIONS =====")

if functions_found:
    for name, lines in sorted(functions_found):
        print(f"{name:55s} source_lines={lines}")
else:
    print("No matching functions found.")

# 2. Identify saved Python and notebook files.
print("\n===== LOCAL IMPLEMENTATION FILES =====")

search_roots = [
    Path.cwd(),
    Path.cwd() / "src",
    Path.cwd() / "features",
]

seen = set()
found_files = []

for root in search_roots:
    if not root.exists():
        continue

    for pattern in ["*.py", "*.ipynb"]:
        for path in root.glob(pattern):
            resolved = str(path.resolve())

            if resolved not in seen:
                seen.add(resolved)
                found_files.append(path)

for path in sorted(found_files):
    print(path)

if not found_files:
    print("No Python/notebook files found in searched directories.")

# 3. Inspect available source-code cells through IPython.
print("\n===== NOTEBOOK HISTORY =====")

ip = get_ipython() if "get_ipython" in globals() else None

if ip is not None and hasattr(ip, "history_manager"):
    history = list(
        ip.history_manager.get_range(
            output=False
        )
    )

    matches = []

    for session, line, code in history:
        lower = code.lower()

        if (
            ("core_31" in lower or
             "usdinr_31_features" in lower)
            and
            any(
                k in lower
                for k in [
                    "rolling", "groupby",
                    "def ", "ofi",
                    "to_parquet"
                ]
            )
        ):
            matches.append({
                "session": session,
                "line": line,
                "length": len(code),
                "preview": code[:140].replace("\n", " ")
            })

    print("Relevant historical cells:", len(matches))

    for m in matches[-25:]:
        print(
            f"session={m['session']} "
            f"cell={m['line']} "
            f"chars={m['length']} "
            f"preview={m['preview']}"
        )
else:
    print("IPython history unavailable.")

print("\n===== DISCOVERY COMPLETE =====")


===== FEATURE PIPELINE DISCOVERY =====

===== AVAILABLE FUNCTIONS =====
make_feature_frame                                      source_lines=15
recovery_34a                                            source_lines=36

===== LOCAL IMPLEMENTATION FILES =====
C:\projects\hedge_fund\research\notebooks\01_kappa_estimation.ipynb
C:\projects\hedge_fund\research\notebooks\02_queue_position.ipynb
C:\projects\hedge_fund\research\notebooks\03_realized_volatility.ipynb
C:\projects\hedge_fund\research\notebooks\04_regime_detection.ipynb
C:\projects\hedge_fund\research\notebooks\05_parameter_optimization.ipynb
C:\projects\hedge_fund\research\notebooks\60s_price_prediction.py
C:\projects\hedge_fund\research\notebooks\calender_and_intraday_seasonality.ipynb
C:\projects\hedge_fund\research\notebooks\core_features_usdinr.ipynb
C:\projects\hedge_fund\research\notebooks\feature_engineering.ipynb
C:\projects\hedge_fund\research\notebooks\feature_engineering_continuation.ipynb
C:\projects\hedge_fund\research

In [40]:

import json
from pathlib import Path

notebooks = [
    "core_features_usdinr.ipynb",
    "feature_engineering.ipynb",
    "feature_engineering_continuation.ipynb"
]

keywords = [
    "ofi_l1_sum_60s_signed_log",
    "queue_replenishment_pressure_60s",
    "relative_migration_60_31a",
    "quote_retreat_imbalance_30_24a",
    "recovery_pressure_34b",
    "side_update_imbalance_60_36a",
    "migration_activity_asym_60_39a"
]

extracted = []

for filename in notebooks:
    path = Path(filename)

    if not path.exists():
        print("NOT FOUND:", filename)
        continue

    nb = json.loads(path.read_text(encoding="utf-8"))
    cells = nb.get("cells", [])

    print(f"\n===== {filename} =====")
    print("Total cells:", len(cells))

    for i, cell in enumerate(cells):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        matched = [
            k for k in keywords if k in source
        ]

        if not matched:
            continue

        extracted.append({
            "notebook": filename,
            "cell_index": i,
            "matched_features": matched,
            "source": source
        })

        print(
            f"Cell {i}: "
            f"{len(source)} characters | "
            f"{', '.join(matched)}"
        )

output = Path("usdinr_feature_source_extract.txt")

with output.open("w", encoding="utf-8") as f:
    for item in extracted:
        f.write(
            f"\n{'=' * 70}\n"
            f"NOTEBOOK: {item['notebook']}\n"
            f"CELL: {item['cell_index']}\n"
            f"MATCHES: {item['matched_features']}\n"
            f"{'=' * 70}\n\n"
            f"{item['source']}\n"
        )

print("\n===== EXTRACTION COMPLETE =====")
print("Matching cells:", len(extracted))
print("Saved to:", output.resolve())



===== core_features_usdinr.ipynb =====
Total cells: 96
Cell 48: 92 characters | ofi_l1_sum_60s_signed_log
Cell 59: 781 characters | queue_replenishment_pressure_60s
Cell 86: 2962 characters | ofi_l1_sum_60s_signed_log, queue_replenishment_pressure_60s
Cell 95: 14736 characters | ofi_l1_sum_60s_signed_log, queue_replenishment_pressure_60s, relative_migration_60_31a, quote_retreat_imbalance_30_24a, recovery_pressure_34b, side_update_imbalance_60_36a, migration_activity_asym_60_39a

===== feature_engineering.ipynb =====
Total cells: 289
Cell 55: 333 characters | ofi_l1_sum_60s_signed_log
Cell 65: 403 characters | ofi_l1_sum_60s_signed_log
Cell 95: 40 characters | ofi_l1_sum_60s_signed_log
Cell 99: 198 characters | ofi_l1_sum_60s_signed_log
Cell 115: 112 characters | ofi_l1_sum_60s_signed_log
Cell 121: 73 characters | ofi_l1_sum_60s_signed_log
Cell 154: 347 characters | queue_replenishment_pressure_60s
Cell 157: 130 characters | ofi_l1_sum_60s_signed_log
Cell 159: 118 characters | queue_r

In [41]:

import json
from pathlib import Path

# Run from the notebooks directory.
source_path = Path("core_features_usdinr.ipynb")
output_path = Path("usdinr_original_core_source.txt")

if not source_path.exists():
    raise FileNotFoundError(source_path.resolve())

with source_path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

cells = notebook["cells"]
extracted = []

for index, cell in enumerate(cells):
    if cell.get("cell_type") != "code":
        continue

    # Cell 95 contains the final research freeze,
    # not the original core generation pipeline.
    if index == 95:
        continue

    code = "".join(cell.get("source", []))

    if not code.strip():
        continue

    extracted.append(
        f"\n{'=' * 72}\n"
        f"ORIGINAL NOTEBOOK CELL {index}\n"
        f"{'=' * 72}\n\n"
        f"{code}\n"
    )

with output_path.open("w", encoding="utf-8") as f:
    f.write(
        "# Original USDINR core feature source\n"
        "# Extracted for static review only.\n"
        "# Cells are NOT executed by this script.\n"
    )
    f.writelines(extracted)

print("===== ORIGINAL CORE EXTRACTION =====")
print("Notebook cells:", len(cells))
print("Extracted code cells:", len(extracted))
print("Output:", output_path.resolve())
print("Size:", output_path.stat().st_size, "bytes")
print("Extraction complete.")


===== ORIGINAL CORE EXTRACTION =====
Notebook cells: 96
Extracted code cells: 60
Output: C:\projects\hedge_fund\research\notebooks\usdinr_original_core_source.txt
Size: 89733 bytes
Extraction complete.


In [43]:

import numpy as np
import pandas as pd
import importlib
import usdinr_core31_rebuild_audit_v2 as core_module

# ============================================================
# USDINR ORIGINAL 31 — RAW FEATURE REBUILD PARITY TEST
#
# Uses corrected module:
#   usdinr_core31_rebuild_audit_v2.py
#
# Compares:
#   Rebuilt original 31 features
#          VS
#   Historical original 31 feature values
#
# Research dates only, through October 19, 2026.
# October 21–23 holdout remains untouched.
# ============================================================

# Reload module so that edits are recognized without
# restarting the Jupyter kernel.
importlib.reload(core_module)

build_core31_from_regularized = (
    core_module.build_core31_from_regularized
)

CORE31 = core_module.CORE31

# ------------------------------------------------------------
# 1. Prepare research data
# ------------------------------------------------------------

keys = ["symbol", "ts_sec"]

reference_keys = (
    usdinr_rebuilt_37_research[keys]
    .copy()
)

reference_keys["symbol"] = (
    reference_keys["symbol"].astype(str)
)

reference_keys["ts_sec"] = pd.to_numeric(
    reference_keys["ts_sec"],
    errors="raise"
).astype(np.int64)

raw_source = df_1s.copy()

raw_source["symbol"] = (
    raw_source["symbol"].astype(str)
)

raw_source["ts_sec"] = pd.to_numeric(
    raw_source["ts_sec"],
    errors="raise"
).astype(np.int64)

# Validate uniqueness before merging.
assert not reference_keys.duplicated(keys).any(), (
    "Duplicate research keys detected."
)

assert not raw_source.duplicated(keys).any(), (
    "Duplicate raw-data keys detected."
)

raw = raw_source.merge(
    reference_keys,
    on=keys,
    how="inner",
    validate="one_to_one"
)

raw = (
    raw.sort_values(
        ["symbol", "ts_sec"]
    )
    .reset_index(drop=True)
)

assert len(raw) == 459122, (
    f"Expected 459122 rows, found {len(raw)}"
)

assert not raw.duplicated(keys).any()

# ------------------------------------------------------------
# 2. Rebuild original 31 features
# ------------------------------------------------------------

print("===== ORIGINAL 31 FEATURE REBUILD =====")
print("Research rows:", len(raw))
print("Feature count:", len(CORE31))

print("\nRebuilding the original 31 features...")

rebuilt = build_core31_from_regularized(
    raw,
    quiet=True
)

# ------------------------------------------------------------
# 3. Prepare historical reference
# ------------------------------------------------------------

reference = core_31[
    keys + CORE31
].copy()

reference["symbol"] = (
    reference["symbol"].astype(str)
)

reference["ts_sec"] = pd.to_numeric(
    reference["ts_sec"],
    errors="raise"
).astype(np.int64)

assert not reference.duplicated(keys).any(), (
    "Duplicate keys in original feature reference."
)

# ------------------------------------------------------------
# 4. Align by exact timestamp and symbol
# ------------------------------------------------------------

comparison = rebuilt.merge(
    reference,
    on=keys,
    how="left",
    suffixes=("_rebuilt", "_original"),
    validate="one_to_one",
    indicator=True
)

assert len(comparison) == 459122, (
    "Unexpected comparison row count."
)

assert comparison["_merge"].eq("both").all(), (
    "Some reconstructed rows are missing "
    "from the original feature reference."
)

print("\nTimestamp alignment: PASSED")
print("Matched rows:", len(comparison))

# ------------------------------------------------------------
# 5. Numerical and missing-value parity
# ------------------------------------------------------------

results = []

for feature in CORE31:

    rebuilt_values = pd.to_numeric(
        comparison[f"{feature}_rebuilt"],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    original_values = pd.to_numeric(
        comparison[f"{feature}_original"],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    rebuilt_missing = np.isnan(
        rebuilt_values
    )

    original_missing = np.isnan(
        original_values
    )

    missing_mismatch = int(
        np.sum(
            rebuilt_missing != original_missing
        )
    )

    finite = (
        np.isfinite(rebuilt_values)
        & np.isfinite(original_values)
    )

    errors = np.abs(
        rebuilt_values[finite]
        - original_values[finite]
    )

    if len(errors) > 0:
        max_error = float(errors.max())
        mean_error = float(errors.mean())
        match_pct = float(
            100 * np.mean(errors <= 1e-5)
        )
    else:
        max_error = np.nan
        mean_error = np.nan
        match_pct = np.nan

    passed = (
        missing_mismatch == 0
        and np.isfinite(max_error)
        and max_error <= 1e-5
    )

    results.append({
        "feature": feature,
        "finite_pairs": int(finite.sum()),
        "missing_mismatch": missing_mismatch,
        "max_abs_error": max_error,
        "mean_abs_error": mean_error,
        "matching_pct_at_1e-5": match_pct,
        "status": "PASS" if passed else "FAIL"
    })

parity = pd.DataFrame(results)

# ------------------------------------------------------------
# 6. Display full parity table
# ------------------------------------------------------------

print("\n===== ORIGINAL 31 REBUILD PARITY =====")

print(
    parity.round(8).to_string(index=False)
)

# ------------------------------------------------------------
# 7. Summarize results
# ------------------------------------------------------------

print("\n===== PARITY SUMMARY =====")

print(
    "Total features:",
    len(parity)
)

print(
    "Features passing:",
    int((parity["status"] == "PASS").sum())
)

print(
    "Features failing:",
    int((parity["status"] == "FAIL").sum())
)

print(
    "Features with missingness mismatches:",
    int((parity["missing_mismatch"] > 0).sum())
)

print(
    "Features with max error above 1e-5:",
    int((parity["max_abs_error"] > 1e-5).sum())
)

# ------------------------------------------------------------
# 8. Display failed features separately
# ------------------------------------------------------------

failed = parity.loc[
    parity["status"] == "FAIL"
].copy()

print("\n===== FAILED FEATURES =====")

if failed.empty:
    print("All 31 features passed numerical parity.")
else:
    print(
        failed.round(8).to_string(index=False)
    )

print("\n===== TEST COMPLETE =====")
print("Historical development sample only.")
print("October 21–23 holdout untouched.")


===== ORIGINAL 31 FEATURE REBUILD =====
Research rows: 459122
Feature count: 31

Rebuilding the original 31 features...

Timestamp alignment: PASSED
Matched rows: 459122

===== ORIGINAL 31 REBUILD PARITY =====
                         feature  finite_pairs  missing_mismatch  max_abs_error  mean_abs_error  matching_pct_at_1e-5 status
       ofi_l1_sum_60s_signed_log        458802                 0            0.0             0.0                 100.0   PASS
         ofi_rate_diff_30_300_sl        457522                 0            0.0             0.0                 100.0   PASS
                ofi_activity_60s        458818                 0            0.0             0.0                 100.0   PASS
               ofi_activity_300s        457538                 0            0.0             0.0                 100.0   PASS
                update_rate_300s        457538                 0            0.0             0.0                 100.0   PASS
             price_move_rate_30s        

In [45]:

import ast
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# EXPERIMENT 42 — REMAINING 19 FEATURE PARITY
#
# No external Python module required.
# Reads feature-construction code from:
#     feature_engineering_continuation.ipynb
#
# Original notebook cell indices:
#     18 -> Six features + nine controls
#     22 -> Recovery and shock age
#     25 -> Side update imbalance
#     31 -> Migration activity asymmetry
#
# Historical research rows only.
# October 21–23 untouched.
# ============================================================

KEYS = ["symbol", "ts_sec"]

REMAINING19 = [
    "quote_retreat_imbalance_30_24a",
    "slope_asymmetry_28a",
    "curvature_asymmetry_28a",
    "weighted_gap_asymmetry_29a",
    "fragility_asymmetry_29a",
    "relative_migration_60_31a",

    "queue_imbalance_28a",
    "depth_imbalance_28a",
    "log_total_depth_28a",
    "spread_bps_28a",
    "bid_l1_concentration_28a",
    "ask_l1_concentration_28a",
    "momentum_30_28a",
    "momentum_60_28a",
    "clock_seconds",

    "recovery_pressure_34b",
    "ask_shock_age_34b",
    "side_update_imbalance_60_36a",
    "migration_activity_asym_60_39a"
]

assert len(REMAINING19) == 19
assert len(set(REMAINING19)) == 19

# ------------------------------------------------------------
# 1. Load original research notebook source
# ------------------------------------------------------------

notebook_path = Path(
    "feature_engineering_continuation.ipynb"
)

if not notebook_path.exists():
    raise FileNotFoundError(
        f"Cannot find {notebook_path.resolve()}"
    )

with notebook_path.open(
    "r", encoding="utf-8"
) as f:
    notebook = json.load(f)

cells = notebook["cells"]

# Expected final calculation statement in each cell.
# We stop immediately after it.
STOP_PATTERNS = {
    18: (
        "calendar_exp31a_features",
        "assign"
    ),
    22: (
        "for col, arr in feature_values.items()",
        "for"
    ),
    25: (
        "for name, values in arrays.items()",
        "for"
    ),
    31: (
        "for col, values in zip(",
        "for"
    )
}

def extract_feature_code(cell_index):

    cell = cells[cell_index]

    assert cell["cell_type"] == "code"

    source = "".join(cell["source"])
    tree = ast.parse(source)

    pattern, kind = STOP_PATTERNS[cell_index]

    stop_line = None

    for node in tree.body:

        snippet = ast.get_source_segment(
            source, node
        ) or ""

        if kind == "assign":
            if isinstance(
                node, (ast.Assign, ast.AnnAssign)
            ):
                targets = (
                    node.targets
                    if isinstance(node, ast.Assign)
                    else [node.target]
                )

                if any(
                    isinstance(t, ast.Name)
                    and t.id == pattern
                    for t in targets
                ):
                    stop_line = node.end_lineno
                    break

        elif kind == "for":
            if isinstance(node, ast.For):
                header = snippet.splitlines()[0]

                if pattern in header:
                    stop_line = node.end_lineno
                    break

    if stop_line is None:
        raise ValueError(
            f"Could not identify feature-construction "
            f"boundary in cell {cell_index}. "
            "Notebook may have changed."
        )

    prefix = "\n".join(
        source.splitlines()[:stop_line]
    )

    # Confirm that the extracted prefix is valid code.
    compile(
        prefix,
        f"<research_cell_{cell_index}>",
        "exec"
    )

    return prefix

# ------------------------------------------------------------
# 2. Prepare exact historical raw sample
# ------------------------------------------------------------

research_keys = (
    usdinr_rebuilt_37_research[KEYS]
    .copy()
)

research_keys["symbol"] = (
    research_keys["symbol"].astype(str)
)

research_keys["ts_sec"] = pd.to_numeric(
    research_keys["ts_sec"],
    errors="raise"
).astype(np.int64)

raw_source = df_1s.copy()

raw_source["symbol"] = (
    raw_source["symbol"].astype(str)
)

raw_source["ts_sec"] = pd.to_numeric(
    raw_source["ts_sec"],
    errors="raise"
).astype(np.int64)

assert not raw_source.duplicated(KEYS).any()
assert not research_keys.duplicated(KEYS).any()

raw_research = raw_source.merge(
    research_keys,
    on=KEYS,
    how="inner",
    validate="one_to_one"
)

raw_research = (
    raw_research.sort_values(
        ["symbol", "ts_sec"]
    )
    .reset_index(drop=True)
)

assert len(raw_research) == 459122

# Confirm the research sample has no holdout rows.
research_dates = pd.to_datetime(
    raw_research["ts_sec"],
    unit="s",
    utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

assert max(research_dates).isoformat() <= "2026-10-19"

print("===== RESEARCH INPUT =====")
print("Rows:", len(raw_research))
print("Sessions:", len(set(
    zip(raw_research["symbol"], research_dates)
)))

# ------------------------------------------------------------
# 3. Execute original feature calculations
# ------------------------------------------------------------

# Use separate namespaces for independent experiments.
# This prevents variables from one experiment from
# contaminating another experiment's calculations.

def run_original_cell(cell_index):

    code = extract_feature_code(cell_index)

    namespace = {
        "np": np,
        "pd": pd,
        "df_1s": raw_research.copy()
    }

    exec(
        compile(
            code,
            f"<feature_cell_{cell_index}>",
            "exec"
        ),
        namespace
    )

    return namespace

print("\n===== REBUILDING FEATURES =====")

ns18 = run_original_cell(18)
print("Experiment 24A–31A: completed")

ns22 = run_original_cell(22)
print("Experiment 34B: completed")

ns25 = run_original_cell(25)
print("Experiment 36A: completed")

ns31 = run_original_cell(31)
print("Experiment 39A: completed")

# ------------------------------------------------------------
# 4. Collect all resulting feature frames
# ------------------------------------------------------------

FRAME_REGISTRY = [
    (ns18, "calendar_exp24a_features"),
    (ns18, "calendar_exp28a_features"),
    (ns18, "calendar_exp29a_features"),
    (ns18, "calendar_exp31a_features"),
    (ns22, "calendar_exp34b_features"),
    (ns25, "calendar_exp36a_features"),
    (ns31, "calendar_exp39a_features")
]

parts = []

for namespace, variable in FRAME_REGISTRY:

    if variable not in namespace:
        raise ValueError(
            f"Feature frame not created: {variable}"
        )

    frame = namespace[variable]

    selected = [
        col for col in REMAINING19
        if col in frame.columns
    ]

    if not selected:
        raise ValueError(
            f"No requested features in {variable}"
        )

    part = frame[
        KEYS + selected
    ].copy()

    part["symbol"] = part["symbol"].astype(str)
    part["ts_sec"] = part["ts_sec"].astype(np.int64)

    assert not part.duplicated(KEYS).any()

    parts.append(part)

# Merge by exact symbol and second.
rebuilt19 = parts[0]

for part in parts[1:]:
    rebuilt19 = rebuilt19.merge(
        part,
        on=KEYS,
        how="inner",
        validate="one_to_one"
    )

missing = [
    col for col in REMAINING19
    if col not in rebuilt19.columns
]

if missing:
    raise ValueError(
        f"Missing rebuilt features: {missing}"
    )

rebuilt19 = rebuilt19[
    KEYS + REMAINING19
]

assert len(rebuilt19) == 459122

# ------------------------------------------------------------
# 5. Load frozen V2 reference
# ------------------------------------------------------------

reference_path = (
    Path("usdinr_final_v2")
    / "usdinr_challenger_v2_50inputs.parquet"
)

reference = pd.read_parquet(
    reference_path,
    columns=KEYS + REMAINING19
)

reference["symbol"] = reference["symbol"].astype(str)
reference["ts_sec"] = reference["ts_sec"].astype(np.int64)

comparison = rebuilt19.merge(
    reference,
    on=KEYS,
    how="left",
    suffixes=("_rebuilt", "_frozen"),
    validate="one_to_one",
    indicator=True
)

assert len(comparison) == 459122
assert comparison["_merge"].eq("both").all()

print("\n===== EXACT KEY ALIGNMENT =====")
print("Matched rows:", len(comparison))
print("Features:", len(REMAINING19))

# ------------------------------------------------------------
# 6. Numerical and missing-value parity
# ------------------------------------------------------------

records = []

for feature in REMAINING19:

    a = pd.to_numeric(
        comparison[f"{feature}_rebuilt"],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    b = pd.to_numeric(
        comparison[f"{feature}_frozen"],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    finite = np.isfinite(a) & np.isfinite(b)

    missing_mismatch = int(
        (np.isnan(a) != np.isnan(b)).sum()
    )

    error = np.abs(a[finite] - b[finite])

    max_error = (
        float(error.max())
        if error.size else np.nan
    )

    mean_error = (
        float(error.mean())
        if error.size else np.nan
    )

    # Same initial absolute threshold as our
    # previous original-31 parity audit.
    passed = (
        missing_mismatch == 0
        and error.size > 0
        and max_error <= 1e-5
    )

    records.append({
        "feature": feature,
        "finite_pairs": int(finite.sum()),
        "missing_mismatch": missing_mismatch,
        "max_abs_error": max_error,
        "mean_abs_error": mean_error,
        "status": "PASS" if passed else "FAIL"
    })

parity19 = pd.DataFrame(records)

# ------------------------------------------------------------
# 7. Final report
# ------------------------------------------------------------

print("\n===== REMAINING 19 FEATURE PARITY =====")

print(
    parity19.round(8).to_string(index=False)
)

print("\n===== PARITY SUMMARY =====")

print(
    "Passed:",
    int((parity19.status == "PASS").sum()),
    "/ 19"
)

print(
    "Failed:",
    int((parity19.status == "FAIL").sum())
)

print(
    "Missingness mismatch features:",
    int((parity19.missing_mismatch > 0).sum())
)

print("\n===== FAILED FEATURES =====")

failed19 = parity19.loc[
    parity19.status == "FAIL"
]

if failed19.empty:
    print("All 19 features passed!")
else:
    print(
        failed19.round(8).to_string(index=False)
    )

print("\n===== EXPERIMENT 42 COMPLETE =====")
print("Original 31: already verified")
print("Remaining 19: see parity summary")
print("Oct 21–23 holdout untouched")


===== RESEARCH INPUT =====
Rows: 459122
Sessions: 16

===== REBUILDING FEATURES =====
===== RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Latest date: 2026-10-19
Experiment 24A–31A: completed
===== 34B RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Last date: 2026-10-19

===== 34B DISTINCT SHOCK EVENTS =====
Bid shocks: 11364
Ask shocks: 26552
Experiment 34B: completed
===== 36A RESEARCH SAMPLE =====
Rows: 459122
Sessions: 16
Last date: 2026-10-19

===== 36A EVENT DIAGNOSTICS =====
Observed seconds: 269455
Book-change seconds: 194598
Bid-change seconds: 79628
Ask-change seconds: 147020
Both-sides change seconds: 32050
Experiment 36A: completed
Experiment 39A: completed

===== EXACT KEY ALIGNMENT =====
Matched rows: 459122
Features: 19

===== REMAINING 19 FEATURE PARITY =====
                       feature  finite_pairs  missing_mismatch  max_abs_error  mean_abs_error status
quote_retreat_imbalance_30_24a        458658                 0            0.0             0.0   PASS
         

In [46]:

from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# EXPORT FROZEN RESEARCH RAW DATA
# Run in the OLD research notebook, only once.
# ============================================================

KEYS = ["symbol", "ts_sec"]

raw_keys = usdinr_rebuilt_37_research[KEYS].copy()
raw_keys["symbol"] = raw_keys["symbol"].astype(str)
raw_keys["ts_sec"] = raw_keys["ts_sec"].astype(np.int64)

source = df_1s.copy()
source["symbol"] = source["symbol"].astype(str)
source["ts_sec"] = source["ts_sec"].astype(np.int64)

assert not source.duplicated(KEYS).any()
assert not raw_keys.duplicated(KEYS).any()

raw_dev = source.merge(
    raw_keys,
    on=KEYS,
    how="inner",
    validate="one_to_one"
)

raw_dev = (
    raw_dev
    .sort_values(["symbol", "ts_sec"])
    .reset_index(drop=True)
)

dates = pd.to_datetime(
    raw_dev["ts_sec"], unit="s", utc=True
).dt.tz_convert("Asia/Kolkata").dt.date

assert len(raw_dev) == 459122
assert dates.max().isoformat() <= "2026-10-19"
assert len(set(zip(raw_dev["symbol"], dates))) == 16

out = Path("usdinr_final_v2")
out.mkdir(parents=True, exist_ok=True)

path = out / "usdinr_raw_development_16sessions.parquet"

raw_dev.to_parquet(path, index=False)

print("===== RAW DEVELOPMENT EXPORT =====")
print("Rows:", len(raw_dev))
print("Sessions:", 16)
print("Columns:", len(raw_dev.columns))
print("First date:", dates.min())
print("Last date:", dates.max())
print("Saved to:", path.resolve())
print("Holdout excluded.")


===== RAW DEVELOPMENT EXPORT =====
Rows: 459122
Sessions: 16
Columns: 65
First date: 2026-09-23
Last date: 2026-10-19
Saved to: C:\projects\hedge_fund\research\notebooks\usdinr_final_v2\usdinr_raw_development_16sessions.parquet
Holdout excluded.
